# Turn-Based Lexical Combat Game — Sentence Transformer (MiniLM) Baseline

This notebook scores the game with a **Sentence Transformer**: `sentence-transformers/all-MiniLM-L6-v2`, a pretrained Transformer that encodes a whole phrase into one vector (no word-vector averaging). It is the contextual baseline that the static word-embedding models (GloVe, Word2Vec, FastText, PPMI-SVD) are compared against. By default the pretrained model is used as is (`MODEL_CHOICE = "base"`); the Appendix can optionally fine-tune it.

Players type 1–3 English words to attack a boss that has a **target word** in a **domain** (Business, Technology, Science, Arts). One model, `sentence-transformers/all-MiniLM-L6-v2`, scores each guess in two steps:

1. **Domain gate** — top-k anchor cosine minus the domain's background on everyday nouns → *primary* (×1.00), *secondary* (×0.75), *weak secondary* (×0.50) or *blocked*.
2. **Target relevance** — `r = (cos − noise_floor) / (ceiling − noise_floor)` per target, mapped to damage by a calibrated curve (tiers: Critical, Strong, Solid, Graze).

**How to run:** set the flags in 0.1, then **Runtime → Run all**. On a fresh Colab runtime it finishes without errors and without waiting for input; the only interaction is the Google Drive pop-up (none when `USE_DRIVE = False`).

| Flag | Default | Meaning |
|---|---|---|
| `USE_DRIVE` | `True` | Persist tuned parameters, holdout log and checkpoints on Drive; otherwise local and session-only |
| `MODEL_CHOICE` | `"base"` | `"fine_tuned"` falls back to base when no checkpoint exists |
| `LOAD_TUNED_PARAMS` | `True` | Reuse a matching calibration file instead of re-running the grid searches |
| `RUN_FINE_TUNING` | `False` | Appendix: build training pairs and fine-tune |
| `RUN_HOLDOUT` | `True` | Section 7: evaluate the human-labelled holdout |
| `PLAY_INTERACTIVE` | `False` | Section 8: play with `input()`; otherwise one simulated battle |
| `PLAY_MODE` | `"random"` | `"random"` topic + target, or `"classic"` 3 stages |
| `SEED` | `42` | numpy / random seed, fine-tuning and random game |

**Sections:** 0 Setup · 1 Data registry · 2 Text utilities and data checks · 3 Engine · 4 Model selection · 5 Calibration · 6 Report · 7 Holdout · 8 Play · Appendix Fine-tuning · 9 Regression check and run summary · Play Game (last cell).

## 0. Setup
- **0.1** Run flags (the only cell you normally edit).
- **0.2** Installs only what Colab lacks (numpy / pandas / scipy / torch are never upgraded), all imports, seeds and the per-section timer.
- **0.3** Paths on Google Drive with a local fallback.

In [1]:
# 0.1 Flags — the only cell you normally edit. Then: Runtime → Run all.
# What this cell does:
#   - Run flags that control every section.
#   - Edit these, then use Runtime -> Run all.
USE_DRIVE = True            # mount Google Drive for files that must persist; False = local /content only
MODEL_CHOICE = "base"       # "base" | "fine_tuned" (falls back to base if no checkpoint exists)
LOAD_TUNED_PARAMS = True    # reuse saved calibration when a matching file exists (skips grid searches)
RUN_FINE_TUNING = False     # Appendix: rebuild training pairs and fine-tune (slow; new checkpoint)
RUN_HOLDOUT = True          # Section 7: evaluate the human-labelled holdout
PLAY_INTERACTIVE = False    # Section 8: True = type guesses; False = one simulated battle
PLAY_MODE = "random"        # "random" (random topic + target) | "classic" (3 fixed stages)
SEED = 42                   # numpy / random seed, fine-tuning seed and random-game seed

In [2]:
# 0.2 Installs (only what Colab lacks), imports, seeds, run tracker
# What this cell does:
#   - Installs only missing packages (sentence-transformers; nltk / wordfreq for fine-tuning).
#   - Pins numpy / pandas / scipy / torch so Colab never needs a restart.
#   - Imports every library and sets random seeds.
#   - Defines section() / log_section() to time each section for the run summary.
import importlib.metadata as _metadata
import subprocess
import sys

ST_REQUIREMENT = "sentence-transformers>=3.0,<6"     # pinned major range used by this notebook
FINE_TUNE_REQUIREMENTS = ["nltk", "wordfreq"]         # needed only by the Appendix
NEVER_UPGRADE = ["numpy", "pandas", "scipy", "torch"] # upgrading these forces a runtime restart

def _installed_version(package: str):
    try:
        return _metadata.version(package)
    except _metadata.PackageNotFoundError:
        return None

def _requirement_met(requirement: str) -> bool:
    from packaging.requirements import Requirement
    req = Requirement(requirement)
    version = _installed_version(req.name)
    return version is not None and req.specifier.contains(version, prereleases=True)

def pip_install(requirements: list) -> bool:
    """Install packages while pinning numpy/pandas/scipy/torch to the installed versions."""
    pins = [f"{p}=={_installed_version(p)}" for p in NEVER_UPGRADE if _installed_version(p)]
    constraints = "/tmp/lexical_constraints.txt"
    with open(constraints, "w") as handle:
        handle.write("\n".join(pins))
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *requirements, "-c", constraints],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f"WARNING: pip install {requirements} failed; continuing with installed packages. "
              f"{result.stderr.strip().splitlines()[-1] if result.stderr.strip() else ''}")
        return False
    return True

_needed = [] if _requirement_met(ST_REQUIREMENT) else [ST_REQUIREMENT]
if RUN_FINE_TUNING:
    _needed += [r for r in FINE_TUNE_REQUIREMENTS if not _requirement_met(r)]
if _needed:
    print("Installing:", _needed)
    pip_install(_needed)
else:
    print("All required packages are already installed.")

import hashlib
import html as _html
import itertools
import json
import os
import random
import time
import warnings
from contextlib import contextmanager
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.ndimage import uniform_filter
from sklearn.metrics import cohen_kappa_score, roc_auc_score
from sentence_transformers import SentenceTransformer

np.random.seed(SEED)
random.seed(SEED)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

# Run tracker: one entry per section, reset on every Run all (R8 summary).
RUN_LOG = []

class _SectionEntry(dict):
    def skip(self, reason: str) -> None:
        """Mark the section as skipped and say why."""
        self["status"], self["reason"] = "skipped", reason
        print(f"Skipped: {reason}")

@contextmanager
def section(name: str):
    """Time a section and record ran / skipped for the final summary."""
    entry = _SectionEntry(section=name, status="ran", reason="", seconds=0.0)
    RUN_LOG[:] = [e for e in RUN_LOG if e["section"] != name]   # idempotent re-runs
    RUN_LOG.append(entry)
    start = time.perf_counter()
    try:
        yield entry
    finally:
        entry["seconds"] = round(time.perf_counter() - start, 2)

def log_section(name: str, start: float) -> None:
    """Record a section that has no skip path (Sections 0–3)."""
    RUN_LOG[:] = [e for e in RUN_LOG if e["section"] != name]
    RUN_LOG.append(_SectionEntry(section=name, status="ran", reason="",
                                 seconds=round(time.perf_counter() - start, 2)))

RUN_START = time.perf_counter()
print(f"sentence-transformers {_installed_version('sentence-transformers')} | numpy {np.__version__} | "
      f"pandas {pd.__version__} | seed {SEED}")

All required packages are already installed.
sentence-transformers 5.7.0 | numpy 2.4.6 | pandas 2.2.3 | seed 42


In [3]:
# 0.3 Paths with Google Drive fallback
# What this cell does:
#   - Mounts Google Drive (falls back to local folders if it fails).
#   - Creates the artifacts / labels / models folders and lists old checkpoint locations.
DRIVE_ROOT = Path("/content/drive/MyDrive")

@dataclass(frozen=True)
class Paths:
    """Where files live. persistent=False means nothing here survives the session."""
    root: Path
    artifacts: Path        # tuned params, holdout run log
    labels: Path           # annotator_*.xlsx / .csv
    models: Path           # new fine-tuned checkpoints
    legacy_models: tuple   # older checkpoints to look for
    persistent: bool

def mount_drive(use_drive: bool) -> bool:
    """Try to mount Drive; return True only if MyDrive is reachable."""
    if not use_drive:
        print("WARNING: USE_DRIVE = False — using local paths; nothing will persist.")
        return False
    try:
        from google.colab import drive
    except ImportError:
        print("WARNING: google.colab is not available — using local paths; nothing will persist.")
        return False
    try:
        drive.mount("/content/drive")
    except Exception as error:   # mount refused / auth failed
        print(f"WARNING: Drive mount failed ({error}) — using local paths; nothing will persist.")
        return False
    if not DRIVE_ROOT.is_dir():
        print("WARNING: Drive mounted but MyDrive not found — using local paths; nothing will persist.")
        return False
    return True

def resolve_paths(use_drive: bool) -> Paths:
    """Drive paths when mounted, else local paths under /content (or the working folder)."""
    if mount_drive(use_drive):
        root = DRIVE_ROOT / "lexical_combat"
        paths = Paths(root, root / "artifacts", DRIVE_ROOT / "lexical_holdout_labels", root / "models",
                      (DRIVE_ROOT / "fine_tuned_lexical_minilm_v3", DRIVE_ROOT / "fine_tuned_lexical_minilm_v2"),
                      True)
    else:
        base = Path("/content") if Path("/content").is_dir() else Path.cwd()
        root = base / "lexical_combat"
        paths = Paths(root, root / "artifacts", root / "labels", root / "models", (), False)
    for folder in (paths.artifacts, paths.labels, paths.models):
        folder.mkdir(parents=True, exist_ok=True)
    return paths

PATHS = resolve_paths(USE_DRIVE)
PERSISTENT = PATHS.persistent
LABEL_SEARCH_DIRS = [PATHS.labels] + ([Path("/content")] if Path("/content").is_dir() else [])
print(f"Paths root: {PATHS.root} | PERSISTENT = {PERSISTENT}")
log_section("0. Setup", RUN_START)

Paths root: /content/lexical_combat | PERSISTENT = False


## 1. Data registry
Pure data, no model calls.
- **1.1** Domains, anchor sentences, example phrases, noise vocabulary and term pools.
- **1.2** Calibration cases, fresh gate vocabulary, holdout v2 and the old holdout (diagnostic only).
- **1.3** Game stages, boss names and the inline holdout annotations.

In [4]:
# 1.1 Domains, anchors and vocabularies (pure data, no model calls)
# What this cell does:
#   - Defines the 4 domains and their anchor sentences (used by the domain gate).
#   - Example phrases, out-of-domain cases and the original 19 test pairs.
#   - Noise vocabulary (everyday nouns) and curated term pools per domain.
_t_s1 = time.perf_counter()
DOMAINS = ["Business", "Technology", "Science", "Arts"]

# Curated phrase bank: example guesses per domain (gate data and term pools).
# Each entry is an example guess associated with one domain.
# These examples support demonstration and evaluation; expand them with human-reviewed data.
DOMAIN_EXAMPLES = {
    "Business": [
        "executive", "ceo", "director", "manager", "revenue", "profit",
        "business logic", "market strategy", "financial analysis",
        "shareholder value", "corporate leadership", "sales forecast"
    ],
    "Technology": [
        "algorithm", "software", "database", "network", "compiler",
        "logic gate", "machine learning", "source code", "cloud computing",
        "data structure", "neural network", "operating system"
    ],
    "Science": [
        "catalyst", "molecule", "quantum mechanics", "chemical reaction",
        "cell biology", "gravity", "atomic structure", "genetic mutation",
        "laboratory experiment", "thermodynamics", "electron", "ecosystem"
    ],
    "Arts": [
        "painting", "sculpture", "abstract painting", "classical music",
        "stage performance", "poetry", "portrait art", "modern dance",
        "cinematography", "watercolor technique", "orchestra", "theater"
    ]
}


# Explicit out-of-domain examples: plausible English phrases that do not belong to
# the stated domain. They help test whether the soft domain gate blocks irrelevant inputs.
OUT_OF_DOMAIN_CASES = [
    ("banana", "Business"), ("sonata", "Business"), ("quantum mechanics", "Business"),
    ("quarterly earnings", "Technology"), ("abstract painting", "Technology"),
    ("shareholder value", "Science"), ("database", "Arts"), ("chemical reaction", "Arts"),
    ("market strategy", "Science"), ("sculpture", "Technology")
]

# Original 19 pair-level tests; kept only so fine-tuning (Appendix) excludes them.
TEST_CASES = [
    # Business target: executive
    ("executive", "ceo", "Business", "in_domain", "strong"),
    ("executive", "director", "Business", "in_domain", "strong"),
    ("executive", "business logic", "Business", "in_domain", "medium"),
    ("executive", "logic", "Business", "ambiguous", "medium"),
    ("executive", "logic gate", "Business", "out_of_domain", "blocked"),
    ("executive", "banana", "Business", "out_of_domain", "blocked"),
    # Technology target: algorithm
    ("algorithm", "data structure", "Technology", "in_domain", "strong"),
    ("algorithm", "source code", "Technology", "in_domain", "medium"),
    ("algorithm", "logic gate", "Technology", "in_domain", "strong"),
    ("algorithm", "business logic", "Technology", "ambiguous", "medium"),
    ("algorithm", "logic", "Technology", "ambiguous", "medium"),
    ("algorithm", "abstract painting", "Technology", "out_of_domain", "blocked"),
    # Science target: catalyst
    ("catalyst", "chemical reaction", "Science", "in_domain", "strong"),
    ("catalyst", "molecule", "Science", "in_domain", "medium"),
    ("catalyst", "market strategy", "Science", "out_of_domain", "blocked"),
    ("catalyst", "quantum mechanics", "Science", "in_domain", "medium"),
    # Arts target: painting (evaluation only; game stages below use three domains)
    ("painting", "abstract painting", "Arts", "in_domain", "strong"),
    ("painting", "sculpture", "Arts", "in_domain", "medium"),
    ("painting", "database", "Arts", "out_of_domain", "blocked")
]

# Rich descriptions provide context beyond isolated category labels.
# V2 anchor bank: exactly 8 anchors per domain. The old bank had 8/10/4/3
# (Business/Technology/Science/Arts); with max-over-anchors scoring, domains
# with more anchors won unrelated words more often (banana -> Technology 40%).
# New coverage: digital logic (logic gate), software architecture (business
# logic layer), chemical kinetics (catalyst), and a full Arts set.
DOMAIN_ANCHORS = {
    "Business": [
        "Business finance accounting financial reporting quarterly report annual report balance sheet income statement cash flow revenue profit budget audit forecast",
        "Business management leadership executive chief executive officer CEO president director manager supervisor board of directors chairman team organization",
        "Business operations strategy planning human resources payroll hiring project management sales marketing procurement supply chain logistics customer service",
        "Commerce and markets include companies, corporations, startups, entrepreneurs, investors, shareholders, the stock market, trade, competition, mergers and acquisitions.",
        "Management roles include director, managing director, chief executive officer, chief financial officer, chief operating officer, department manager and team leader.",
        "Corporate governance covers the board of directors, executives, stakeholders, company policy, compliance, decision-making and business performance.",
        "Financial reporting covers quarterly reports, annual reports, financial statements, balance sheets, income statements, audits and revenue forecasts.",
        "Economics and finance include investment, banking, loans, interest, assets, liabilities, pricing, costs and profit margins."
    ],
    "Technology": [
        "Technology computing computer science software development programming source code debugging compiler API application database",
        "Algorithms and data structures array linked list stack queue tree graph hash table heap sorting searching recursion time complexity pseudocode",
        "Artificial intelligence AI machine learning deep learning neural network model training inference natural language processing computer vision",
        "Computer hardware motherboard CPU processor GPU RAM memory storage hard drive SSD circuit board peripherals input output devices",
        "Digital logic and electronics logic gate AND OR NOT XOR boolean logic binary circuit transistor flip-flop microcontroller embedded system",
        "Cybersecurity and networks encryption cryptography firewall router switch protocol IP address authentication malware vulnerability",
        "Software architecture includes the backend, the frontend, the business logic layer, the data access layer, microservices, APIs, cloud computing, servers and deployment.",
        "An algorithm is a finite sequence of computational instructions used to solve a problem or process data in a software system."
    ],
    "Science": [
        "Science scientific research physics chemistry biology experiments hypotheses theories measurements natural phenomena",
        "Chemistry studies molecules, atoms, elements, compounds, chemical reactions, chemical bonds, acids, bases and the periodic table.",
        "Chemical kinetics covers catalysts, enzymes, reaction rates, activation energy, reagents, oxidation, reduction, equilibrium and thermodynamics.",
        "Physics includes mechanics, gravity, electricity, magnetism, quantum mechanics, particles, electrons, forces, energy and motion.",
        "Biology includes genetics, DNA, cells, proteins, organisms, evolution, ecosystems, photosynthesis and molecular biology.",
        "Laboratory science uses microscopes, test tubes, beakers, samples, data collection, the scientific method, observation and peer review.",
        "Earth and space science covers geology, astronomy, planets, stars, climate, the atmosphere, oceans and natural forces.",
        "Science keywords atom molecule electron proton neutron catalyst enzyme gene cell energy force gravity laboratory experiment"
    ],
    "Arts": [
        "Arts visual art painting sculpture drawing illustration design creativity aesthetics artist gallery museum",
        "Painting is a visual art form that uses pigment, color, composition, canvas, brushwork and artistic expression.",
        "Music includes melody, harmony, rhythm, songs, composers, orchestras, symphonies, opera, concerts and instruments such as guitar and piano.",
        "The performing arts include theater, drama, acting, stage performance, ballet, modern dance, choreography and musicals.",
        "Literature and writing include poetry, novels, short stories, poems, authors, storytelling, fiction and literary style.",
        "Film and photography include cinema, movies, cinematography, the film director, screenplays, camera shots and photography.",
        "Art history and styles include the renaissance, impressionism, abstract art, portraits, landscapes, still life and modern art.",
        "Crafts and design include graphic design, fashion design, pottery, ceramics, calligraphy, printmaking and textile art."
    ]
}


# Everyday nouns with no game-domain sense. Words with a common technical,
# business, science or art meaning (apple, cookie, cloud, tree, window, plate,
# bucket, forest, lake, candle, zipper, ...) are deliberately left out.
NOISE_VOCAB = [
    "banana", "grape", "lemon", "mango", "peach", "cherry", "watermelon", "pineapple",
    "strawberry", "potato", "carrot", "onion", "tomato", "cabbage", "pumpkin", "rice",
    "bread", "cheese", "butter", "soup", "sandwich", "pizza", "noodle", "pancake",
    "coffee", "tea", "juice", "milk", "honey", "popcorn", "cupcake", "lollipop",
    "dog", "cat", "horse", "cow", "sheep", "rabbit", "duck", "frog", "tiger",
    "elephant", "giraffe", "monkey", "squirrel", "butterfly", "penguin", "dolphin",
    "turtle", "parrot", "chair", "sofa", "pillow", "blanket", "curtain", "carpet",
    "cup", "spoon", "fork", "kettle", "broom", "towel", "soap", "shampoo",
    "toothbrush", "shoe", "sock", "hat", "scarf", "glove", "jacket", "umbrella",
    "backpack", "bicycle", "skateboard", "kite", "doll", "teddy bear", "beach",
    "river", "mountain", "meadow", "desert", "island", "rain", "snow", "sunset",
    "rainbow", "garden", "flower", "grass", "sand", "pebble", "door", "roof",
    "staircase", "kitchen", "bathroom", "bedroom", "picnic", "holiday", "birthday",
    "wedding", "breakfast", "lunch", "dinner", "nap", "sneeze", "yawn", "hug",
    "playground", "sandcastle", "snowman", "balloon", "ladder", "fence", "mailbox",
    "doorbell", "slipper", "raincoat",
]

# Curated in-domain terms. Used for the per-target ceiling and for hints.
# Holdout guesses are intentionally NOT listed here.
DOMAIN_VOCAB_EXTRA = {
    "Business": [
        "chief executive officer", "president", "vice president", "chairman",
        "managing director", "board of directors", "executive officer", "boss",
        "leadership", "management", "administrator", "entrepreneur", "investor",
        "stakeholder", "budget", "salary", "merger", "acquisition", "marketing",
        "accounting", "invoice", "governance", "corporation", "headquarters",
    ],
    "Technology": [
        "program", "programming", "code", "sorting", "searching", "recursion",
        "pseudocode", "flowchart", "heuristic", "computation", "software engineering",
        "function", "loop", "binary search", "optimization", "processor", "router",
        "server", "artificial intelligence", "deep learning", "data mining",
        "automation", "cryptography", "debugging",
    ],
    "Science": [
        "enzyme", "reaction rate", "activation energy", "chemical compound",
        "oxidation", "reagent", "chemical bond", "acid", "hydrogen", "oxygen",
        "element", "photosynthesis", "evolution", "velocity", "energy", "dna",
        "hypothesis", "microscope", "periodic table", "accelerant", "laboratory",
        "compound",
    ],
    "Arts": [
        "canvas", "brushstroke", "oil painting", "watercolor", "fresco", "mural",
        "sketch", "drawing", "illustration", "artist", "painter", "gallery",
        "masterpiece", "color palette", "portrait", "landscape painting",
        "still life", "impressionism", "renaissance art", "ballet", "opera",
        "symphony", "novel", "photography", "composer",
    ],
}
# Broader general coverage (round 3). The old pools were built around the
# stage targets only, so new targets (manager, encryption, enzyme, melody) got
# a low ceiling and good guesses saturated at r = 1. These are general domain
# terms; holdout guesses are excluded automatically below.
DOMAIN_VOCAB_COVERAGE = {
    "Business": ["staff", "personnel", "recruitment", "workplace", "department", "organization",
                 "customer", "client", "sales", "logistics", "supply chain", "procurement",
                 "human resources", "performance review", "business plan"],
    "Technology": ["software development", "hardware", "network security", "authentication",
                   "malware", "firewall", "web development", "mobile app", "user interface",
                   "data analysis", "computer vision", "cybersecurity", "internet protocol",
                   "microchip", "operating system"],
    "Science": ["biochemistry", "molecular biology", "organic chemistry", "physiology",
                "microbiology", "cell", "gene", "organism", "atom", "particle", "magnetism",
                "nucleus", "chemical process", "glucose", "nutrient"],
    "Arts": ["musical note", "chord", "lyrics", "singer", "piano", "concert", "musician",
             "choreography", "poet", "sculptor", "film director", "sonata", "drama",
             "aesthetic", "pottery"],
}

# Broad field labels. Correct but lazy guesses -> generic penalty.
# The old R2 list also contained DOMAIN_SEEDS (director, manager, molecule,
# database, neural network, ...), which wrongly penalised the best answers.
GENERIC_TERMS_V2 = {
    "business", "commerce", "economy", "economics", "company", "industry", "finance",
    "money", "work", "job", "office", "technology", "tech", "computing", "computer",
    "computer science", "engineering", "digital", "internet", "it", "science",
    "scientific", "research", "chemistry", "physics", "biology", "nature",
    "art", "arts", "creativity", "culture", "music", "entertainment", "design",
}

# One target per domain: sets the median span floor and the damaging-leak check.
CALIBRATION_TARGETS = {"executive": "Business", "algorithm": "Technology",
                       "catalyst": "Science", "painting": "Arts"}

In [5]:
# 1.2 Calibration, holdout and diagnostic sets (pure data)
# What this cell does:
#   - Calibration cases: (target, guess, domain, gate label, damage tier).
#   - Fresh gate vocabulary that is NOT written in the anchors (avoids circular tuning).
#   - Holdout v2 (all-new targets) and the old holdout (diagnostic use only).
#   - Label overrides, cross-domain ambiguous pairs and reference plays for pacing.
# ------------------------------------------------------------
# 1. Pair-level calibration cases (target, guess, domain, gate label, tier)
# ------------------------------------------------------------
# Tiers: strong | medium | weak | blocked.
TEST_CASES_V2 = [
    ("executive", "ceo", "Business", "in_domain", "strong"),
    ("executive", "director", "Business", "in_domain", "strong"),
    ("executive", "business logic", "Business", "in_domain", "weak"),
    ("executive", "logic", "Business", "out_of_domain", "blocked"),
    ("executive", "logic gate", "Business", "out_of_domain", "blocked"),
    ("executive", "banana", "Business", "out_of_domain", "blocked"),
    ("algorithm", "data structure", "Technology", "in_domain", "strong"),
    ("algorithm", "source code", "Technology", "in_domain", "medium"),
    ("algorithm", "logic gate", "Technology", "in_domain", "medium"),
    ("algorithm", "business logic", "Technology", "ambiguous", "weak"),
    ("algorithm", "logic", "Technology", "ambiguous", "weak"),
    ("algorithm", "abstract painting", "Technology", "out_of_domain", "blocked"),
    ("catalyst", "chemical reaction", "Science", "in_domain", "strong"),
    ("catalyst", "molecule", "Science", "in_domain", "medium"),
    ("catalyst", "market strategy", "Science", "out_of_domain", "blocked"),
    ("catalyst", "quantum mechanics", "Science", "in_domain", "weak"),
    ("painting", "artwork", "Arts", "in_domain", "strong"),
    ("painting", "sculpture", "Arts", "in_domain", "medium"),
    ("painting", "database", "Arts", "out_of_domain", "blocked"),
]

# Extra banded cases so every calibration target has >= 8 cases (damage calibration, LOTO-CV).
# Not in the term pools, the holdout or the reference plays.
DAMAGE_CASES_EXTRA = [
    ("executive", "senior management", "Business", "in_domain", "strong"),
    ("executive", "company boss", "Business", "in_domain", "strong"),
    ("executive", "board member", "Business", "in_domain", "medium"),
    ("executive", "senior official", "Business", "in_domain", "medium"),
    ("executive", "office worker", "Business", "in_domain", "weak"),
    ("executive", "sales team", "Business", "in_domain", "weak"),
    ("executive", "sunflower", "Business", "out_of_domain", "blocked"),
    ("executive", "symphony", "Business", "out_of_domain", "blocked"),
    ("algorithm", "computer instructions", "Technology", "in_domain", "strong"),
    ("algorithm", "problem solving", "Technology", "in_domain", "medium"),
    ("algorithm", "graph theory", "Technology", "in_domain", "medium"),
    ("algorithm", "calculator", "Technology", "in_domain", "weak"),
    ("algorithm", "computer screen", "Technology", "in_domain", "weak"),
    ("algorithm", "dinosaur", "Technology", "out_of_domain", "blocked"),
    ("algorithm", "ballet", "Technology", "out_of_domain", "blocked"),
    ("catalyst", "reaction speed", "Science", "in_domain", "strong"),
    ("catalyst", "chemical agent", "Science", "in_domain", "medium"),
    ("catalyst", "platinum", "Science", "in_domain", "medium"),
    ("catalyst", "chemistry lab", "Science", "in_domain", "medium"),
    ("catalyst", "thermodynamics", "Science", "in_domain", "weak"),
    ("catalyst", "electron", "Science", "in_domain", "weak"),
    ("catalyst", "teacup", "Science", "out_of_domain", "blocked"),
    ("catalyst", "stock price", "Science", "out_of_domain", "blocked"),
    ("painting", "brush stroke", "Arts", "in_domain", "strong"),
    ("painting", "picture", "Arts", "in_domain", "medium"),
    ("painting", "art gallery", "Arts", "in_domain", "medium"),
    ("painting", "photograph", "Arts", "in_domain", "medium"),
    ("painting", "poem", "Arts", "in_domain", "weak"),
    ("painting", "dance", "Arts", "in_domain", "weak"),
    ("painting", "tractor", "Arts", "out_of_domain", "blocked"),
    ("painting", "algebra", "Arts", "out_of_domain", "blocked"),
]

# ------------------------------------------------------------
# 2. Fresh gate-calibration vocabulary (not written in DOMAIN_ANCHORS,
#    DOMAIN_EXAMPLES, the term pools or the holdout; V2 re-checks this)
# ------------------------------------------------------------
FRESH_GATE_IN_DOMAIN = {
    "Business": ["bankruptcy", "franchise", "wholesale", "retail", "bookkeeping", "inflation",
                 "recession", "advertising", "negotiation", "contract", "partnership",
                 "consultant", "tariff", "commission", "workforce", "pension", "inventory",
                 "monopoly", "layoff", "accountant"],
    "Technology": ["keyboard", "browser", "laptop", "smartphone", "bandwidth", "javascript",
                   "pixel", "bluetooth", "search engine", "website", "chatbot", "robotics",
                   "virtual reality", "blockchain", "byte", "touchscreen", "printer", "modem",
                   "antivirus"],
    "Science": ["volcano", "fossil", "telescope", "galaxy", "magnet", "bacteria", "chromosome",
                "mitochondria", "isotope", "radiation", "friction", "momentum", "wavelength",
                "solvent", "titration", "earthquake", "vaccine", "neuron", "photon",
                "black hole", "inertia"],
    "Arts": ["violin", "choir", "jazz", "novelist", "playwright", "sonnet", "comedy", "tragedy",
             "mosaic", "origami", "graffiti", "animation", "costume", "actor", "documentary",
             "ballerina", "stained glass", "woodcarving", "hip hop"],
}
# Fresh words whose cross-domain row is defensible -> 'ambiguous' instead of 'out_of_domain'.
FRESH_CROSS_AMBIGUOUS = {
    ("pixel", "Arts"), ("animation", "Technology"), ("blockchain", "Business"),
    ("robotics", "Science"), ("virtual reality", "Arts"), ("neuron", "Technology"),
    ("advertising", "Arts"), ("magnet", "Technology"),
}
# Cross-domain phrases a player could defend in the listed domain (should pass, x0.75 / x0.5).
FRESH_AMBIGUOUS_PAIRS = [
    ("film producer", "Business"), ("art auction", "Business"), ("music industry", "Business"),
    ("tech company", "Business"), ("software startup", "Business"),
    ("biotech startup", "Science"), ("data scientist", "Science"), ("rocket engine", "Science"),
    ("solar panel", "Science"), ("pharmaceutical company", "Science"),
    ("bioinformatics", "Technology"), ("medical device", "Technology"),
    ("game design", "Technology"), ("digital art", "Technology"), ("lab equipment", "Technology"),
    ("sound engineer", "Arts"), ("building architect", "Arts"), ("video game", "Arts"),
    ("web design", "Arts"), ("science museum", "Arts"), ("graphic tablet", "Arts"),
]
# Everyday words NOT in NOISE_VOCAB (that list sets the debias background), out for every domain.
FRESH_NOISE_OUT = ["lighthouse", "sailboat", "cactus", "pajamas", "volleyball", "marshmallow",
                   "waterfall", "hammock", "crocodile", "necklace", "flamingo", "haystack"]

# ------------------------------------------------------------
# 3. Holdout v2 — balanced design, 4 domains x 7 guesses, all-new targets
# ------------------------------------------------------------
# Per domain: 4 in-domain (1 strong, 2 medium, 1 weak), 1 ambiguous (weak),
# 2 out-of-domain (1 tempting distractor, 1 everyday word). Every target and
# guess is absent from DOMAIN_ANCHORS, DOMAIN_EXAMPLES, the term pools,
# NOISE_VOCAB, calibration sets, fine-tuning curated pairs and the old holdout.
# Labels below are the design intent; Section 7 replaces them with human labels.
HOLDOUT_CASES_DRAFT = [
    ("brand", "trademark", "Business", "in_domain", "strong"),
    ("brand", "logo", "Business", "in_domain", "medium"),
    ("brand", "customer loyalty", "Business", "in_domain", "medium"),
    ("brand", "packaging", "Business", "in_domain", "weak"),
    ("brand", "fashion label", "Business", "ambiguous", "weak"),
    ("brand", "cowboy", "Business", "out_of_domain", "blocked"),
    ("brand", "jellyfish", "Business", "out_of_domain", "blocked"),
    ("email", "inbox", "Technology", "in_domain", "strong"),
    ("email", "spam filter", "Technology", "in_domain", "medium"),
    ("email", "mail server", "Technology", "in_domain", "medium"),
    ("email", "chat app", "Technology", "in_domain", "weak"),
    ("email", "newsletter", "Technology", "ambiguous", "weak"),
    ("email", "postcard", "Technology", "out_of_domain", "blocked"),
    ("email", "seashell", "Technology", "out_of_domain", "blocked"),
    ("comet", "asteroid", "Science", "in_domain", "strong"),
    ("comet", "orbit", "Science", "in_domain", "medium"),
    ("comet", "solar system", "Science", "in_domain", "medium"),
    ("comet", "light year", "Science", "in_domain", "weak"),
    ("comet", "space shuttle", "Science", "ambiguous", "weak"),
    ("comet", "horoscope", "Science", "out_of_domain", "blocked"),
    ("comet", "doorknob", "Science", "out_of_domain", "blocked"),
    ("cartoon", "comic strip", "Arts", "in_domain", "strong"),
    ("cartoon", "caricature", "Arts", "in_domain", "medium"),
    ("cartoon", "animated film", "Arts", "in_domain", "medium"),
    ("cartoon", "puppet show", "Arts", "in_domain", "weak"),
    ("cartoon", "television channel", "Arts", "ambiguous", "weak"),
    ("cartoon", "stock chart", "Arts", "out_of_domain", "blocked"),
    ("cartoon", "toothpaste", "Arts", "out_of_domain", "blocked"),
]
HOLDOUT_SELECTION_NOTES = pd.DataFrame([
    ("brand", "target", "Not in any anchor/pool; has a second sense (cattle brand) that tests polysemy."),
    ("brand", "trademark", "Near-synonym: checks that a true synonym reaches the strong band without r saturation."),
    ("brand", "logo / customer loyalty", "Clearly related, not synonyms: one visual, one marketing outcome."),
    ("brand", "packaging", "Loose marketing link: should only graze."),
    ("brand", "fashion label", "Arts/fashion phrase with a real business sense: tests ambiguous -> secondary."),
    ("brand", "cowboy", "Tempting distractor via the cattle-brand sense: tests damaging leak."),
    ("brand", "jellyfish", "Everyday word: plain negative control."),
    ("email", "target", "Common tech word outside the anchors; tests a target with an everyday sense."),
    ("email", "inbox", "Near-synonym in use."),
    ("email", "spam filter / mail server", "Related components of an email system."),
    ("email", "chat app", "Another messaging technology: loose link."),
    ("email", "newsletter", "Business/writing sense, delivered by email: ambiguous."),
    ("email", "postcard", "Tempting distractor: physical mail, not technology."),
    ("email", "seashell", "Everyday word: plain negative control."),
    ("comet", "target", "Astronomy is the thinnest Science area in pools and anchors: tests coverage."),
    ("comet", "asteroid", "Sibling object: closest valid answer."),
    ("comet", "orbit / solar system", "Related concepts at two levels (motion, system)."),
    ("comet", "light year", "Astronomy unit: weak link to comets."),
    ("comet", "space shuttle", "Technology/engineering with a science sense: ambiguous."),
    ("comet", "horoscope", "Tempting distractor: astrology is not science."),
    ("comet", "doorknob", "Everyday word: plain negative control."),
    ("cartoon", "target", "Visual Arts outside painting/music, which dominate the calibration data."),
    ("cartoon", "comic strip", "Near-synonym form."),
    ("cartoon", "caricature / animated film", "Related art forms: drawing style and moving image."),
    ("cartoon", "puppet show", "Children's entertainment: loose link."),
    ("cartoon", "television channel", "Technology/business phrase with an entertainment sense: ambiguous."),
    ("cartoon", "stock chart", "Tempting distractor: a drawn chart, but finance."),
    ("cartoon", "toothpaste", "Everyday word: plain negative control."),
], columns=["target", "guess", "why chosen"])

# Old holdout (manager / encryption / enzyme / melody). It was inspected while
# fixing the gate, so it is now a DIAGNOSTIC set (Section 5.2), not a holdout.
OLD_HOLDOUT_DIAG_CASES = [
    ("manager", "supervisor", "Business", "in_domain", "strong"),
    ("manager", "team leader", "Business", "in_domain", "strong"),
    ("manager", "employee", "Business", "in_domain", "medium"),
    ("manager", "payroll", "Business", "in_domain", "weak"),
    ("manager", "spreadsheet", "Business", "ambiguous", "weak"),
    ("manager", "orchestra conductor", "Business", "out_of_domain", "blocked"),
    ("manager", "giraffe", "Business", "out_of_domain", "blocked"),
    ("manager", "firmware", "Business", "out_of_domain", "blocked"),
    ("encryption", "cipher", "Technology", "in_domain", "strong"),
    ("encryption", "private key", "Technology", "in_domain", "strong"),
    ("encryption", "password", "Technology", "in_domain", "medium"),
    ("encryption", "hacker", "Technology", "in_domain", "medium"),
    ("encryption", "secret message", "Technology", "ambiguous", "weak"),
    ("encryption", "tulip", "Technology", "out_of_domain", "blocked"),
    ("encryption", "stock market", "Technology", "out_of_domain", "blocked"),
    ("enzyme", "protein", "Science", "in_domain", "strong"),
    ("enzyme", "metabolism", "Science", "in_domain", "medium"),
    ("enzyme", "amino acid", "Science", "in_domain", "medium"),
    ("enzyme", "digestion", "Science", "in_domain", "medium"),
    ("enzyme", "fermentation", "Science", "in_domain", "weak"),
    ("enzyme", "guitar", "Science", "out_of_domain", "blocked"),
    ("enzyme", "dividend", "Science", "out_of_domain", "blocked"),
    ("melody", "harmony", "Arts", "in_domain", "strong"),
    ("melody", "tune", "Arts", "in_domain", "strong"),
    ("melody", "rhythm", "Arts", "in_domain", "medium"),
    ("melody", "song", "Arts", "in_domain", "medium"),
    ("melody", "frequency", "Arts", "ambiguous", "weak"),
    ("melody", "spreadsheet", "Arts", "out_of_domain", "blocked"),
]


RELABEL_NOTES = pd.DataFrame([
    ("executive / logic", "ambiguous · medium", "out_of_domain · blocked",
     "Bare 'logic' has no business sense; a player would not expect it to hurt a Business boss."),
    ("executive / business logic", "in_domain · medium", "in_domain · weak",
     "Business-flavoured but not about executives: graze, not a solid hit."),
    ("algorithm / logic gate", "in_domain · strong", "in_domain · medium",
     "Hardware building block, related to but not a synonym of 'algorithm'."),
    ("algorithm / logic, business logic", "ambiguous · medium", "ambiguous · weak",
     "Loose overlap only; should chip, not hit."),
    ("catalyst / quantum mechanics", "in_domain · medium", "in_domain · weak",
     "Correct domain, wrong sub-field: a graze is the fair outcome."),
    ("painting / abstract painting", "in_domain · strong", "painting / artwork",
     "'abstract painting' contains the target word and is now rejected as input."),
], columns=["Case", "Old label", "New label", "Reason"])

# Round 3: review of CROSS_DOMAIN_AMBIGUOUS. Policy: 'ambiguous' = a player can
# name a real, common sense in the stage domain -> should PASS as secondary.
RELABEL_NOTES_V3 = pd.DataFrame([
    ("director / Arts", "ambiguous", "ambiguous (keep, should pass)",
     "'film director' is a common Arts sense; secondary x0.75 is fair."),
    ("neural network / Science", "ambiguous", "ambiguous (keep)",
     "Computational neuroscience is a real science sense."),
    ("machine learning / Science", "ambiguous", "ambiguous (keep)",
     "Data science / scientific ML is a real sense."),
    ("electron / Technology", "ambiguous", "ambiguous (keep)", "Electronics."),
    ("algorithm / Science", "ambiguous", "ambiguous (keep)", "Computer science / computational science."),
    ("thermodynamics / Technology", "ambiguous", "ambiguous (keep)", "Engine and heat-system engineering."),
    ("network / Science", "ambiguous", "out_of_domain",
     "Bare 'network' points to computing; the science senses need a modifier."),
    ("cinematography / Technology", "ambiguous", "out_of_domain",
     "Camera craft is an Arts skill; the technology link is indirect."),
    ("ecosystem / Business", "ambiguous", "out_of_domain",
     "'business ecosystem' is jargon; bare 'ecosystem' is biology."),
], columns=["Case", "Old label", "New label", "Reason"])


# Per-hit damage bands in percent of boss HP. Tiers: Critical >=38, Strong >=26,
# Solid >=14, Graze <14. "strong" therefore accepts Strong or Critical.
TIER_BANDS_V2 = {"strong": (30, 45), "medium": (14, 30), "weak": (4, 14), "blocked": (0, 0)}


GATE_LABEL_OVERRIDES = {
    ("logic", "Technology"): "ambiguous", ("logic", "Business"): "out_of_domain",
    ("business logic", "Business"): "in_domain", ("business logic", "Technology"): "ambiguous",
    ("logic gate", "Technology"): "in_domain", ("logic gate", "Business"): "out_of_domain",
    ("strategy", "Business"): "in_domain", ("strategy", "Technology"): "ambiguous",
    ("model", "Science"): "ambiguous", ("model", "Arts"): "ambiguous",
}
CROSS_DOMAIN_AMBIGUOUS = {
    ("director", "Arts"), ("neural network", "Science"), ("machine learning", "Science"),
    ("electron", "Technology"), ("algorithm", "Science"), ("thermodynamics", "Technology"),
}
CROSS_DOMAIN_RELABELLED_OUT = {("network", "Science"), ("cinematography", "Technology"),
                               ("ecosystem", "Business")}


REFERENCE_PLAYS = {
    "executive": {
        "expert": (["ceo", "president", "managing director", "chairman"], (3, 4)),
        "typical": (["banana", "logic", "business logic", "market strategy", "director",
                     "manager", "leadership", "revenue", "ceo", "accounting"], (5, 8)),
    },
    "algorithm": {
        "expert": (["pseudocode", "sorting", "binary search", "recursion"], (3, 4)),
        "typical": (["banana", "logic", "programming", "source code", "software", "database",
                     "data structure", "sorting", "machine learning", "neural network"], (5, 8)),
    },
    "catalyst": {
        "expert": (["enzyme", "chemical reaction", "reaction rate", "activation energy"], (3, 4)),
        "typical": (["banana", "market strategy", "quantum mechanics", "molecule", "electron",
                     "thermodynamics", "chemistry", "chemical reaction", "gravity",
                     "reaction rate"], (5, 8)),
    },
}
PLAYER_LOG_GUESSES_V2 = {
    "executive": ["watermelon", "captain", "marketing", "management", "accounting",
                  "governor", "managing director", "leadership", "corporate officer", "ceo"],
    "algorithm": ["sorting", "programming", "hard drive", "symphony", "blueprint",
                  "software", "database", "neural network"],
    "catalyst": ["chemistry", "molecule", "electron", "thermodynamics", "chemical reaction"],
}

In [6]:
# 1.3 Game data and holdout annotations (pure data)
# What this cell does:
#   - The 3 fixed stages of classic mode and the boss names.
#   - Holdout annotations used as human labels in Section 7.

# The three fixed stages of classic mode.
GAME_STAGES = [
    {"stage": 1, "enemy": "Corporate Titan", "target": "executive", "domain": "Business", "hp": 180},
    {"stage": 2, "enemy": "Cyber Sentinel", "target": "algorithm", "domain": "Technology", "hp": 200},
    {"stage": 3, "enemy": "Alchemist Golem", "target": "catalyst", "domain": "Science", "hp": 200},
]

# Boss names per topic for random mode.
BOSS_NAMES = {
    "Business": ["Corporate Titan", "Boardroom Baron", "Market Hydra"],
    "Technology": ["Cyber Sentinel", "Kernel Wraith", "Firewall Golem"],
    "Science": ["Alchemist Golem", "Quantum Specter", "Nebula Warden"],
    "Arts": ["Muse Phantom", "Canvas Chimera", "Symphony Siren"],
}

# Holdout labels: (gate_label, tier) per item, same order as HOLDOUT_CASES_DRAFT.
# Files named annotator_<n>.xlsx / .csv (Section 7) override entries with the same name.
GATE_CHOICES = ["in_domain", "ambiguous", "out_of_domain"]
TIER_CHOICES = ["strong", "medium", "weak", "blocked"]
HOLDOUT_ANNOTATIONS = {
    # Annotator 1 (project owner), labelled independently on 2026-11-17 via annotator_1.xlsx.
    "annotator_1": [
        ("in_domain", "strong"),        #  0 brand / trademark
        ("in_domain", "strong"),        #  1 brand / logo
        ("in_domain", "medium"),        #  2 brand / customer loyalty
        ("in_domain", "medium"),        #  3 brand / packaging
        ("ambiguous", "weak"),          #  4 brand / fashion label
        ("out_of_domain", "blocked"),   #  5 brand / cowboy
        ("out_of_domain", "blocked"),   #  6 brand / jellyfish
        ("in_domain", "strong"),        #  7 email / inbox
        ("in_domain", "strong"),        #  8 email / spam filter
        ("in_domain", "strong"),        #  9 email / mail server
        ("in_domain", "medium"),        # 10 email / chat app
        ("in_domain", "medium"),        # 11 email / newsletter
        ("out_of_domain", "blocked"),   # 12 email / postcard
        ("out_of_domain", "blocked"),   # 13 email / seashell
        ("in_domain", "strong"),        # 14 comet / asteroid
        ("in_domain", "strong"),        # 15 comet / orbit
        ("in_domain", "medium"),        # 16 comet / solar system
        ("in_domain", "medium"),        # 17 comet / light year
        ("ambiguous", "weak"),          # 18 comet / space shuttle
        ("out_of_domain", "blocked"),   # 19 comet / horoscope
        ("out_of_domain", "blocked"),   # 20 comet / doorknob
        ("in_domain", "strong"),        # 21 cartoon / comic strip
        ("in_domain", "strong"),        # 22 cartoon / caricature
        ("in_domain", "strong"),        # 23 cartoon / animated film
        ("in_domain", "medium"),        # 24 cartoon / puppet show
        ("ambiguous", "weak"),          # 25 cartoon / television channel
        ("out_of_domain", "blocked"),   # 26 cartoon / stock chart
        ("out_of_domain", "blocked"),   # 27 cartoon / toothpaste
    ],
    "annotator_2": [None] * len(HOLDOUT_CASES_DRAFT),
    "annotator_3": [None] * len(HOLDOUT_CASES_DRAFT),
}
log_section("1. Data registry", _t_s1)

## 2. Text utilities and data checks
Still no model calls.
- **2.1** `clean_text()` normalisation, input validation (with the target-word rule) and anchor helpers.
- **2.2** Derived phrase sets, the non-circular gate calibration set and leakage checks.

In [7]:
# 2.1 Text utilities (pure functions, used by the engine and the data checks)
# What this cell does:
#   - clean_text(): lowercase, non-letters -> spaces.
#   - Helpers for duplicate checks, generic terms, term pools and input validation.
#   - anchor_domains_of(): finds which domains' anchors contain a phrase.
_t_s2 = time.perf_counter()

def clean_text(text: str) -> str:
    """Lowercase, replace non-letters with spaces, collapse spaces ('data-base' -> 'data base')."""
    text = str(text).lower().strip()
    text = "".join(ch if (ch.isalpha() or ch.isspace()) else " " for ch in text)
    return " ".join(text.split())

def duplicate_key(text: str) -> str:
    """Key that ignores word order and simple plurals ('business logics' == 'logic business')."""
    words = []
    for word in clean_text(text).split():
        if len(word) > 3 and word.endswith("s") and not word.endswith("ss"):
            word = word[:-1]
        words.append(word)
    return " ".join(sorted(words))

def word_keys(text: str) -> set:
    """Normalised word keys ('ceos' -> 'ceo') used for target-word checks."""
    return {duplicate_key(w) for w in clean_text(text).split()}

def is_generic_term(cleaned_guess: str, generic_terms) -> bool:
    """True for broad field labels such as 'science' or 'business'."""
    return (cleaned_guess in generic_terms
            or duplicate_key(cleaned_guess) in {duplicate_key(t) for t in generic_terms})

def domain_term_pool(domain: str, examples: dict, vocab: dict, generic_terms) -> list:
    """De-duplicated, non-generic in-domain terms for one domain."""
    out, seen = [], set()
    for term in list(examples.get(domain, [])) + vocab.get(domain, []):
        cleaned = clean_text(term)
        if cleaned and cleaned not in seen and not is_generic_term(cleaned, generic_terms):
            seen.add(cleaned)
            out.append(cleaned)
    return out

def validate_player_input(guess: str, target: str, used_guesses, block_target_word: bool = True) -> tuple:
    """Clean and validate a 1–3 word guess. Returns (is_valid, cleaned, reason)."""
    cleaned = clean_text(guess)
    target_clean = clean_text(target)
    if not cleaned:
        return False, cleaned, "Please enter 1–3 English words."
    if not 1 <= len(cleaned.split()) <= 3:
        return False, cleaned, "Your guess must contain between 1 and 3 words."
    if duplicate_key(cleaned) == duplicate_key(target_clean):
        return False, cleaned, "Your guess cannot be the target word itself."
    if block_target_word and (word_keys(cleaned) & word_keys(target_clean)):
        return False, cleaned, "Your guess cannot contain the target word."
    if cleaned in used_guesses:
        return False, cleaned, f"'{cleaned}' was already used in this battle."
    if duplicate_key(cleaned) in {duplicate_key(u) for u in used_guesses}:
        return False, cleaned, f"'{cleaned}' is too similar to a guess already used in this battle."
    return True, cleaned, ""

def norm_tokens(text: str) -> list:
    """Lowercase tokens with simple plural stripping ('reactions' -> 'reaction')."""
    out = []
    for word in clean_text(text).split():
        if len(word) > 3 and word.endswith("s") and not word.endswith("ss"):
            word = word[:-1]
        out.append(word)
    return out

ANCHOR_TOKEN_SEQS = {d: [norm_tokens(a) for a in DOMAIN_ANCHORS[d]] for d in DOMAINS}

def anchor_domains_of(phrase: str) -> list:
    """Domains whose anchor text contains the phrase as a contiguous token run."""
    toks = norm_tokens(phrase)
    hits = []
    for d, seqs in ANCHOR_TOKEN_SEQS.items():
        if any(seq[i:i + len(toks)] == toks for seq in seqs for i in range(len(seq) - len(toks) + 1)):
            hits.append(d)
    return hits

In [8]:
# 2.2 Derived phrase sets, gate calibration set and leakage checks (no model calls)
# What this cell does:
#   - Builds term pools without any holdout phrase.
#   - Builds the gate calibration set (in-domain / ambiguous / out-of-domain phrases).
#   - Runs leakage checks and prints the set sizes.

# Term pools: curated extras + general coverage, minus every holdout phrase.
HOLDOUT_PHRASES = {clean_text(p) for case in HOLDOUT_CASES_DRAFT + OLD_HOLDOUT_DIAG_CASES for p in case[:2]}
POOL_VOCAB = {d: DOMAIN_VOCAB_EXTRA[d] + [t for t in DOMAIN_VOCAB_COVERAGE[d] if clean_text(t) not in HOLDOUT_PHRASES]
              for d in DOMAINS}
CALIBRATION_PHRASES = (
    {clean_text(p) for case in TEST_CASES_V2 + DAMAGE_CASES_EXTRA for p in case[:2]}
    | {clean_text(p) for terms in FRESH_GATE_IN_DOMAIN.values() for p in terms}
    | {clean_text(p) for p, _ in FRESH_AMBIGUOUS_PAIRS}
    | {clean_text(p) for p in FRESH_NOISE_OUT}
)
EVAL_PHRASES = HOLDOUT_PHRASES | CALIBRATION_PHRASES   # excluded from fine-tuning (Appendix)

test_df_v2 = pd.DataFrame(TEST_CASES_V2, columns=[
    "target", "guess", "required_domain", "human_label", "human_expected_tier"])
damage_cases_df = pd.DataFrame(TEST_CASES_V2 + DAMAGE_CASES_EXTRA, columns=[
    "target", "guess", "required_domain", "human_label", "human_expected_tier"])
DOMAIN_REFERENCE_TARGET = {d: t for t, d in CALIBRATION_TARGETS.items()}

# Fresh vocabulary: drop any term that also appears in anchors, examples, holdout, pools or noise.
_examples = {clean_text(p) for ps in DOMAIN_EXAMPLES.values() for p in ps}
_pool_terms = {t for d in DOMAINS for t in domain_term_pool(d, DOMAIN_EXAMPLES, POOL_VOCAB, GENERIC_TERMS_V2)}
_noise_set = {clean_text(w) for w in NOISE_VOCAB}

def fresh_violation(phrase: str):
    """Reason a fresh calibration term overlaps another set, else None."""
    p = clean_text(phrase)
    if anchor_domains_of(p): return "in anchors"
    if p in _examples: return "in DOMAIN_EXAMPLES"
    if p in HOLDOUT_PHRASES: return "in holdout"
    if p in _pool_terms: return "in term pool"
    if p in _noise_set: return "in NOISE_VOCAB"
    return None

fresh_dropped = []
FRESH_IN_DOMAIN_CLEAN = {}
for d, terms in FRESH_GATE_IN_DOMAIN.items():
    FRESH_IN_DOMAIN_CLEAN[d] = []
    for t in terms:
        why = fresh_violation(t)
        if why:
            fresh_dropped.append((t, d, why))
        else:
            FRESH_IN_DOMAIN_CLEAN[d].append(clean_text(t))
FRESH_AMBIGUOUS_CLEAN = [(clean_text(p), d) for p, d in FRESH_AMBIGUOUS_PAIRS if not fresh_violation(p)]
FRESH_NOISE_CLEAN = [clean_text(w) for w in FRESH_NOISE_OUT if not fresh_violation(w)]
fresh_dropped += [(p, d, fresh_violation(p)) for p, d in FRESH_AMBIGUOUS_PAIRS if fresh_violation(p)]
fresh_dropped += [(w, "all", fresh_violation(w)) for w in FRESH_NOISE_OUT if fresh_violation(w)]
for d, terms in FRESH_IN_DOMAIN_CLEAN.items():
    assert len(terms) >= 15, f"{d}: only {len(terms)} fresh terms left; add more in Section 1.2."

def build_gate_calibration_set() -> pd.DataFrame:
    """One label per (phrase, domain); later sources override earlier ones."""
    labels, sources = {}, {}
    def put(phrase, domain, label, source):
        labels[(clean_text(phrase), domain)] = label
        sources[(clean_text(phrase), domain)] = source
    for domain, phrases in DOMAIN_EXAMPLES.items():          # lowest priority
        for phrase in phrases:
            for other in DOMAINS:
                put(phrase, other, "in_domain" if other == domain else "out_of_domain",
                    "DOMAIN_EXAMPLES")
    for phrase, domain in OUT_OF_DOMAIN_CASES:
        put(phrase, domain, "out_of_domain", "OUT_OF_DOMAIN_CASES")
    for phrase, domain in CROSS_DOMAIN_AMBIGUOUS:
        put(phrase, domain, "ambiguous", "CROSS_DOMAIN_AMBIGUOUS")
    for phrase, domain in CROSS_DOMAIN_RELABELLED_OUT:
        put(phrase, domain, "out_of_domain", "CROSS_DOMAIN_RELABELLED")
    for (phrase, domain), label in GATE_LABEL_OVERRIDES.items():
        put(phrase, domain, label, "GATE_LABEL_OVERRIDES")
    for domain, terms in FRESH_IN_DOMAIN_CLEAN.items():
        for term in terms:
            for other in DOMAINS:
                if other == domain:
                    put(term, other, "in_domain", "FRESH_IN_DOMAIN")
                elif (term, other) in FRESH_CROSS_AMBIGUOUS:
                    put(term, other, "ambiguous", "FRESH_CROSS_AMBIGUOUS")
                else:
                    put(term, other, "out_of_domain", "FRESH_CROSS_OUT")
    for phrase, domain in FRESH_AMBIGUOUS_CLEAN:
        put(phrase, domain, "ambiguous", "FRESH_AMBIGUOUS")
    for word in FRESH_NOISE_CLEAN:
        for domain in DOMAINS:
            put(word, domain, "out_of_domain", "FRESH_NOISE_OUT")
    for _, guess, domain, label, _ in TEST_CASES_V2 + DAMAGE_CASES_EXTRA:   # highest priority
        put(guess, domain, label, "BANDED_CASES")
    frame = pd.DataFrame([{"phrase": p, "domain": d, "label": l, "source": sources[(p, d)]}
                          for (p, d), l in labels.items()])
    frame["anchor_domains"] = frame["phrase"].map(lambda p: ",".join(anchor_domains_of(p)))
    frame["anchor_echo"] = frame["anchor_domains"] != ""
    return frame

gate_set_all_df = build_gate_calibration_set()
gate_set_df = gate_set_all_df[~gate_set_all_df["anchor_echo"]].reset_index(drop=True)   # tuning
anchor_echo_df = gate_set_all_df[gate_set_all_df["anchor_echo"]].reset_index(drop=True)  # sanity only

print(f"Registry: {len(TEST_CASES_V2) + len(DAMAGE_CASES_EXTRA)} banded cases | "
      f"{sum(map(len, FRESH_GATE_IN_DOMAIN.values()))} fresh in-domain terms | "
      f"{len(FRESH_AMBIGUOUS_PAIRS)} fresh ambiguous pairs | {len(HOLDOUT_CASES_DRAFT)} holdout cases")
print("Holdout per domain:", pd.Series([d for _, _, d, _, _ in HOLDOUT_CASES_DRAFT]).value_counts().to_dict())
_leaks = sorted({g for _, g, _, _, _ in HOLDOUT_CASES_DRAFT
                 if clean_text(g) in _pool_terms or clean_text(g) in set(gate_set_all_df["phrase"])})
print("Relabelled TEST_CASES (round 2):")
display(RELABEL_NOTES)
print("CROSS_DOMAIN_AMBIGUOUS review (round 3):")
display(RELABEL_NOTES_V3)
print("Banded cases per target (need >= 8):")
display(damage_cases_df.groupby(["target", "human_expected_tier"]).size().unstack(fill_value=0)
        .assign(total=lambda f: f.sum(axis=1)))
print(f"Gate set: {len(gate_set_all_df)} rows -> tuning {len(gate_set_df)} | anchor-echo (sanity only) "
      f"{len(anchor_echo_df)}")
display(gate_set_df.groupby(["label", "source"]).size().rename("rows").reset_index())
print("Fresh in-domain terms per domain:", {d: len(t) for d, t in FRESH_IN_DOMAIN_CLEAN.items()})
if fresh_dropped:
    print("Fresh terms dropped by the overlap check:", fresh_dropped)
_anchor_holdout = [g for _, g, _, _, _ in HOLDOUT_CASES_DRAFT if anchor_domains_of(g)]
print(f"Holdout leakage into pools / gate set: {_leaks if _leaks else 'none'} | "
      f"holdout guesses written in anchors: {_anchor_holdout if _anchor_holdout else 'none'}")
print("Holdout design (why each word was chosen):")
display(HOLDOUT_SELECTION_NOTES)
log_section("2. Text utilities and checks", _t_s2)

Registry: 50 banded cases | 79 fresh in-domain terms | 21 fresh ambiguous pairs | 28 holdout cases
Holdout per domain: {'Business': 7, 'Technology': 7, 'Science': 7, 'Arts': 7}
Relabelled TEST_CASES (round 2):


,Case,Old label,New label,Reason
0,executive / logic,ambiguous · medium,out_of_domain · blocked,Bare 'logic' has no business sense; a player would not expect it to hurt a Business boss.
1,executive / business logic,in_domain · medium,in_domain · weak,"Business-flavoured but not about executives: graze, not a solid hit."
2,algorithm / logic gate,in_domain · strong,in_domain · medium,"Hardware building block, related to but not a synonym of 'algorithm'."
3,"algorithm / logic, business logic",ambiguous · medium,ambiguous · weak,"Loose overlap only; should chip, not hit."
4,catalyst / quantum mechanics,in_domain · medium,in_domain · weak,"Correct domain, wrong sub-field: a graze is the fair outcome."
5,painting / abstract painting,in_domain · strong,painting / artwork,'abstract painting' contains the target word and is now rejected as input.


CROSS_DOMAIN_AMBIGUOUS review (round 3):


,Case,Old label,New label,Reason
0,director / Arts,ambiguous,"ambiguous (keep, should pass)",'film director' is a common Arts sense; secondary x0.75 is fair.
1,neural network / Science,ambiguous,ambiguous (keep),Computational neuroscience is a real science sense.
2,machine learning / Science,ambiguous,ambiguous (keep),Data science / scientific ML is a real sense.
3,electron / Technology,ambiguous,ambiguous (keep),Electronics.
4,algorithm / Science,ambiguous,ambiguous (keep),Computer science / computational science.
5,thermodynamics / Technology,ambiguous,ambiguous (keep),Engine and heat-system engineering.
6,network / Science,ambiguous,out_of_domain,Bare 'network' points to computing; the science senses need a modifier.
7,cinematography / Technology,ambiguous,out_of_domain,Camera craft is an Arts skill; the technology link is indirect.
8,ecosystem / Business,ambiguous,out_of_domain,'business ecosystem' is jargon; bare 'ecosystem' is biology.


Banded cases per target (need >= 8):


human_expected_tier,blocked,medium,strong,weak,total
target,,,,,
algorithm,3,4,2,4,13
catalyst,3,4,2,3,12
executive,5,2,4,3,14
painting,3,4,2,2,11


Gate set: 616 rows -> tuning 467 | anchor-echo (sanity only) 149


,label,source,rows
0,ambiguous,FRESH_AMBIGUOUS,21
1,ambiguous,FRESH_CROSS_AMBIGUOUS,8
2,in_domain,BANDED_CASES,20
3,in_domain,DOMAIN_EXAMPLES,13
4,in_domain,FRESH_IN_DOMAIN,79
5,out_of_domain,BANDED_CASES,10
6,out_of_domain,DOMAIN_EXAMPLES,36
7,out_of_domain,FRESH_CROSS_OUT,229
8,out_of_domain,FRESH_NOISE_OUT,48
9,out_of_domain,OUT_OF_DOMAIN_CASES,3


Fresh in-domain terms per domain: {'Business': 20, 'Technology': 19, 'Science': 21, 'Arts': 19}
Holdout leakage into pools / gate set: none | holdout guesses written in anchors: none
Holdout design (why each word was chosen):


,target,guess,why chosen
0,brand,target,Not in any anchor/pool; has a second sense (cattle brand) that tests polysemy.
1,brand,trademark,Near-synonym: checks that a true synonym reaches the strong band without r saturation.
2,brand,logo / customer loyalty,"Clearly related, not synonyms: one visual, one marketing outcome."
3,brand,packaging,Loose marketing link: should only graze.
4,brand,fashion label,Arts/fashion phrase with a real business sense: tests ambiguous -> secondary.
5,brand,cowboy,Tempting distractor via the cattle-brand sense: tests damaging leak.
6,brand,jellyfish,Everyday word: plain negative control.
7,email,target,Common tech word outside the anchors; tests a target with an everyday sense.
8,email,inbox,Near-synonym in use.
9,email,spam filter / mail server,Related components of an email system.


## 3. Engine
`EngineConfig` holds every tunable parameter with its pre-calibration default. `LexicalCombatEngine` owns the model, anchor embeddings, backgrounds, term pools, target cache and span floor; its methods never touch notebook globals. The engine only calls `model.encode(...)`, so any encoder with the `SentenceTransformer.encode` signature can be plugged in.

In [9]:
# 3. Engine: EngineConfig (every tunable parameter) + LexicalCombatEngine (all scoring)
# What this cell does:
#   - EngineConfig: every tunable parameter (gate thresholds, damage curve, tiers).
#   - LexicalCombatEngine: domain gate, target relevance, damage and battle simulation.
#   - evaluate_test_set(): runs a table of cases through the engine.
_t_s3 = time.perf_counter()

@dataclass(frozen=True)
class EngineConfig:
    """Tunable parameters. Defaults are the pre-calibration values; Section 5 replaces some."""
    # Domain gate (debiased score = top-k anchor cosine minus the domain's noise background)
    anchor_topk: int = 2                     # mean of the k best anchors per domain
    gate_primary_min: float = 0.20           # primary: required domain is #1 and >= this
    gate_secondary_min: float = 0.08         # secondary / weak: required domain >= this
    gate_secondary_max_gap: float = 0.04     # secondary: may trail the #1 domain by <= this
    gate_weak_max_gap: float = 0.10          # weak secondary: may trail by <= this
    domain_primary_bonus: float = 1.00       # damage multiplier per gate result
    domain_secondary_multiplier: float = 0.75
    domain_weak_secondary_multiplier: float = 0.50
    display_softmax_temperature: float = 0.05  # display only, not used for decisions
    # Target relevance r = (cos - noise) / (ceiling - noise)
    noise_percentile: int = 90               # noise floor = percentile of cos(target, everyday nouns)
    ceiling_topk: int = 3                    # ceiling = mean of the top-k pool cosines
    min_calibration_span: float = 0.15       # fixed span floor (min_span_mode == "fixed")
    min_span_mode: str = "median"            # "median": floor = median span of calibration targets
    primary_graze_floor: bool = True         # primary guess above median noise gets chip damage
    primary_graze_noise_pct: int = 50
    # Damage curve on r in [0, 1]
    relevance_min: float = 0.10              # r below this -> no effect
    damage_gamma: float = 1.0                # >1 stricter mid-range, <1 more generous
    graze_min_percent: float = 5             # damage at r == relevance_min
    max_hit_percent: float = 45              # damage at r == 1 (per-hit cap)
    chip_percent: float = 4                  # minimum damage of an allowed, relevant hit
    # Penalties and input rules
    generic_penalty: float = 0.60            # broad field labels ("science")
    echo_relative: float = 0.60              # echo if cos to an earlier hit is this far from noise to 1.0
    echo_penalty: float = 0.50
    block_target_word_in_guess: bool = True  # a guess may not contain any word of the target
    tier_cuts: tuple = ((38, "CRITICAL STRIKE"), (26, "STRONG HIT"), (14, "SOLID HIT"))

@dataclass(frozen=True)
class EngineData:
    """Vocabularies the engine encodes. Built once from Sections 1–2."""
    domains: tuple
    anchors: dict
    noise_vocab: tuple
    examples: dict
    pool_vocab: dict
    generic_terms: frozenset
    calibration_targets: dict

GATE_LABEL_OF_KIND = {"primary": "in_domain", "secondary": "ambiguous",
                      "weak_secondary": "ambiguous", "out_of_domain": "out_of_domain"}

def kinds_to_labels(kinds) -> np.ndarray:
    """Map gate kinds to the human label vocabulary."""
    kinds = np.asarray(kinds)
    return np.where(kinds == "primary", "in_domain",
                    np.where(kinds == "out_of_domain", "out_of_domain", "ambiguous"))

def topk_mean(similarities, k: int):
    """Mean of the k largest values along the last axis."""
    sims = np.asarray(similarities, dtype=float)
    k = max(1, min(int(k), sims.shape[-1]))
    return np.sort(sims, axis=-1)[..., -k:].mean(axis=-1)

def normalized_relevance(similarity, noise, ceiling):
    """r = (cos - noise) / (ceiling - noise), clipped to [0, 1]."""
    return np.clip((np.asarray(similarity, dtype=float) - noise) / (ceiling - noise), 0.0, 1.0)

def gate_kind_arrays(required, lead, primary_min, secondary_min, secondary_gap, weak_gap) -> np.ndarray:
    """Vectorised gate. lead = required score - best other domain score."""
    required = np.asarray(required, dtype=float)
    lead = np.asarray(lead, dtype=float)
    is_primary = (required >= primary_min) & (lead >= 0)
    passes = required >= secondary_min
    is_secondary = ~is_primary & passes & (lead >= -secondary_gap)
    is_weak = ~is_primary & ~is_secondary & passes & (lead >= -weak_gap)
    return np.select([is_primary, is_secondary, is_weak],
                     ["primary", "secondary", "weak_secondary"], "out_of_domain")

def timed_call(function, *args, **kwargs):
    """Run a function; return (result, elapsed milliseconds)."""
    start = time.perf_counter()
    result = function(*args, **kwargs)
    return result, (time.perf_counter() - start) * 1000

class LexicalCombatEngine:
    """Owns the model, anchor embeddings, backgrounds, pools, target cache and span floor."""

    def __init__(self, model, data: EngineData, config: EngineConfig = EngineConfig()):
        self.data = data
        self._config = config
        self.refresh(model)

    # ---- configuration -------------------------------------------------
    @property
    def config(self) -> EngineConfig:
        return self._config

    @config.setter
    def config(self, new_config: EngineConfig) -> None:
        """Apply new parameters; the span floor depends on them, so recompute it."""
        self._config = new_config
        self.span_floor = self.resolve_span_floor()

    # ---- encoding and engine state --------------------------------------
    def encode(self, texts) -> np.ndarray:
        """L2-normalised sentence vectors."""
        return self.model.encode(texts, convert_to_numpy=True, normalize_embeddings=True,
                                 show_progress_bar=False)

    def refresh(self, model=None) -> None:
        """(Re)encode anchors, noise words and term pools with the given model."""
        if model is not None:
            self.model = model
        d = self.data
        texts, owners = [], []
        for domain, descriptions in d.anchors.items():
            for description in descriptions:
                texts.append(description)
                owners.append(domain)
        vectors = self.encode(texts)
        self.anchor_vectors = {dom: vectors[[i for i, o in enumerate(owners) if o == dom]] for dom in d.domains}
        self.noise_vectors = self.encode(list(d.noise_vocab))
        self.background_by_k = {
            k: {dom: float(topk_mean(self.noise_vectors @ self.anchor_vectors[dom].T, k).mean())
                for dom in d.domains}
            for k in (1, 2, 3, 4, 5)
        }
        self.pools = {}
        for dom in d.domains:
            pool = domain_term_pool(dom, d.examples, d.pool_vocab, d.generic_terms)
            self.pools[dom] = (pool, self.encode(pool))
        self.targets = {}
        pair_sims = self.noise_vectors @ self.noise_vectors.T
        upper = pair_sims[np.triu_indices(len(d.noise_vocab), k=1)]
        self.noise_pair_mean = float(upper.mean())     # anisotropy check
        self.noise_pair_p90 = float(np.percentile(upper, 90))
        self.span_floor = self.resolve_span_floor()

    @property
    def background(self) -> dict:
        return self.background_by_k[self.config.anchor_topk]

    # ---- domain gate ------------------------------------------------------
    def domain_scores(self, vectors, k: int = None):
        """Debiased (n, 4) and raw (n, 4) domain scores at top-k."""
        k = self.config.anchor_topk if k is None else int(k)
        vectors = np.atleast_2d(vectors)
        raw = np.column_stack([topk_mean(vectors @ self.anchor_vectors[dom].T, k) for dom in self.data.domains])
        background = np.array([self.background_by_k[k][dom] for dom in self.data.domains])
        return raw - background, raw

    def domain_scores_from_vector(self, vector, k: int = None):
        """(debiased scores, raw scores, display probabilities, nearest anchors) for one vector."""
        k = self.config.anchor_topk if k is None else int(k)
        raw, adjusted, nearest = {}, {}, {}
        for dom in self.data.domains:
            sims = self.anchor_vectors[dom] @ vector
            raw[dom] = float(topk_mean(sims, k))
            adjusted[dom] = raw[dom] - self.background_by_k[k][dom]
            nearest[dom] = self.data.anchors[dom][int(np.argmax(sims))]
        values = np.array([adjusted[dom] for dom in self.data.domains])
        weights = np.exp((values - values.max()) / self.config.display_softmax_temperature)
        probabilities = dict(zip(self.data.domains, (weights / weights.sum()).astype(float)))
        return adjusted, raw, probabilities, nearest

    def gate_multipliers(self) -> dict:
        c = self.config
        return {"primary": c.domain_primary_bonus, "secondary": c.domain_secondary_multiplier,
                "weak_secondary": c.domain_weak_secondary_multiplier, "out_of_domain": 0.0}

    def gate_kinds(self, required, lead) -> np.ndarray:
        """Gate kinds with the current thresholds."""
        c = self.config
        return gate_kind_arrays(required, lead, c.gate_primary_min, c.gate_secondary_min,
                                c.gate_secondary_max_gap, c.gate_weak_max_gap)

    def gate(self, domain_scores: dict, required_domain: str) -> tuple:
        """Soft gate on debiased scores. Returns (allowed, multiplier, kind)."""
        if required_domain not in domain_scores:
            raise ValueError(f"Unknown domain: {required_domain}")
        required = float(domain_scores[required_domain])
        lead = required - max(v for dom, v in domain_scores.items() if dom != required_domain)
        kind = str(self.gate_kinds([required], [lead])[0])
        return kind != "out_of_domain", self.gate_multipliers()[kind], kind

    # ---- target relevance ---------------------------------------------------
    def target_calibration(self, target: str, domain: str) -> dict:
        """Cached target vector plus its cosines to noise words and the domain pool."""
        key = (clean_text(target), domain)
        if key not in self.targets:
            target_clean = clean_text(target)
            vector = self.encode([target_clean])[0]
            pool, pool_vectors = self.pools[domain]
            target_keys = word_keys(target_clean)
            keep = [i for i, term in enumerate(pool) if not (word_keys(term) & target_keys)]
            self.targets[key] = {
                "vector": vector,
                "noise_sims": self.noise_vectors @ vector,
                "pool_sims": pool_vectors[keep] @ vector,
                "pool_terms": [pool[i] for i in keep],
            }
        return self.targets[key]

    def raw_bounds(self, cal: dict, noise_percentile=None, ceiling_topk=None) -> tuple:
        """(noise_floor, ceiling) before the span floor."""
        noise_percentile = self.config.noise_percentile if noise_percentile is None else noise_percentile
        ceiling_topk = self.config.ceiling_topk if ceiling_topk is None else ceiling_topk
        return (float(np.percentile(cal["noise_sims"], noise_percentile)),
                float(topk_mean(cal["pool_sims"], ceiling_topk)))

    def median_calibration_span(self, noise_percentile=None, ceiling_topk=None) -> float:
        """Median raw span (ceiling - noise) over the calibration targets."""
        spans = []
        for target, domain in self.data.calibration_targets.items():
            noise, ceiling = self.raw_bounds(self.target_calibration(target, domain), noise_percentile, ceiling_topk)
            spans.append(ceiling - noise)
        return float(np.median(spans))

    def resolve_span_floor(self) -> float:
        if self.config.min_span_mode == "median":
            return max(self.median_calibration_span(), 0.05)
        return self.config.min_calibration_span

    def bounds(self, cal: dict, noise_percentile=None, ceiling_topk=None, span_floor=None) -> tuple:
        """(noise_floor, ceiling) with ceiling >= noise + span floor."""
        noise, ceiling = self.raw_bounds(cal, noise_percentile, ceiling_topk)
        span_floor = self.span_floor if span_floor is None else span_floor
        return noise, max(ceiling, noise + span_floor)

    relevance = staticmethod(normalized_relevance)

    def graze_floor_mask(self, similarity, cal: dict, kind):
        """True where a primary-gate guess beats the median noise word (gets chip damage)."""
        if not self.config.primary_graze_floor:
            return np.zeros(np.shape(similarity), dtype=bool)
        floor = float(np.percentile(cal["noise_sims"], self.config.primary_graze_noise_pct))
        return (np.asarray(kind) == "primary") & (np.asarray(similarity, dtype=float) >= floor)

    # ---- damage ---------------------------------------------------------------
    def hit_percent(self, relevance, multiplier, allowed, generic_multiplier=1.0, echo_multiplier=1.0,
                    relevance_min=None, gamma=None, floor_mask=None):
        """Damage in percent of the boss's starting HP (single source of truth).

        r < r_min or r <= 0 -> 0; base = GRAZE + (MAX - GRAZE) * ((r - r_min) / (1 - r_min)) ** gamma;
        hit = base * domain * generic * echo; chip floor for allowed hits (and floor_mask rows); cap at MAX.
        """
        c = self.config
        relevance_min = c.relevance_min if relevance_min is None else relevance_min
        gamma = c.damage_gamma if gamma is None else gamma
        r = np.asarray(relevance, dtype=float)
        scaled = np.clip((r - relevance_min) / (1.0 - relevance_min), 0.0, 1.0)
        base = np.where((r < relevance_min) | (r <= 0.0), 0.0,
                        c.graze_min_percent + (c.max_hit_percent - c.graze_min_percent) * scaled ** gamma)
        allowed = np.asarray(allowed, dtype=bool)
        hit = base * np.asarray(multiplier, float) * np.asarray(generic_multiplier, float) \
            * np.asarray(echo_multiplier, float)
        chip_ok = allowed & (base > 0)
        if floor_mask is not None:
            chip_ok = chip_ok | (allowed & np.asarray(floor_mask, dtype=bool))
        hit = np.where(chip_ok, np.maximum(hit, c.chip_percent), 0.0)
        return np.minimum(hit, c.max_hit_percent)

    def base_damage(self, relevance) -> float:
        """Base damage percent for a relevance with domain multiplier 1."""
        return float(self.hit_percent(relevance, 1.0, True))

    def tier(self, damage_percent: float, allowed: bool) -> str:
        if not allowed:
            return "BLOCKED — OUT OF DOMAIN"
        if damage_percent <= 0:
            return "NO EFFECT"
        for cut, name in self.config.tier_cuts:
            if damage_percent >= cut:
                return name
        return "WEAK GRAZE"

    def is_generic(self, cleaned: str) -> bool:
        return is_generic_term(cleaned, self.data.generic_terms)

    def validate(self, guess: str, target: str, used) -> tuple:
        return validate_player_input(guess, target, used, self.config.block_target_word_in_guess)

    def echo_threshold(self) -> float:
        """Raw cosine above which a guess counts as a semantic repeat (model-relative)."""
        floor = self.noise_pair_p90
        return floor + self.config.echo_relative * (1.0 - floor)

    def echo_similarity(self, guess_vector, previous_vectors, cal: dict, required_domain: str) -> float:
        """Max cosine to earlier guesses that would have dealt damage."""
        best = 0.0
        noise, ceiling = self.bounds(cal)
        for vector in previous_vectors:
            adjusted, _, _, _ = self.domain_scores_from_vector(vector)
            allowed, _, _ = self.gate(adjusted, required_domain)
            relevance = float(normalized_relevance(vector @ cal["vector"], noise, ceiling))
            if allowed and relevance >= self.config.relevance_min:
                best = max(best, float(vector @ guess_vector))
        return best

    # ---- attacks, battles, hints ----------------------------------------------
    def evaluate_attack(self, guess: str, target: str, required_domain: str, used_guesses, enemy_max_hp: int) -> dict:
        """Validate, score and compute damage for one guess."""
        c = self.config
        is_valid, cleaned, reason = self.validate(guess, target, used_guesses)
        if not is_valid:
            return {"valid": False, "reason": reason}
        cal = self.target_calibration(target, required_domain)
        vectors, latency_ms = timed_call(self.encode, [cleaned] + sorted(used_guesses))
        guess_vector = vectors[0]

        similarity = float(guess_vector @ cal["vector"])
        noise, ceiling = self.bounds(cal)
        relevance = float(normalized_relevance(similarity, noise, ceiling))

        adjusted, raw, probabilities, nearest = self.domain_scores_from_vector(guess_vector)
        allowed, multiplier, match_type = self.gate(adjusted, required_domain)
        lead = adjusted[required_domain] - max(v for dom, v in adjusted.items() if dom != required_domain)

        generic = self.is_generic(cleaned)
        generic_multiplier = c.generic_penalty if generic else 1.0
        echo_similarity = self.echo_similarity(guess_vector, vectors[1:], cal, required_domain)
        echo = echo_similarity >= self.echo_threshold()
        echo_multiplier = c.echo_penalty if echo else 1.0

        base_percent = self.base_damage(relevance) if allowed else 0.0
        floor_mask = bool(self.graze_floor_mask(similarity, cal, match_type))
        hit_percent = float(self.hit_percent(relevance, multiplier, allowed, generic_multiplier,
                                             echo_multiplier, floor_mask=floor_mask))
        damage = int(round(enemy_max_hp * hit_percent / 100))
        if hit_percent > 0:
            damage = max(damage, 1)
        return {
            "valid": True, "word": cleaned, "similarity": similarity,
            "noise_floor": noise, "ceiling": ceiling, "relevance": relevance,
            "domain_scores": adjusted, "domain_raw_scores": raw,
            "domain_background": dict(self.background), "domain_lead": lead,
            "graze_floor": floor_mask,
            "domain_probs": probabilities, "nearest_anchors": nearest,
            "domain_allowed": allowed, "domain_match_type": match_type, "multiplier": multiplier,
            "is_generic": generic, "generic_multiplier": generic_multiplier,
            "echo_similarity": echo_similarity, "is_echo": echo, "echo_multiplier": echo_multiplier,
            "base_percent": base_percent, "damage_percent": round(hit_percent, 1),
            "hit_percent": hit_percent, "damage": damage,
            "tier": self.tier(hit_percent, allowed), "latency_ms": latency_ms,
        }

    @staticmethod
    def resolve_damage(result: dict, enemy_max_hp: int, current_hp: int) -> dict:
        """Add overkill bookkeeping (the per-hit cap is already applied)."""
        if not result["valid"]:
            return result
        calculated = result["damage"] if result["domain_allowed"] else 0
        actual = max(0, min(calculated, current_hp))
        result.update({"calculated_damage": calculated, "actual_damage": actual,
                       "overkill": calculated - actual, "battle_tier": result["tier"]})
        return result

    def simulate_battle(self, target: str, domain: str, initial_hp: int, guesses, on_turn=None) -> pd.DataFrame:
        """Non-interactive battle; one row per guess. on_turn(result, hp, domain) renders a turn."""
        hp, used, rows = initial_hp, set(), []
        for phrase in guesses:
            if hp <= 0:
                break
            result = self.evaluate_attack(phrase, target, domain, used, initial_hp)
            if not result["valid"]:
                rows.append({"Guess": clean_text(phrase), "Tier": "INVALID", "Note": result["reason"]})
                continue
            result = self.resolve_damage(result, initial_hp, hp)
            used.add(result["word"])
            hp_before, hp = hp, hp - result["actual_damage"]
            if on_turn is not None:
                on_turn(result, hp, domain)
            rows.append({
                "Guess": result["word"], "Cosine": round(result["similarity"], 3),
                "Relevance r": round(result["relevance"], 2), "Gate": result["domain_match_type"],
                "Generic": result["is_generic"], "Echo": result["is_echo"],
                "Hit %": round(result["hit_percent"], 1), "Tier": result["tier"],
                "Damage": result["actual_damage"], "HP before": hp_before, "HP left": max(hp, 0),
            })
        return pd.DataFrame(rows)

    @staticmethod
    def summarize_battle(battle_df: pd.DataFrame, initial_hp: int) -> dict:
        """Turns, outcome and tier counts for one battle table."""
        valid = battle_df[battle_df["Tier"] != "INVALID"]
        tiers = valid["Tier"]
        defeated = bool(len(valid) and valid["HP left"].iloc[-1] <= 0)
        return {
            "Valid turns": len(valid), "Defeated": defeated,
            "Turns to defeat": len(valid) if defeated else None,
            "HP left": int(valid["HP left"].iloc[-1]) if len(valid) else initial_hp,
            "Critical": int((tiers == "CRITICAL STRIKE").sum()),
            "Strong": int((tiers == "STRONG HIT").sum()),
            "Solid": int((tiers == "SOLID HIT").sum()),
            "Graze": int((tiers == "WEAK GRAZE").sum()),
            "No effect / blocked": int(tiers.isin(["NO EFFECT", "BLOCKED — OUT OF DOMAIN"]).sum()),
        }

    def hint(self, target: str, domain: str, used_guesses) -> str:
        """A useful but not optimal pool term (relevance 0.35–0.75)."""
        cal = self.target_calibration(target, domain)
        noise, ceiling = self.bounds(cal)
        relevance = normalized_relevance(cal["pool_sims"], noise, ceiling)
        used = {duplicate_key(u) for u in used_guesses}
        options = [(abs(r - 0.55), t) for t, r in zip(cal["pool_terms"], relevance)
                   if duplicate_key(t) not in used and 0.35 <= r <= 0.75]
        if not options:
            options = [(abs(r - 0.55), t) for t, r in zip(cal["pool_terms"], relevance)
                       if duplicate_key(t) not in used]
        return min(options)[1] if options else "No unused hint is available."

ENGINE_DATA = EngineData(
    domains=tuple(DOMAINS), anchors=DOMAIN_ANCHORS, noise_vocab=tuple(NOISE_VOCAB),
    examples=DOMAIN_EXAMPLES, pool_vocab=POOL_VOCAB, generic_terms=frozenset(GENERIC_TERMS_V2),
    calibration_targets=dict(CALIBRATION_TARGETS),
)

def evaluate_test_set(engine: LexicalCombatEngine, test_cases: pd.DataFrame) -> pd.DataFrame:
    """Run labelled (target, guess, domain) rows through evaluate_attack; same columns as before."""
    rows = []
    for row in test_cases.itertuples(index=False):
        result = engine.evaluate_attack(row.guess, row.target, row.required_domain, set(), 200)
        if not result["valid"]:
            rows.append({"Target": row.target, "Guess": row.guess,
                         "Required Domain": row.required_domain, "Predicted Tier": "INVALID",
                         "Human Label": row.human_label, "Predicted Label": "invalid",
                         "Label Aligned": False})
            continue
        predicted = GATE_LABEL_OF_KIND[result["domain_match_type"]]
        rows.append({
            "Target": row.target, "Guess": row.guess, "Required Domain": row.required_domain,
            "Cosine Similarity": round(result["similarity"], 3),
            "Relevance r": round(result["relevance"], 2),
            "Domain Scores": " | ".join(f"{d}: {result['domain_scores'][d]:+.2f}" for d in engine.data.domains),
            "Domain Match": result["domain_match_type"], "Predicted Tier": result["tier"],
            "Hit %": round(result["hit_percent"], 1), "Damage Dealt": result["damage"],
            "Inference Latency (ms)": round(result["latency_ms"], 2),
            "Human Label": row.human_label, "Predicted Label": predicted,
            "Label Aligned": predicted == row.human_label,
        })
    return pd.DataFrame(rows)
log_section("3. Engine", _t_s3)

## 4. Model selection
Loads the Sentence Transformer — pretrained `all-MiniLM-L6-v2` (`"base"`) or a fine-tuned checkpoint (`"fine_tuned"`, falls back to base with a warning) — and builds the engine.

In [10]:
# 4. Model selection: load base or fine-tuned (with fallback) and build the engine
# What this cell does:
#   - Loads the Sentence Transformer: pretrained MiniLM or a fine-tuned checkpoint.
#   - Falls back to the base model if no checkpoint exists.
#   - Builds the engine with default (pre-calibration) settings.
BASE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
FINE_TUNED_DIRNAME = "fine_tuned_lexical_minilm_v3"
_MODEL_CACHE = {}   # loaded models are reused across re-runs (loading is deterministic)

def fine_tuned_candidates() -> list:
    """Checkpoint folders to try, newest location first."""
    return [PATHS.models / FINE_TUNED_DIRNAME, *PATHS.legacy_models]

def find_fine_tuned_checkpoint():
    """First existing checkpoint folder, else None."""
    return next((p for p in fine_tuned_candidates() if p.is_dir()), None)

def load_model(choice: str):
    """Return (model, resolved_choice, source). Falls back to base when no checkpoint exists."""
    if choice not in ("base", "fine_tuned"):
        raise ValueError("MODEL_CHOICE must be 'base' or 'fine_tuned'.")
    if choice == "fine_tuned":
        checkpoint = find_fine_tuned_checkpoint()
        if checkpoint is None:
            print("WARNING: MODEL_CHOICE = 'fine_tuned' but no checkpoint was found — using 'base'.")
            choice = "base"
        else:
            key = str(checkpoint)
            if key not in _MODEL_CACHE:
                _MODEL_CACHE[key] = SentenceTransformer(key)
            return _MODEL_CACHE[key], "fine_tuned", key
    if BASE_MODEL_NAME not in _MODEL_CACHE:
        _MODEL_CACHE[BASE_MODEL_NAME] = SentenceTransformer(BASE_MODEL_NAME)
    return _MODEL_CACHE[BASE_MODEL_NAME], "base", BASE_MODEL_NAME

def available_model_choices() -> list:
    """Model choices that can be loaded in this runtime."""
    return ["base"] + (["fine_tuned"] if find_fine_tuned_checkpoint() is not None else [])

with section("4. Model selection"):
    model, ACTIVE_MODEL, MODEL_SOURCE = load_model(MODEL_CHOICE)
    engine = LexicalCombatEngine(model, ENGINE_DATA, EngineConfig())
    print(f"Active model: {ACTIVE_MODEL} ({MODEL_SOURCE}) | available: {available_model_choices()}")
    print("Domain background (mean debias term on everyday nouns): "
          + " | ".join(f"{d}: {v:.3f}" for d, v in engine.background.items()))
    print(f"Span floor ({engine.config.min_span_mode}): {engine.span_floor:.3f} | "
          f"anisotropy (mean cos between everyday nouns): {engine.noise_pair_mean:.3f}")
    for stage in GAME_STAGES:
        cal = engine.target_calibration(stage["target"], stage["domain"])
        noise, ceiling = engine.bounds(cal)
        top = sorted(zip(cal["pool_sims"], cal["pool_terms"]), reverse=True)[:engine.config.ceiling_topk]
        print(f"  {stage['target']:<10} noise floor {noise:.3f} | ceiling {ceiling:.3f} | "
              f"ceiling terms: {', '.join(t for _, t in top)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Active model: base (sentence-transformers/all-MiniLM-L6-v2) | available: ['base']
Domain background (mean debias term on everyday nouns): Business: 0.061 | Technology: 0.086 | Science: 0.070 | Arts: 0.083
Span floor (median): 0.371 | anisotropy (mean cos between everyday nouns): 0.291
  executive  noise floor 0.272 | ceiling 0.654 | ceiling terms: ceo, leadership, corporate leadership
  algorithm  noise floor 0.227 | ceiling 0.598 | ceiling terms: computation, optimization, binary search
  catalyst   noise floor 0.267 | ceiling 0.638 | ceiling terms: enzyme, hydrogen, chemical reaction


## 5. Calibration
- **5.2** Diagnoses the gate on the old holdout with the default thresholds (read-only).
- **5.3** Gate calibration functions (grid, plateau selection, 5-fold grouped CV).
- **5.4** Damage-curve calibration functions (grid, per-target loss + pacing, LOTO-CV).
- **5.5** Runs them (or loads a matching file) and applies the result with `engine.config = replace(engine.config, **params)`.
- **5.6** Reviews the tuned gate.

In [11]:
# 5.2 Gate diagnosis on the old holdout (diagnostic set, read-only; runs before tuning)
# What this cell does:
#   - Diagnoses the gate on the old holdout with the default thresholds (read-only).
#   - Explains why each in-domain phrase missed PRIMARY.
#   - Shows the circularity check and ceiling terms per target.
# Cause rules for an in-domain row that did not reach PRIMARY:
#   negative lead      : another domain scores higher (lead < 0)
#   top-k dilution     : passes at top-1 but the top-k mean drags it below the cut
#   threshold too high : required domain is #1 but below the primary cut at any k
# The new holdout (Section 7) is never looked at before Section 7.

def diagnose_gate_rows(engine: LexicalCombatEngine, frame: pd.DataFrame, params: dict) -> pd.DataFrame:
    """One row per (guess, domain): debiased/raw scores at k=1..3, nearest anchors, cause."""
    phrases = frame["guess"].map(clean_text).tolist()
    vectors = engine.encode(phrases)
    rows = []
    for i, row in enumerate(frame.itertuples(index=False)):
        d = row.required_domain
        info = {"target": row.target, "guess": row.guess, "domain": d, "human": row.human_label}
        for k in (1, 2, 3):
            adj, raw = engine.domain_scores(vectors[i], k)
            adj, raw = dict(zip(DOMAINS, adj[0])), dict(zip(DOMAINS, raw[0]))
            others = {x: v for x, v in adj.items() if x != d}
            best_other = max(others, key=others.get)
            info[f"req_k{k}"] = adj[d]
            info[f"lead_k{k}"] = adj[d] - others[best_other]
            if k == params["k"]:
                info["raw"] = raw[d]
                info["background"] = engine.background_by_k[k][d]
                info["top_domain"] = max(adj, key=adj.get)
                info["rival"] = best_other
        req, lead = info["req_k" + str(params["k"])], info["lead_k" + str(params["k"])]
        kind = str(gate_kind_arrays([req], [lead],
                                    params["primary"], params["secondary"], params["gap"],
                                    params["weak_gap"])[0])
        info["gate"] = kind
        sims = engine.anchor_vectors[d] @ vectors[i]
        info["nearest_anchor_cos"] = float(sims.max())
        info["nearest_anchor"] = DOMAIN_ANCHORS[d][int(np.argmax(sims))][:70]
        rival_sims = engine.anchor_vectors[info["rival"]] @ vectors[i]
        info["rival_anchor"] = DOMAIN_ANCHORS[info["rival"]][int(np.argmax(rival_sims))][:50]
        if kind == "primary":
            cause = "ok (primary)"
        elif lead < 0:
            cause = f"negative lead ({info['rival']} wins)"
        elif info["req_k1"] >= params["primary"] and info["lead_k1"] >= 0:
            cause = "top-k dilution"
        else:
            cause = "threshold too high"
        info["cause"] = cause if row.human_label == "in_domain" else ""
        rows.append(info)
    return pd.DataFrame(rows)

with section("5.2 Gate diagnosis (old holdout)"):
    c = engine.config   # pre-calibration defaults
    diag_params = {"primary": c.gate_primary_min, "secondary": c.gate_secondary_min,
                   "gap": c.gate_secondary_max_gap, "weak_gap": c.gate_weak_max_gap, "k": c.anchor_topk}
    holdout_draft_df = pd.DataFrame(OLD_HOLDOUT_DIAG_CASES, columns=[
        "target", "guess", "required_domain", "human_label", "human_expected_tier"])
    holdout_diag = diagnose_gate_rows(engine, holdout_draft_df, diag_params)
    print(f"Gate used for the diagnosis: {diag_params}")
    display(holdout_diag[["target", "guess", "domain", "human", "gate", "req_k1", "req_k2", "req_k3",
                          "raw", "background", "lead_k2", "top_domain", "nearest_anchor_cos",
                          "nearest_anchor", "rival_anchor", "cause"]].round(3))

    in_rows = holdout_diag[holdout_diag["human"] == "in_domain"]
    print(f"In-domain holdout rows reaching PRIMARY: {(in_rows['gate'] == 'primary').sum()}/{len(in_rows)}")
    print("Cause counts for in-domain rows that missed PRIMARY:")
    display(in_rows.loc[in_rows["gate"] != "primary", "cause"].value_counts().rename("rows").to_frame())
    focus = holdout_diag[holdout_diag["guess"].isin(["harmony", "protein", "cipher"])]
    print("Focus rows (harmony / protein / cipher):")
    display(focus[["guess", "domain", "gate", "req_k1", "req_k2", "req_k3", "lead_k2",
                   "nearest_anchor_cos", "cause"]].round(3))

    # Circularity evidence: in-domain phrases WRITTEN in anchors score far higher
    # than fresh in-domain phrases, so thresholds tuned on them are too strict.
    _all_vectors = dict(zip(sorted(set(gate_set_all_df["phrase"])),
                            engine.encode(sorted(set(gate_set_all_df["phrase"])))))
    _circ = gate_set_all_df[gate_set_all_df["label"] == "in_domain"].copy()
    _circ["required"] = [float(dict(zip(DOMAINS, engine.domain_scores(_all_vectors[p], diag_params["k"])[0][0]))[d])
                         for p, d in zip(_circ["phrase"], _circ["domain"])]
    print("Debiased required-domain score of in-domain gate phrases (circularity check):")
    display(_circ.groupby("anchor_echo")["required"].describe()[["count", "mean", "25%", "50%", "75%"]]
            .rename(index={True: "written in anchors", False: "fresh / not in anchors"}).round(3))
    _req_col = "req_k" + str(diag_params["k"])
    print(f"Holdout in-domain median required score: {in_rows[_req_col].median():.3f}")

    # Ceiling coverage of the term pool for every holdout target.
    print("\nCeiling terms per holdout target (pool coverage):")
    _cov = []
    for target, domain in sorted({(t, d) for t, _, d, _, _ in OLD_HOLDOUT_DIAG_CASES}):
        cal = engine.target_calibration(target, domain)
        noise, ceiling = engine.raw_bounds(cal)
        top = sorted(zip(cal["pool_sims"], cal["pool_terms"]), reverse=True)[:5]
        _cov.append({"target": target, "noise": noise, "ceiling (raw)": ceiling, "span": ceiling - noise,
                     "span floor": engine.span_floor, "top-5 pool terms": ", ".join(f"{t} ({s:.2f})" for s, t in top)})
    display(pd.DataFrame(_cov).round(3))

Gate used for the diagnosis: {'primary': 0.2, 'secondary': 0.08, 'gap': 0.04, 'weak_gap': 0.1, 'k': 2}


,target,guess,domain,human,gate,req_k1,req_k2,req_k3,raw,background,lead_k2,top_domain,nearest_anchor_cos,nearest_anchor,rival_anchor,cause
0,manager,supervisor,Business,in_domain,primary,0.348,0.346,0.289,0.408,0.061,0.306,Business,0.424,Business management leadership executive chief executive officer CEO p,Arts visual art painting sculpture drawing illustr,ok (primary)
1,manager,team leader,Business,in_domain,primary,0.336,0.344,0.289,0.405,0.061,0.358,Business,0.412,Business management leadership executive chief executive officer CEO p,Cybersecurity and networks encryption cryptography,ok (primary)
2,manager,employee,Business,in_domain,primary,0.264,0.275,0.276,0.337,0.061,0.261,Business,0.340,"Management roles include director, managing director, chief executive",Artificial intelligence AI machine learning deep l,ok (primary)
3,manager,payroll,Business,in_domain,primary,0.344,0.296,0.272,0.357,0.061,0.255,Business,0.420,Business operations strategy planning human resources payroll hiring p,An algorithm is a finite sequence of computational,ok (primary)
4,manager,spreadsheet,Business,ambiguous,secondary,0.146,0.145,0.149,0.207,0.061,0.005,Business,0.222,Business finance accounting financial reporting quarterly report annua,An algorithm is a finite sequence of computational,
5,manager,orchestra conductor,Business,out_of_domain,weak_secondary,0.185,0.169,0.155,0.230,0.061,-0.075,Arts,0.261,Business management leadership executive chief executive officer CEO p,"Music includes melody, harmony, rhythm, songs, com",
6,manager,giraffe,Business,out_of_domain,out_of_domain,-0.017,-0.017,-0.017,0.044,0.061,-0.120,Arts,0.060,Business management leadership executive chief executive officer CEO p,"Painting is a visual art form that uses pigment, c",
7,manager,firmware,Business,out_of_domain,out_of_domain,0.063,0.066,0.072,0.127,0.061,-0.111,Technology,0.140,"Management roles include director, managing director, chief executive",Computer hardware motherboard CPU processor GPU RA,
8,encryption,cipher,Technology,in_domain,secondary,0.199,0.142,0.116,0.229,0.086,0.125,Technology,0.302,Cybersecurity and networks encryption cryptography firewall router swi,"Chemistry studies molecules, atoms, elements, comp",threshold too high
9,encryption,private key,Technology,in_domain,secondary,0.119,0.093,0.078,0.180,0.086,0.114,Technology,0.222,Cybersecurity and networks encryption cryptography firewall router swi,"Management roles include director, managing direct",threshold too high


In-domain holdout rows reaching PRIMARY: 8/17
Cause counts for in-domain rows that missed PRIMARY:


,rows
cause,
threshold too high,7
top-k dilution,2


Focus rows (harmony / protein / cipher):


,guess,domain,gate,req_k1,req_k2,req_k3,lead_k2,nearest_anchor_cos,cause
8,cipher,Technology,secondary,0.199,0.142,0.116,0.125,0.302,threshold too high
15,protein,Science,secondary,0.111,0.113,0.097,0.078,0.194,threshold too high
22,harmony,Arts,secondary,0.255,0.148,0.111,0.131,0.354,top-k dilution


Debiased required-domain score of in-domain gate phrases (circularity check):


,count,mean,25%,50%,75%
anchor_echo,,,,,
fresh / not in anchors,112.0,0.178,0.087,0.155,0.247
written in anchors,38.0,0.311,0.240,0.295,0.379


Holdout in-domain median required score: 0.163

Ceiling terms per holdout target (pool coverage):


,target,noise,ceiling (raw),span,span floor,top-5 pool terms
0,encryption,0.209,0.556,0.347,0.371,"cryptography (0.77), network security (0.45), authentication (0.44), cloud computing (0.40), cybersecurity (0.33)"
1,enzyme,0.230,0.527,0.297,0.371,"biochemistry (0.64), catalyst (0.48), glucose (0.46), laboratory (0.42), molecular biology (0.42)"
2,manager,0.356,0.667,0.311,0.371,"management (0.80), executive officer (0.60), leadership (0.60), managing director (0.59), chief executive officer (0..."
3,melody,0.376,0.597,0.221,0.371,"musical note (0.65), symphony (0.57), artist (0.57), piano (0.57), poetry (0.56)"


In [12]:
# 5.3 Gate calibration (functions only; Section 5.5 runs them)
# What this cell does:
#   - Gate grid search: ANCHOR_TOPK in {1,2,3} x thresholds, plateau selection.
#   - 5-fold grouped CV and metrics (macro F1, pass / block rates, leak split).
#   - Functions only; 5.5 runs them.
# Tunes on gate_set_df (anchor-echo rows excluded), ANCHOR_TOPK in {1,2,3}, plateau selection,
# 5-fold grouped CV. Leaks are split into gate leak (0 damage) and damaging leak.

LABELS = ["in_domain", "ambiguous", "out_of_domain"]
LABEL_CODE = {label: i for i, label in enumerate(LABELS)}
KIND_CODE = {"primary": 0, "secondary": 1, "weak_secondary": 1, "out_of_domain": 2}
MAX_OUT_OF_DOMAIN_LEAK = 0.03     # at most 3% of out-of-domain phrases may pass
MIN_IN_DOMAIN_PRIMARY = 0.80      # at least 80% of in-domain phrases must be PRIMARY
GATE_CV_FOLDS = 5
SELECTION_TOL = 0.005             # plateau scores within this of the best count as ties
GATE_GRID = {
    "k": [1, 2, 3],
    "primary": [float(x) for x in np.round(np.arange(0.02, 0.401, 0.02), 2)],
    "secondary": [float(x) for x in np.round(np.arange(0.00, 0.201, 0.02), 2)],
    "gap": [0.0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12],
    "weak_extra": [0.02, 0.05, 0.08, 0.12, 0.16, 0.20],
}
GATE_NUMERIC_DIMS = ["primary", "secondary", "gap", "weak_extra"]

def classification_metrics(y_true, y_pred):
    """Accuracy, balanced accuracy and macro F1 over the three gate labels."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    recalls, f1s = [], []
    for label in LABELS:
        tp = np.sum((y_true == label) & (y_pred == label))
        fp = np.sum((y_true != label) & (y_pred == label))
        fn = np.sum((y_true == label) & (y_pred != label))
        recalls.append(tp / (tp + fn) if tp + fn else 0.0)
        f1s.append(2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0)
    return {"accuracy": float(np.mean(y_true == y_pred)),
            "balanced_accuracy": float(np.mean(recalls)), "macro_f1": float(np.mean(f1s))}

def pass_block_metrics(y_true, kinds):
    """Game-feel metrics on gate kinds (primary / secondary / weak_secondary / out_of_domain)."""
    y_true, kinds = np.asarray(y_true), np.asarray(kinds)
    passed = kinds != "out_of_domain"
    def rate(mask, values):
        return float(np.mean(values[mask])) if mask.any() else 1.0
    in_mask, amb_mask, out_mask = (y_true == "in_domain"), (y_true == "ambiguous"), (y_true == "out_of_domain")
    out_block = rate(out_mask, ~passed)
    return {"in_domain_primary": rate(in_mask, kinds == "primary"), "in_domain_pass": rate(in_mask, passed),
            "ambiguous_pass": rate(amb_mask, passed), "out_block": out_block, "out_leak": 1 - out_block}

def gate_inputs(engine: LexicalCombatEngine, frame: pd.DataFrame, ks=(1, 2, 3)) -> dict:
    """Required-domain score and lead for every row at each top-k."""
    unique = sorted(set(frame["phrase"]))
    vectors = engine.encode(unique)
    index = {p: i for i, p in enumerate(unique)}
    rows = np.array([index[p] for p in frame["phrase"]])
    cols = np.array([DOMAINS.index(d) for d in frame["domain"]])
    out = {}
    for k in ks:
        adjusted, _ = engine.domain_scores(vectors, k)
        adj = adjusted[rows]
        required = adj[np.arange(len(rows)), cols]
        masked = adj.copy()
        masked[np.arange(len(rows)), cols] = -np.inf
        out[k] = {"required": required, "lead": required - masked.max(axis=1),
                  "top_domain": np.array(DOMAINS)[adj.argmax(axis=1)]}
    return out

def _grid_axes() -> list:
    """Threshold axes in grid order."""
    return [GATE_GRID[name] for name in ["primary", "secondary", "gap", "weak_extra"]]

def grid_predictions(required, lead):
    """(combos, n) gate codes for every primary/secondary/gap/weak_extra combo."""
    p, s, g, e = [a.reshape(-1, 1) for a in np.meshgrid(*_grid_axes(), indexing="ij")]
    is_primary = (required >= p) & (lead >= 0)
    passes = required >= s
    is_sec = ~is_primary & passes & (lead >= -g)
    is_weak = ~is_primary & ~is_sec & passes & (lead >= -(g + e))
    return np.where(is_primary, 0, np.where(is_sec | is_weak, 1, 2)).astype(np.int8)

def grid_metrics(pred, y, mask):
    """Vectorised metrics over combos for the rows in mask."""
    pred, y = pred[:, mask], y[mask]
    def class_rate(cls_mask, cond):
        return cond[:, cls_mask].mean(axis=1) if cls_mask.any() else np.ones(len(pred))
    in_primary = class_rate(y == 0, pred == 0)
    amb_pass = class_rate(y == 1, pred != 2)
    out_block = class_rate(y == 2, pred == 2)
    f1 = []
    for c in range(3):
        tp = ((pred == c) & (y == c)).sum(axis=1)
        fp = ((pred == c) & (y != c)).sum(axis=1)
        fn = ((pred != c) & (y == c)).sum(axis=1)
        denom = 2 * tp + fp + fn
        f1.append(np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0))
    macro_f1 = np.mean(f1, axis=0)
    leak = 1 - out_block
    score = ((in_primary + amb_pass + out_block) / 3
             - 3 * np.clip(leak - MAX_OUT_OF_DOMAIN_LEAK, 0, None)
             - np.clip(MIN_IN_DOMAIN_PRIMARY - in_primary, 0, None)
             + 0.1 * macro_f1)
    feasible = (leak <= MAX_OUT_OF_DOMAIN_LEAK) & (in_primary >= MIN_IN_DOMAIN_PRIMARY)
    return {"score": score, "feasible": feasible, "in_domain_primary": in_primary,
            "ambiguous_pass": amb_pass, "out_leak": leak, "macro_f1": macro_f1}

def select_plateau(score_nd, numeric_axes, tol=SELECTION_TOL):
    """Index of the best neighbourhood-mean score; ties broken toward the grid centre."""
    size = [3 if axis in numeric_axes else 1 for axis in range(score_nd.ndim)]
    smooth = uniform_filter(score_nd, size=size, mode="nearest")
    candidates = np.argwhere(smooth >= smooth.max() - tol)
    shape = np.array(score_nd.shape)
    def centre_distance(idx):
        return sum(((idx[a] / max(shape[a] - 1, 1)) - 0.5) ** 2 for a in numeric_axes)
    best = min(candidates, key=lambda idx: (centre_distance(idx), -score_nd[tuple(idx)]))
    return tuple(int(i) for i in best), smooth

def edge_report(chosen, grid, numeric_names):
    """Which chosen values sit on the first/last grid value."""
    return {name: (chosen[name] in (grid[name][0], grid[name][-1])) for name in numeric_names}

def _phrase_folds(phrases, n_folds, seed=42) -> np.ndarray:
    """Grouped folds: every row of a phrase lands in the same fold."""
    unique = sorted(set(phrases))
    rng = np.random.default_rng(seed)
    rng.shuffle(unique)
    fold_of = {p: i % n_folds for i, p in enumerate(unique)}
    return np.array([fold_of[p] for p in phrases])

def damage_against_reference(engine: LexicalCombatEngine, phrases, domains) -> np.ndarray:
    """Hit % if each phrase were played against its domain's calibration target."""
    out = []
    for phrase, domain in zip(phrases, domains):
        result = engine.evaluate_attack(phrase, DOMAIN_REFERENCE_TARGET[domain], domain, set(), 200)
        out.append(result["hit_percent"] if result["valid"] else 0.0)
    return np.array(out)

def leak_breakdown(labels, kinds, hit_percents):
    """Gate leak = out-of-domain passes gate with 0 damage; damaging leak = passes AND damages."""
    labels, kinds, hits = np.asarray(labels), np.asarray(kinds), np.asarray(hit_percents, float)
    out = labels == "out_of_domain"
    passed = kinds != "out_of_domain"
    n = max(int(out.sum()), 1)
    gate_leak = int((out & passed & (hits <= 0)).sum())
    damaging = int((out & passed & (hits > 0)).sum())
    return {"out_of_domain rows": int(out.sum()), "gate leak (0 dmg)": gate_leak,
            "damaging leak": damaging, "gate leak %": gate_leak / n, "damaging leak %": damaging / n}

def gate_config_params(grid_params: dict) -> dict:
    """Grid choice -> EngineConfig fields (weak gap = gap + weak_extra, as before)."""
    return {"anchor_topk": int(grid_params["k"]),
            "gate_primary_min": float(grid_params["primary"]),
            "gate_secondary_min": float(grid_params["secondary"]),
            "gate_secondary_max_gap": float(grid_params["gap"]),
            "gate_weak_max_gap": float(grid_params["gap"] + grid_params["weak_extra"])}

def calibrate_gate(engine: LexicalCombatEngine, frame: pd.DataFrame, verbose: bool = True) -> dict:
    """Grid + plateau selection + grouped CV. Returns a summary dict; result["params"] uses EngineConfig names."""
    y = frame["label"].map(LABEL_CODE).to_numpy()
    inputs = gate_inputs(engine, frame, GATE_GRID["k"])
    preds = {k: grid_predictions(inputs[k]["required"], inputs[k]["lead"]) for k in GATE_GRID["k"]}
    grid_shape = [len(a) for a in _grid_axes()]
    numeric_axes = [1, 2, 3, 4]                       # axis 0 = k (categorical)

    def choose(mask):
        metrics = {k: grid_metrics(preds[k], y, mask) for k in GATE_GRID["k"]}
        score_nd = np.stack([metrics[k]["score"].reshape(grid_shape) for k in GATE_GRID["k"]])
        idx, smooth = select_plateau(score_nd, numeric_axes)
        k = GATE_GRID["k"][idx[0]]
        flat = int(np.ravel_multi_index(idx[1:], grid_shape))
        params = {"k": k, **{n: GATE_GRID[n][i] for n, i in zip(["primary", "secondary", "gap", "weak_extra"], idx[1:])}}
        return params, flat, metrics, smooth, score_nd

    full_mask = np.ones(len(frame), dtype=bool)
    params, flat, metrics, smooth, score_nd = choose(full_mask)
    tuned_codes = preds[params["k"]][flat]

    folds = _phrase_folds(frame["phrase"].tolist(), GATE_CV_FOLDS)
    oof_codes, fold_rows = np.zeros(len(frame), dtype=np.int8), []
    for f in range(GATE_CV_FOLDS):
        train, test = folds != f, folds == f
        p_f, flat_f, _, _, _ = choose(train)
        oof_codes[test] = preds[p_f["k"]][flat_f][test]
        fold_rows.append({"fold": f, **p_f, "test rows": int(test.sum())})

    def code_to_kind(codes):
        return np.array(["primary", "secondary", "out_of_domain"])[codes]

    def summarise(codes):
        kinds = code_to_kind(codes)
        return {**classification_metrics(frame["label"], kinds_to_labels(kinds)),
                **pass_block_metrics(frame["label"], kinds)}

    tuned, cv = summarise(tuned_codes), summarise(oof_codes)
    edges = edge_report(params, GATE_GRID, GATE_NUMERIC_DIMS)
    feasible = bool(metrics[params["k"]]["feasible"][flat])

    scored = frame.copy()
    k = params["k"]
    scored["required"], scored["lead"] = inputs[k]["required"], inputs[k]["lead"]
    scored["top_domain"] = inputs[k]["top_domain"]
    scored["gate"] = code_to_kind(tuned_codes)
    scored["predicted"] = kinds_to_labels(scored["gate"])

    # Top settings table (raw and plateau score) for the chosen k.
    axes = [a.ravel() for a in np.meshgrid(*_grid_axes(), indexing="ij")]
    top = pd.DataFrame(dict(zip(["primary", "secondary", "gap", "weak_extra"], axes)))
    for name in ["score", "feasible", "in_domain_primary", "ambiguous_pass", "out_leak", "macro_f1"]:
        top[name] = metrics[k][name]
    top["plateau"] = smooth[GATE_GRID["k"].index(k)].ravel()
    top = top.sort_values(["plateau", "score"], ascending=False).head(10)

    result = {"params": gate_config_params(params), "grid_params": params, "feasible": feasible, "tuned": tuned, "cv": cv, "edges": edges,
              "folds": pd.DataFrame(fold_rows), "scored": scored, "top": top}
    if verbose:
        if not feasible:
            print(f"WARNING: chosen setting misses leak <= {MAX_OUT_OF_DOMAIN_LEAK:.0%} or "
                  f"in-domain primary >= {MIN_IN_DOMAIN_PRIMARY:.0%}; best trade-off used.")
        print(f"Top plateau settings at k={k}:")
        display(top.round(3))
        print(f"Applied gate: k={k} | primary>={params['primary']:.2f} (and #1) | secondary>={params['secondary']:.2f} "
              f"gap<={params['gap']:.2f} | weak gap<={params['gap'] + params['weak_extra']:.2f}")
        print("Edge check (True = on the grid edge):", edges)
        print("Tuned set vs 5-fold grouped CV:")
        display(pd.DataFrame({"tuned": tuned, "cv": cv}).round(3))
        print("Per-fold choices (stability):")
        display(result["folds"])
    return result

In [13]:
# 5.4 Damage-curve calibration (functions only; Section 5.5 runs them)
# What this cell does:
#   - Damage-curve grid search (noise percentile, ceiling top-k, relevance min, gamma).
#   - Loss = per-target band distance + pacing penalty; plateau selection.
#   - Leave-one-target-out CV. Functions only; 5.5 runs them.
# Wide grid, per-target loss (mean + 0.5·std across targets), plateau selection,
# leave-one-target-out CV. Echo flags in battles use the parameters active when
# build_damage_data() runs (pre-calibration defaults), as before.

PACING_WEIGHT = 1.0
STD_WEIGHT = 0.5
DAMAGE_GRID = {
    "noise_percentile": [60, 65, 70, 75, 80, 85, 90, 95],
    "ceiling_topk": [1, 2, 3, 4, 5],
    "relevance_min": [float(x) for x in np.round(np.arange(0.0, 0.2001, 0.025), 3)],
    "gamma": [float(x) for x in np.round(np.arange(0.5, 1.501, 0.1), 2)],
    "span_mode": ["fixed", "median"],
}
DAMAGE_NUMERIC_DIMS = ["noise_percentile", "ceiling_topk", "relevance_min", "gamma"]
DAMAGE_DIM_ORDER = ["noise_percentile", "ceiling_topk", "relevance_min", "gamma", "span_mode"]

def band_distance(percent, band) -> float:
    """Distance from a damage percent to a (low, high) band; 0 inside."""
    low, high = band
    return float(max(low - percent, 0.0, percent - high))

def score_guess(engine: LexicalCombatEngine, target: str, domain: str, guess: str, used=()):
    """Everything the grid needs for one guess; None if the input is invalid."""
    ok, cleaned, _ = engine.validate(guess, target, set(used))
    if not ok:
        return None
    cal = engine.target_calibration(target, domain)
    vectors = engine.encode([cleaned] + list(used))
    adjusted, _, _, _ = engine.domain_scores_from_vector(vectors[0])
    allowed, multiplier, kind = engine.gate(adjusted, domain)
    echo_sim = engine.echo_similarity(vectors[0], vectors[1:], cal, domain)
    similarity = float(vectors[0] @ cal["vector"])
    return {"target": clean_text(target), "domain": domain, "guess": cleaned,
            "similarity": similarity, "allowed": allowed, "multiplier": multiplier, "kind": kind,
            "floor_mask": bool(engine.graze_floor_mask(similarity, cal, kind)),
            "generic_multiplier": engine.config.generic_penalty if engine.is_generic(cleaned) else 1.0,
            "echo_multiplier": engine.config.echo_penalty if echo_sim >= engine.echo_threshold() else 1.0}

def build_damage_data(engine: LexicalCombatEngine) -> tuple:
    """Score banded hits and reference battles with the engine's current gate and parameters."""
    hits = []
    for row in damage_cases_df.itertuples(index=False):
        scored = score_guess(engine, row.target, row.required_domain, row.guess)
        if scored:
            hits.append({**scored, "band": TIER_BANDS_V2[row.human_expected_tier],
                         "expected": row.human_expected_tier})
    stages = {s["target"]: s for s in GAME_STAGES}
    battles = []
    for target, plays in REFERENCE_PLAYS.items():
        for style, (guesses, turn_range) in plays.items():
            used, rows = [], []
            for guess in guesses:
                scored = score_guess(engine, target, stages[target]["domain"], guess, used)
                if scored:
                    rows.append(scored)
                    used.append(scored["guess"])
            battles.append({"target": target, "style": style, "range": turn_range,
                            "hp": stages[target]["hp"], "rows": pd.DataFrame(rows)})
    return pd.DataFrame(hits), battles

def _bounds_table(engine: LexicalCombatEngine, targets_domains) -> tuple:
    """noise[t][pi] and raw ceiling[t][ki] for every grid value."""
    noise, ceil = {}, {}
    for target, domain in targets_domains:
        cal = engine.target_calibration(target, domain)
        noise[target] = np.array([np.percentile(cal["noise_sims"], p) for p in DAMAGE_GRID["noise_percentile"]])
        ceil[target] = np.array([topk_mean(cal["pool_sims"], k) for k in DAMAGE_GRID["ceiling_topk"]])
    return noise, ceil

def _percents(engine: LexicalCombatEngine, frame, noise_tab, ceil_tab, pi, ki, span_floor):
    """(R, G, rows) hit % for one noise percentile / top-k / span floor."""
    targets = frame["target"].to_numpy()
    noise = np.array([noise_tab[t][pi] for t in targets])
    ceiling = np.maximum(np.array([ceil_tab[t][ki] for t in targets]), noise + span_floor)
    relevance = normalized_relevance(frame["similarity"].to_numpy(), noise, ceiling)
    r_min = np.array(DAMAGE_GRID["relevance_min"]).reshape(-1, 1, 1)
    gamma = np.array(DAMAGE_GRID["gamma"]).reshape(1, -1, 1)
    return engine.hit_percent(relevance, frame["multiplier"].to_numpy(), frame["allowed"].to_numpy(bool),
                             frame["generic_multiplier"].to_numpy(), frame["echo_multiplier"].to_numpy(),
                             r_min, gamma, frame["floor_mask"].to_numpy(bool))

def _turns_vectorised(percents, hp):
    """percents (..., n) -> turns (..., ) as float (nan = never) and remaining HP."""
    damage = np.where(percents > 0, np.maximum(np.round(hp * percents / 100), 1), 0)
    cumulative = np.cumsum(damage, axis=-1)
    won = cumulative >= hp
    turns = np.where(won.any(axis=-1), won.argmax(axis=-1) + 1, np.nan)
    remaining = np.maximum(hp - cumulative[..., -1], 0)
    return turns, remaining

def calibrate_damage(engine: LexicalCombatEngine, verbose: bool = True) -> dict:
    """Full grid, plateau selection on all targets, LOTO-CV. result["params"] uses EngineConfig names."""
    hits, battles = build_damage_data(engine)
    targets = sorted(hits["target"].unique())
    pairs = set(zip(hits["target"], hits["domain"])) | {(t, d) for t, d in CALIBRATION_TARGETS.items()}
    noise_tab, ceil_tab = _bounds_table(engine, pairs)
    P, K, R, G, S = (len(DAMAGE_GRID[n]) for n in DAMAGE_DIM_ORDER)
    T = len(targets)
    band_loss = np.zeros((P, K, R, G, S, T))
    pace_loss = np.zeros((P, K, R, G, S, T))
    hit_pct = np.zeros((P, K, R, G, S, len(hits)))
    battle_turns = np.full((P, K, R, G, S, len(battles)), np.nan)
    lows = np.array([b[0] for b in hits["band"]])
    highs = np.array([b[1] for b in hits["band"]])
    target_idx = np.array([targets.index(t) for t in hits["target"]])
    for pi, ki, si in itertools.product(range(P), range(K), range(S)):
        if DAMAGE_GRID["span_mode"][si] == "median":
            spans = [ceil_tab[t][ki] - noise_tab[t][pi] for t in CALIBRATION_TARGETS]
            span_floor = max(float(np.median(spans)), 0.05)
        else:
            span_floor = engine.config.min_calibration_span
        pct = _percents(engine, hits, noise_tab, ceil_tab, pi, ki, span_floor)      # (R, G, n)
        hit_pct[pi, ki, :, :, si] = pct
        dist = np.maximum(np.maximum(lows - pct, 0), pct - highs)
        for t in range(T):
            band_loss[pi, ki, :, :, si, t] = dist[..., target_idx == t].mean(axis=-1)
        penalties = {t: [] for t in range(T)}
        for bi, battle in enumerate(battles):
            if battle["rows"].empty:
                continue
            bp = _percents(engine, battle["rows"], noise_tab, ceil_tab, pi, ki, span_floor)
            turns, remaining = _turns_vectorised(bp, battle["hp"])
            battle_turns[pi, ki, :, :, si, bi] = turns
            low, high = battle["range"]
            pen = np.where(np.isnan(turns), 6.0 + 10.0 * remaining / battle["hp"],
                           3.0 * np.maximum(np.maximum(low - np.nan_to_num(turns), 0), np.nan_to_num(turns) - high))
            penalties[targets.index(battle["target"])].append(pen)
        for t in range(T):
            if penalties[t]:
                pace_loss[pi, ki, :, :, si, t] = np.mean(penalties[t], axis=0)
    per_target = band_loss + PACING_WEIGHT * pace_loss
    numeric_axes = [0, 1, 2, 3]

    def choose_index(target_mask):
        sub = per_target[..., target_mask]
        score = -(sub.mean(axis=-1) + STD_WEIGHT * sub.std(axis=-1))   # higher = better
        idx, _ = select_plateau(score, numeric_axes, tol=SELECTION_TOL)
        return idx

    def params_of(idx):
        return {n: DAMAGE_GRID[n][i] for n, i in zip(DAMAGE_DIM_ORDER, idx)}

    def eval_rows(idx, row_mask, battle_ids):
        pct = hit_pct[idx][row_mask]
        in_band = ((pct >= lows[row_mask]) & (pct <= highs[row_mask]))
        pace = []
        for bi in battle_ids:
            turns = battle_turns[idx][bi]
            low, high = battles[bi]["range"]
            pace.append({"battle": f"{battles[bi]['target']}/{battles[bi]['style']}",
                         "turns": None if np.isnan(turns) else int(turns), "goal": f"{low}–{high}",
                         "met": bool(not np.isnan(turns) and low <= turns <= high)})
        return in_band, pace

    all_mask = np.ones(T, dtype=bool)
    final_idx = choose_index(all_mask)
    final_params = params_of(final_idx)
    tuned_in_band, tuned_pace = eval_rows(final_idx, np.ones(len(hits), bool), range(len(battles)))

    fold_rows, cv_in_band, cv_pace = [], [], []
    for t, name in enumerate(targets):
        mask = all_mask.copy()
        mask[t] = False
        idx = choose_index(mask)
        in_band, pace = eval_rows(idx, target_idx == t,
                                  [bi for bi, b in enumerate(battles) if b["target"] == name])
        cv_in_band.extend(in_band.tolist())
        cv_pace.extend(pace)
        fold_rows.append({"held-out target": name, **params_of(idx), "cases": int(in_band.size),
                          "held-out in-band": float(in_band.mean()) if in_band.size else np.nan,
                          "held-out pacing met": f"{sum(p['met'] for p in pace)}/{len(pace)}" if pace else "—"})

    edges = edge_report(final_params, DAMAGE_GRID, DAMAGE_NUMERIC_DIMS)
    summary = pd.DataFrame({
        "tuned (all targets)": {"in-band": float(np.mean(tuned_in_band)),
                                "pacing met": f"{sum(p['met'] for p in tuned_pace)}/{len(tuned_pace)}"},
        "LOTO-CV (held-out)": {"in-band": float(np.mean(cv_in_band)),
                               "pacing met": f"{sum(p['met'] for p in cv_pace)}/{len(cv_pace)}"},
    })
    result = {"params": damage_config_params(final_params), "grid_params": final_params, "edges": edges, "summary": summary,
              "folds": pd.DataFrame(fold_rows), "tuned_pace": pd.DataFrame(tuned_pace),
              "cv_pace": pd.DataFrame(cv_pace), "tuned_in_band": float(np.mean(tuned_in_band)),
              "cv_in_band": float(np.mean(cv_in_band)),
              "all_pacing_met": all(p["met"] for p in tuned_pace),
              "cv_all_pacing_met": all(p["met"] for p in cv_pace)}
    if verbose:
        print("Applied:", final_params)
        print("Edge check (True = on the grid edge):", edges)
        print("Tuned vs leave-one-target-out CV:")
        display(summary)
        print("Per-fold choices:")
        display(result["folds"].round(3))
        print("Pacing with the final parameters:")
        display(result["tuned_pace"])
    return result

def damage_config_params(grid_params: dict) -> dict:
    """Grid choice -> EngineConfig fields."""
    return {"noise_percentile": int(grid_params["noise_percentile"]),
            "ceiling_topk": int(grid_params["ceiling_topk"]),
            "relevance_min": float(grid_params["relevance_min"]),
            "damage_gamma": float(grid_params["gamma"]),
            "min_span_mode": grid_params["span_mode"]}

In [14]:
# 5.5 Run (or load) calibration and apply it to the engine
# What this cell does:
#   - Loads a matching tuned_params_<model>.json, or runs the gate + damage grids.
#   - Applies the tuned parameters to the engine and saves them.
CALIBRATION_FILE = PATHS.artifacts / f"tuned_params_{ACTIVE_MODEL}.json"

def to_jsonable(value):
    """Convert numpy / pandas values for json.dump."""
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if isinstance(value, pd.DataFrame):
        return {"__frame__": to_jsonable(value.to_dict(orient="split"))}
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, float) and value != value:
        return None
    return value

def from_jsonable(value):
    """Inverse of to_jsonable for the frames we store."""
    if isinstance(value, dict):
        if "__frame__" in value:
            f = value["__frame__"]
            return pd.DataFrame(f["data"], index=f["index"], columns=f["columns"])
        return {k: from_jsonable(v) for k, v in value.items()}
    if isinstance(value, list):
        return [from_jsonable(v) for v in value]
    return value

def calibration_fingerprint() -> str:
    """Hash of everything the grid searches depend on; a stale file is never reused."""
    payload = {
        "model": MODEL_SOURCE, "defaults": asdict(EngineConfig()),
        "gate_set": gate_set_df.to_dict(orient="list"), "cases": damage_cases_df.to_dict(orient="list"),
        "plays": REFERENCE_PLAYS, "gate_grid": GATE_GRID, "damage_grid": DAMAGE_GRID,
        "limits": [MAX_OUT_OF_DOMAIN_LEAK, MIN_IN_DOMAIN_PRIMARY, GATE_CV_FOLDS, SELECTION_TOL,
                   PACING_WEIGHT, STD_WEIGHT],
    }
    return hashlib.sha1(json.dumps(to_jsonable(payload), sort_keys=True).encode()).hexdigest()[:16]

def load_calibration(path: Path, fingerprint: str):
    """(gate_cal, damage_cal, comparison) from a saved file, or None with a WARNING."""
    if not path.is_file():
        print(f"WARNING: LOAD_TUNED_PARAMS = True but {path.name} does not exist — running the grid searches.")
        return None
    stored = json.loads(path.read_text())
    if stored.get("fingerprint") != fingerprint:
        print(f"WARNING: {path.name} was made with other data/settings — running the grid searches.")
        return None
    return from_jsonable(stored["gate"]), from_jsonable(stored["damage"]), from_jsonable(stored.get("comparison"))

def save_calibration(path: Path, fingerprint: str, gate_cal: dict, damage_cal: dict, comparison=None) -> None:
    """Write the calibration file (only when PERSISTENT)."""
    if not PERSISTENT:
        print("Calibration not saved: PERSISTENT = False (Drive not mounted).")
        return
    gate = {k: v for k, v in gate_cal.items() if k != "scored"}
    path.write_text(json.dumps(to_jsonable({"fingerprint": fingerprint, "gate": gate, "damage": damage_cal,
                                            "comparison": comparison}), indent=1))
    print(f"Calibration saved: {path}")

def score_gate_set(engine: LexicalCombatEngine, frame: pd.DataFrame) -> pd.DataFrame:
    """Gate result per row with the engine's thresholds (weak secondary shown as secondary, as in tuning)."""
    k = engine.config.anchor_topk
    inputs = gate_inputs(engine, frame, [k])[k]
    kinds = engine.gate_kinds(inputs["required"], inputs["lead"])
    scored = frame.copy()
    scored["required"], scored["lead"], scored["top_domain"] = inputs["required"], inputs["lead"], inputs["top_domain"]
    scored["gate"] = np.where(kinds == "weak_secondary", "secondary", kinds)
    scored["predicted"] = kinds_to_labels(scored["gate"])
    return scored

with section("5.5 Calibration") as _s:
    CALIBRATION_FINGERPRINT = calibration_fingerprint()
    _loaded = load_calibration(CALIBRATION_FILE, CALIBRATION_FINGERPRINT) if LOAD_TUNED_PARAMS else None
    STORED_COMPARISON = None
    if _loaded is None:
        CALIBRATION_SOURCE = "grid search"
        GATE_CAL = calibrate_gate(engine, gate_set_df)
        engine.config = replace(engine.config, **GATE_CAL["params"])
        # Preliminary leak split: damage still uses the pre-calibration curve (Section 6 repeats it).
        _gs = GATE_CAL["scored"]
        _need = ((_gs["label"] == "out_of_domain") & (_gs["gate"] != "out_of_domain")).to_numpy()
        _hits = np.zeros(len(_gs))
        if _need.any():
            _hits[_need] = damage_against_reference(engine, _gs.loc[_need, "phrase"], _gs.loc[_need, "domain"])
        print("Leak split on the tuning set (vs each domain's calibration target):",
              leak_breakdown(_gs["label"], _gs["gate"], _hits))
        DAMAGE_CAL = calibrate_damage(engine)
        engine.config = replace(engine.config, **DAMAGE_CAL["params"])
        save_calibration(CALIBRATION_FILE, CALIBRATION_FINGERPRINT, GATE_CAL, DAMAGE_CAL)
    else:
        CALIBRATION_SOURCE = f"loaded from {CALIBRATION_FILE.name}"
        GATE_CAL, DAMAGE_CAL, STORED_COMPARISON = _loaded
        engine.config = replace(engine.config, **GATE_CAL["params"], **DAMAGE_CAL["params"])
        GATE_CAL["scored"] = score_gate_set(engine, gate_set_df)
        print(f"Calibration {CALIBRATION_SOURCE} (set LOAD_TUNED_PARAMS = False to re-run the grids).")
        print("Gate:", GATE_CAL["grid_params"], "| edges:", GATE_CAL["edges"])
        display(pd.DataFrame({"tuned": GATE_CAL["tuned"], "cv": GATE_CAL["cv"]}).round(3))
        print("Damage:", DAMAGE_CAL["grid_params"], "| edges:", DAMAGE_CAL["edges"])
        display(DAMAGE_CAL["summary"])

    curve = pd.DataFrame({"relevance r": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]})
    curve["base hit %"] = [round(engine.base_damage(r), 1) for r in curve["relevance r"]]
    curve["tier (primary)"] = [engine.tier(p, True) for p in curve["base hit %"]]
    curve["secondary ×0.75 %"] = [round(float(engine.hit_percent(r, engine.config.domain_secondary_multiplier, True)), 1)
                                  for r in curve["relevance r"]]
    print(f"Damage curve after calibration (span floor {engine.span_floor:.3f}):")
    display(curve)

Top plateau settings at k=2:


,primary,secondary,gap,weak_extra,score,feasible,in_domain_primary,ambiguous_pass,out_leak,macro_f1,plateau
790,0.04,0.14,0.10,0.16,0.852,False,0.857,0.586,0.037,0.705,0.836
791,0.04,0.14,0.10,0.20,0.852,False,0.857,0.586,0.037,0.705,0.836
773,0.04,0.14,0.04,0.20,0.852,False,0.857,0.586,0.037,0.705,0.836
779,0.04,0.14,0.06,0.20,0.852,False,0.857,0.586,0.037,0.705,0.836
784,0.04,0.14,0.08,0.16,0.852,False,0.857,0.586,0.037,0.705,0.836
785,0.04,0.14,0.08,0.20,0.852,False,0.857,0.586,0.037,0.705,0.836
795,0.04,0.14,0.12,0.12,0.852,False,0.857,0.586,0.037,0.705,0.836
796,0.04,0.14,0.12,0.16,0.852,False,0.857,0.586,0.037,0.705,0.836
797,0.04,0.14,0.12,0.20,0.852,False,0.857,0.586,0.037,0.705,0.836
767,0.04,0.14,0.02,0.20,0.852,False,0.857,0.586,0.037,0.705,0.835


Applied gate: k=2 | primary>=0.04 (and #1) | secondary>=0.14 gap<=0.08 | weak gap<=0.20
Edge check (True = on the grid edge): {'primary': False, 'secondary': False, 'gap': False, 'weak_extra': False}
Tuned set vs 5-fold grouped CV:


,tuned,cv
accuracy,0.891,0.878
balanced_accuracy,0.676,0.647
macro_f1,0.705,0.663
in_domain_primary,0.857,0.848
in_domain_pass,0.857,0.848
ambiguous_pass,0.586,0.483
out_block,0.963,0.954
out_leak,0.037,0.046


Per-fold choices (stability):


,fold,k,primary,secondary,gap,weak_extra,test rows
0,0,2,0.04,0.14,0.06,0.16,94
1,1,2,0.04,0.14,0.08,0.16,103
2,2,3,0.04,0.14,0.06,0.12,103
3,3,3,0.02,0.12,0.08,0.12,88
4,4,2,0.04,0.14,0.06,0.08,79


... (4 outputs left out)

Pacing with the final parameters:


,battle,turns,goal,met
0,executive/expert,4,3–4,True
1,executive/typical,8,5–8,True
2,algorithm/expert,4,3–4,True
3,algorithm/typical,8,5–8,True
4,catalyst/expert,3,3–4,True
5,catalyst/typical,8,5–8,True


Calibration not saved: PERSISTENT = False (Drive not mounted).
Damage curve after calibration (span floor 0.150):


,relevance r,base hit %,tier (primary),secondary ×0.75 %
0,0.0,0.0,NO EFFECT,0.0
1,0.1,7.5,WEAK GRAZE,5.6
2,0.2,10.8,WEAK GRAZE,8.1
3,0.3,14.4,SOLID HIT,10.8
4,0.4,18.3,SOLID HIT,13.7
5,0.5,22.4,SOLID HIT,16.8
6,0.6,26.7,STRONG HIT,20.0
7,0.7,31.1,STRONG HIT,23.3
8,0.8,35.6,STRONG HIT,26.7
9,0.9,40.2,CRITICAL STRIKE,30.2


In [15]:
# 5.6 Gate review with the tuned thresholds (confusion matrix, anchor-echo sanity set, errors)
# What this cell does:
#   - Confusion matrix of the tuned gate on the calibration set.
#   - Anchor-echo sanity set and cross-domain ambiguous phrases.
#   - Lists the worst gate errors (leaks first).
with section("5.6 Gate review"):
    gate_scores = GATE_CAL["scored"]
    print("Confusion matrix (rows = human, columns = predicted):")
    display(pd.crosstab(gate_scores["label"], gate_scores["predicted"]).reindex(
        index=LABELS, columns=LABELS, fill_value=0))

    _echo_inputs = gate_inputs(engine, anchor_echo_df, [engine.config.anchor_topk])[engine.config.anchor_topk]
    _echo_kinds = engine.gate_kinds(_echo_inputs["required"], _echo_inputs["lead"])
    print("Anchor-echo sanity set:", {**pass_block_metrics(anchor_echo_df["label"], _echo_kinds),
                                      "macro_f1": classification_metrics(anchor_echo_df["label"],
                                                                         kinds_to_labels(_echo_kinds))["macro_f1"]})
    print("CROSS_DOMAIN_AMBIGUOUS with the chosen gate:")
    _amb = anchor_echo_df.assign(gate=_echo_kinds)
    display(_amb[_amb["source"] == "CROSS_DOMAIN_AMBIGUOUS"][["phrase", "domain", "label", "gate"]])

    print("Rows to review (leaks first, then wrongly blocked in-domain phrases):")
    errors = gate_scores[(gate_scores["predicted"] != gate_scores["label"])].copy()
    errors["severity"] = np.select(
        [(errors["label"] == "out_of_domain"), (errors["predicted"] == "out_of_domain")],
        ["1 leak", "2 wrongly blocked"], "3 tier mismatch")
    display(errors.sort_values(["severity", "required"], ascending=[True, False])[
        ["severity", "phrase", "domain", "label", "gate", "top_domain", "required", "lead", "source"]
    ].round(3).head(30))

Confusion matrix (rows = human, columns = predicted):


predicted,in_domain,ambiguous,out_of_domain
label,,,
in_domain,96,0,16
ambiguous,11,6,12
out_of_domain,12,0,314


Anchor-echo sanity set: {'in_domain_primary': 1.0, 'in_domain_pass': 1.0, 'ambiguous_pass': 0.2727272727272727, 'out_block': 0.98, 'out_leak': 0.020000000000000018, 'macro_f1': 0.7350453214530885}
CROSS_DOMAIN_AMBIGUOUS with the chosen gate:


,phrase,domain,label,gate
11,director,Arts,ambiguous,weak_secondary
30,algorithm,Science,ambiguous,out_of_domain
54,machine learning,Science,ambiguous,out_of_domain
70,neural network,Science,ambiguous,out_of_domain
97,thermodynamics,Technology,ambiguous,out_of_domain
101,electron,Technology,ambiguous,out_of_domain


Rows to review (leaks first, then wrongly blocked in-domain phrases):


,severity,phrase,domain,label,gate,top_domain,required,lead,source
431,1 leak,necklace,Arts,out_of_domain,primary,Arts,0.159,0.129,FRESH_NOISE_OUT
185,1 leak,virtual reality,Science,out_of_domain,primary,Science,0.124,0.035,FRESH_CROSS_OUT
395,1 leak,lighthouse,Arts,out_of_domain,primary,Arts,0.105,0.124,FRESH_NOISE_OUT
202,1 leak,printer,Arts,out_of_domain,primary,Arts,0.090,0.039,FRESH_CROSS_OUT
411,1 leak,volleyball,Arts,out_of_domain,primary,Arts,0.079,0.039,FRESH_NOISE_OUT
466,1 leak,algebra,Arts,out_of_domain,primary,Arts,0.074,0.005,BANDED_CASES
399,1 leak,sailboat,Arts,out_of_domain,primary,Arts,0.070,0.047,FRESH_NOISE_OUT
419,1 leak,waterfall,Arts,out_of_domain,primary,Arts,0.068,0.017,FRESH_NOISE_OUT
402,1 leak,cactus,Science,out_of_domain,primary,Science,0.068,0.060,FRESH_NOISE_OUT
316,1 leak,sonnet,Technology,out_of_domain,primary,Technology,0.049,0.059,FRESH_CROSS_OUT


## 6. Report
Banded cases, gate leak vs damaging leak, pacing per stage, tuned vs CV, Base vs Fine-tuned (each recalibrated), and the active parameters.

In [16]:
# 6. Report: banded cases, leak split, pacing, calibration summary, Base vs Fine-tuned
# What this cell does:
#   - Runs the calibration cases through evaluate_attack().
#   - Banded cases, gate leak vs damaging leak, pacing per stage, tuned vs CV.
#   - Base vs Fine-tuned comparison (each model recalibrated).
COMPARE_WITH_RECALIBRATION = True   # each model gets its own V3 + V4 calibration in the comparison

def snapshot_params(engine: LexicalCombatEngine) -> dict:
    """Tuned parameters under their report names."""
    c = engine.config
    return {"GATE_PRIMARY_MIN": c.gate_primary_min, "GATE_SECONDARY_MIN": c.gate_secondary_min,
            "GATE_SECONDARY_MAX_GAP": c.gate_secondary_max_gap, "GATE_WEAK_MAX_GAP": c.gate_weak_max_gap,
            "ANCHOR_TOPK": c.anchor_topk, "NOISE_PERCENTILE": c.noise_percentile,
            "CEILING_TOPK": c.ceiling_topk, "RELEVANCE_MIN": c.relevance_min,
            "DAMAGE_GAMMA": c.damage_gamma, "MIN_SPAN_MODE": c.min_span_mode}

def gate_kinds_for(engine: LexicalCombatEngine, frame: pd.DataFrame) -> np.ndarray:
    """Gate kinds for a (phrase, domain) frame with the engine's thresholds."""
    k = engine.config.anchor_topk
    inputs = gate_inputs(engine, frame, [k])[k]
    return engine.gate_kinds(inputs["required"], inputs["lead"])

def intrinsic_metrics(engine: LexicalCombatEngine) -> dict:
    """Model-level numbers that do not depend on tuned thresholds."""
    pos, neg, blocked, rel_blocked = [], [], [], []
    for row in damage_cases_df.itertuples(index=False):
        cal = engine.target_calibration(row.target, row.required_domain)
        sim = float(engine.encode([clean_text(row.guess)])[0] @ cal["vector"])
        if row.human_expected_tier in ("strong", "medium"):
            pos.append(sim)
        if row.human_expected_tier == "blocked":
            noise, ceiling = engine.bounds(cal)
            blocked.append(sim)
            rel_blocked.append(float(normalized_relevance(sim, noise, ceiling)))
    for target, domain in set(zip(damage_cases_df["target"], damage_cases_df["required_domain"])):
        neg.extend(engine.target_calibration(target, domain)["noise_sims"].tolist())
    gate_in = gate_set_df[gate_set_df["label"] != "ambiguous"]
    k = engine.config.anchor_topk
    g = gate_inputs(engine, gate_in, [k])[k]
    return {"AUC related vs noise": roc_auc_score(np.r_[np.ones(len(pos)), np.zeros(len(neg))], np.r_[pos, neg]),
            "Anisotropy (noise-pair mean cos)": engine.noise_pair_mean,
            "Mean cos strong/medium": float(np.mean(pos)),
            "Out-of-domain mean cos (blocked pairs)": float(np.mean(blocked)),
            "Mean r blocked": float(np.mean(rel_blocked)),
            "Gate AUC in vs out (debiased score)": roc_auc_score((gate_in["label"] == "in_domain").astype(int),
                                                                 g["required"])}

def compare_models(tuned_config: EngineConfig, recalibrate: bool = COMPARE_WITH_RECALIBRATION) -> pd.DataFrame:
    """One row per available model; each starts from the tuned config, then (optionally) recalibrates."""
    rows = []
    for choice in available_model_choices():
        candidate_model, _, _ = load_model(choice)
        candidate = LexicalCombatEngine(candidate_model, ENGINE_DATA, tuned_config)
        row = {"Model": "Base" if choice == "base" else "Fine-tuned", **intrinsic_metrics(candidate)}
        if recalibrate:
            gate = calibrate_gate(candidate, gate_set_df, verbose=False)
            candidate.config = replace(candidate.config, **gate["params"])
            damage = calibrate_damage(candidate, verbose=False)
            row.update({"Gate macro F1 (tuned / CV)": f"{gate['tuned']['macro_f1']:.3f} / {gate['cv']['macro_f1']:.3f}",
                        "In-domain primary (CV)": gate["cv"]["in_domain_primary"],
                        "Damage in-band (tuned / CV)": f"{damage['tuned_in_band']:.3f} / {damage['cv_in_band']:.3f}",
                        "Pacing met (tuned)": damage["summary"].loc["pacing met", "tuned (all targets)"]})
        rows.append(row)
    return pd.DataFrame(rows).set_index("Model")

with section("6. Report"):
    # 6.1 Banded cases through the real pipeline
    v2_eval = evaluate_test_set(engine, damage_cases_df)
    v2_eval["Expected tier"] = damage_cases_df["human_expected_tier"].values
    v2_eval["Target band"] = v2_eval["Expected tier"].map(lambda t: f"{TIER_BANDS_V2[t][0]}–{TIER_BANDS_V2[t][1]}%")
    v2_eval["In band"] = [band_distance(p, TIER_BANDS_V2[t]) == 0 if p == p else False
                          for p, t in zip(v2_eval.get("Hit %", pd.Series(np.nan, index=v2_eval.index)),
                                          v2_eval["Expected tier"])]
    print("Banded cases through evaluate_attack():")
    display(v2_eval[["Target", "Guess", "Human Label", "Domain Match", "Label Aligned", "Cosine Similarity",
                     "Relevance r", "Hit %", "Predicted Tier", "Target band", "In band"]])
    print("In-band rate per target:")
    display(v2_eval.groupby("Target")["In band"].agg(["mean", "size"]).round(3))
    print(f"Gate label alignment: {v2_eval['Label Aligned'].mean():.1%} | "
          f"In damage band: {v2_eval['In band'].mean():.1%}")

    # 6.2 Leak split: gate leak (passes, 0 damage) vs damaging leak (passes and damages)
    leak_rows = []
    for name, frame in [("gate tuning set", gate_set_df), ("anchor-echo set", anchor_echo_df)]:
        kinds = gate_kinds_for(engine, frame)
        hits = np.zeros(len(frame))
        need = ((frame["label"] == "out_of_domain") & (kinds != "out_of_domain")).to_numpy()
        if need.any():
            hits[need] = damage_against_reference(engine, frame.loc[need, "phrase"], frame.loc[need, "domain"])
        leak_rows.append({"set": name, **leak_breakdown(frame["label"], kinds, hits)})
    leak_rows.append({"set": "banded cases", **leak_breakdown(v2_eval["Human Label"],
                                                              v2_eval["Domain Match"].fillna("out_of_domain"),
                                                              v2_eval["Hit %"].fillna(0))})
    print("\nLeak split:")
    display(pd.DataFrame(leak_rows).round(3))

    # 6.3 Whole battles: pacing per stage
    pacing_rows = []
    for stage in GAME_STAGES:
        t = stage["target"]
        plays = {style: guesses for style, (guesses, _) in REFERENCE_PLAYS[t].items()}
        plays["playlog"] = PLAYER_LOG_GUESSES_V2[t]
        for style, guesses in plays.items():
            battle = engine.simulate_battle(t, stage["domain"], stage["hp"], guesses)
            if style == "typical":
                print(f"\n{stage['enemy']} — typical play:")
                display(battle)
            target_range = REFERENCE_PLAYS[t].get(style, (None, None))[1]
            summary = engine.summarize_battle(battle, stage["hp"])
            met = (target_range is not None and summary["Turns to defeat"] is not None
                   and target_range[0] <= summary["Turns to defeat"] <= target_range[1])
            pacing_rows.append({"Stage": stage["stage"], "Enemy": stage["enemy"], "Play": style,
                                "Goal turns": f"{target_range[0]}–{target_range[1]}" if target_range else "—",
                                "Goal met": met if target_range else "—", **summary})
    pacing_df = pd.DataFrame(pacing_rows)
    print("\nPacing summary (expert 3–4, typical 5–8):")
    display(pacing_df)

    # 6.4 Calibration summary (tuned vs CV, edge checks)
    calibration_summary = pd.DataFrame([
        {"part": "gate", "metric": "macro F1", "tuned": GATE_CAL["tuned"]["macro_f1"], "cv": GATE_CAL["cv"]["macro_f1"]},
        {"part": "gate", "metric": "in-domain primary", "tuned": GATE_CAL["tuned"]["in_domain_primary"],
         "cv": GATE_CAL["cv"]["in_domain_primary"]},
        {"part": "gate", "metric": "ambiguous pass", "tuned": GATE_CAL["tuned"]["ambiguous_pass"],
         "cv": GATE_CAL["cv"]["ambiguous_pass"]},
        {"part": "gate", "metric": "out-of-domain leak", "tuned": GATE_CAL["tuned"]["out_leak"],
         "cv": GATE_CAL["cv"]["out_leak"]},
        {"part": "damage", "metric": "in-band", "tuned": DAMAGE_CAL["tuned_in_band"], "cv": DAMAGE_CAL["cv_in_band"]},
        {"part": "damage", "metric": "all pacing goals met", "tuned": DAMAGE_CAL["all_pacing_met"],
         "cv": DAMAGE_CAL["cv_all_pacing_met"]},
    ])
    print(f"\nCalibration ({CALIBRATION_SOURCE}): tuned set vs cross-validation")
    display(calibration_summary)
    PARAMS_ON_EDGE = {**{f"gate.{k}": v for k, v in GATE_CAL["edges"].items()},
                      **{f"damage.{k}": v for k, v in DAMAGE_CAL["edges"].items()}}
    print("Parameters on a grid edge:", [k for k, v in PARAMS_ON_EDGE.items() if v] or "none")

    # 6.5 Base vs Fine-tuned (reused from the calibration file when it matches)
    _choices = available_model_choices()
    if STORED_COMPARISON and STORED_COMPARISON.get("choices") == _choices:
        model_comparison = STORED_COMPARISON["table"].rename_axis("Model")
        print("\nBase vs Fine-tuned (loaded with the calibration file):")
    else:
        model_comparison = compare_models(engine.config)
        if PERSISTENT and CALIBRATION_SOURCE == "grid search":
            save_calibration(CALIBRATION_FILE, CALIBRATION_FINGERPRINT, GATE_CAL, DAMAGE_CAL,
                             {"choices": _choices, "table": model_comparison})
        print("\nBase vs Fine-tuned (each model with its own calibration when recalibrate=True):")
    display(model_comparison.round(3))
    FINE_TUNE_VERDICT = None
    if "Fine-tuned" in model_comparison.index:
        b, f = model_comparison.loc["Base"], model_comparison.loc["Fine-tuned"]
        passed = (f["AUC related vs noise"] >= b["AUC related vs noise"]
                  and f["Anisotropy (noise-pair mean cos)"] <= b["Anisotropy (noise-pair mean cos)"])
        FINE_TUNE_VERDICT = "PASS" if passed else "NEGATIVE RESULT"
        print(f"Fine-tuning criterion (AUC >= Base and anisotropy <= Base): {FINE_TUNE_VERDICT}")
        if not passed:
            print("Report as a negative result: fine-tuning did not improve ranking and/or increased "
                  "anisotropy; Base remains the main model.")
    else:
        print("Fine-tuned checkpoint not found; only Base is shown.")

    print("\nActive parameters:")
    c = engine.config
    display(pd.Series({**snapshot_params(engine), "SPAN_FLOOR": round(engine.span_floor, 3),
                       "MAX_HIT_PERCENT": c.max_hit_percent, "GENERIC_PENALTY": c.generic_penalty,
                       "PRIMARY_GRAZE_FLOOR": c.primary_graze_floor, "MODEL_CHOICE": ACTIVE_MODEL,
                       "Echo threshold (raw cos) / penalty": f"{engine.echo_threshold():.2f} / {c.echo_penalty}"})
            .to_frame("value"))

Banded cases through evaluate_attack():


,Target,Guess,Human Label,Domain Match,Label Aligned,Cosine Similarity,Relevance r,Hit %,Predicted Tier,Target band,In band
0,executive,ceo,in_domain,primary,True,0.726,1.00,45.0,CRITICAL STRIKE,30–45%,True
1,executive,director,in_domain,primary,True,0.492,0.53,23.8,SOLID HIT,30–45%,False
2,executive,business logic,in_domain,primary,True,0.281,0.11,8.0,WEAK GRAZE,4–14%,True
3,executive,logic,out_of_domain,out_of_domain,True,0.260,0.07,0.0,BLOCKED — OUT OF DOMAIN,0–0%,True
4,executive,logic gate,out_of_domain,out_of_domain,True,0.231,0.01,0.0,BLOCKED — OUT OF DOMAIN,0–0%,True
5,executive,banana,out_of_domain,out_of_domain,True,0.218,0.00,0.0,BLOCKED — OUT OF DOMAIN,0–0%,True
6,algorithm,data structure,in_domain,primary,True,0.374,0.42,19.0,SOLID HIT,30–45%,False
7,algorithm,source code,in_domain,primary,True,0.334,0.33,15.5,SOLID HIT,14–30%,True
8,algorithm,logic gate,in_domain,primary,True,0.343,0.35,16.2,SOLID HIT,14–30%,True
9,algorithm,business logic,ambiguous,weak_secondary,True,0.204,0.04,4.0,WEAK GRAZE,4–14%,True


In-band rate per target:


,mean,size
Target,,
algorithm,0.615,13
catalyst,0.500,12
executive,0.714,14
painting,0.727,11


Gate label alignment: 92.0% | In damage band: 64.0%

Leak split:


,set,out_of_domain rows,gate leak (0 dmg),damaging leak,gate leak %,damaging leak %
0,gate tuning set,326,4,8,0.012,0.025
1,anchor-echo set,100,2,0,0.020,0.000
2,banded cases,14,0,1,0.000,0.071


... (10 outputs left out)

Parameters on a grid edge: ['damage.ceiling_topk', 'damage.relevance_min']

Base vs Fine-tuned (each model with its own calibration when recalibrate=True):


,AUC related vs noise,Anisotropy (noise-pair mean cos),Mean cos strong/medium,Out-of-domain mean cos (blocked pairs),Mean r blocked,Gate AUC in vs out (debiased score),Gate macro F1 (tuned / CV),In-domain primary (CV),Damage in-band (tuned / CV),Pacing met (tuned)
Model,,,,,,,,,,
Base,0.99,0.291,0.461,0.206,0.04,0.952,0.705 / 0.663,0.848,0.640 / 0.600,6/6


Fine-tuned checkpoint not found; only Base is shown.

Active parameters:


,value
GATE_PRIMARY_MIN,0.04
GATE_SECONDARY_MIN,0.14
GATE_SECONDARY_MAX_GAP,0.08
GATE_WEAK_MAX_GAP,0.2
ANCHOR_TOPK,2
NOISE_PERCENTILE,65
CEILING_TOPK,1
RELEVANCE_MIN,0.0
DAMAGE_GAMMA,1.2
MIN_SPAN_MODE,fixed


## 7. Holdout
Labels: inline `HOLDOUT_ANNOTATIONS` plus any `annotator_*.xlsx` / `.csv` in the labels folder or `/content`. 0 complete sets → draft; 1 → single annotator; ≥ 2 → majority vote with kappa. A final run log is written only when Drive is mounted **and** labels come from ≥ 2 independent annotators; otherwise results are marked **NOT FINAL**.

In [17]:
# 7. Holdout: labels (inline + annotator files), majority vote / kappa, final evaluation
# What this cell does:
#   - Collects holdout labels (inline + optional annotator files), majority vote, kappa.
#   - Evaluates the active model on the holdout once.
#   - Checks the success criteria and logs a run fingerprint.
HOLDOUT_ITEMS = [(t, g, d) for t, g, d, _, _ in HOLDOUT_CASES_DRAFT]   # labels hidden from annotators
HOLDOUT_RUN_LOG = PATHS.artifacts / "holdout_final_run.json"

def export_label_sheets(names=("annotator_1", "annotator_2", "annotator_3")) -> None:
    """Write blank CSV sheets (no draft labels) to PATHS.labels for independent labelling."""
    sheet = pd.DataFrame(HOLDOUT_ITEMS, columns=["target", "guess", "domain"])
    sheet.insert(0, "item", range(len(sheet)))
    sheet["gate_label"], sheet["tier"] = "", ""
    for name in names:
        sheet.to_csv(PATHS.labels / f"{name}.csv", index=False)
    print(f"Wrote {len(names)} sheets to {PATHS.labels}. Gate labels {GATE_CHOICES}, tiers {TIER_CHOICES}.")

def read_label_sheet(path: Path) -> pd.DataFrame:
    """One annotator file (.xlsx sheet 'Labels' or .csv)."""
    return pd.read_excel(path, sheet_name="Labels") if path.suffix == ".xlsx" else pd.read_csv(path)

def collect_annotations() -> dict:
    """Inline HOLDOUT_ANNOTATIONS plus annotator_* files (files override the same name). Returns a new dict."""
    annotations = {name: list(values) for name, values in HOLDOUT_ANNOTATIONS.items()}
    paths = sorted(p for folder in LABEL_SEARCH_DIRS for ext in ("xlsx", "csv")
                   for p in folder.glob(f"annotator_*.{ext}"))
    for path in paths:
        sheet = read_label_sheet(path)
        sheet = sheet[pd.to_numeric(sheet["item"], errors="coerce").notna()].sort_values("item")
        if len(sheet) != len(HOLDOUT_ITEMS):
            print(f"WARNING: {path.name} has {len(sheet)} rows, expected {len(HOLDOUT_ITEMS)} — file ignored.")
            continue
        values = [(str(g).strip(), str(t).strip())
                  if isinstance(g, str) and g.strip() and isinstance(t, str) and t.strip() else None
                  for g, t in zip(sheet["gate_label"], sheet["tier"])]
        if any(v is not None for v in values):
            annotations[path.stem] = values
            print(f"Loaded {path.stem} from {path}: {sum(v is not None for v in values)}/{len(values)} filled")
    return annotations

def fleiss_kappa(labels_by_rater, categories):
    """Fleiss' kappa for items labelled by the same number of raters."""
    counts = np.array([[sum(1 for r in item if r == c) for c in categories] for item in labels_by_rater])
    n = counts.sum(axis=1)[0]
    if n < 2:
        return np.nan
    p_j = counts.sum(axis=0) / counts.sum()
    p_i = ((counts ** 2).sum(axis=1) - n) / (n * (n - 1))
    p_bar, p_e = p_i.mean(), (p_j ** 2).sum()
    return float((p_bar - p_e) / (1 - p_e)) if p_e < 1 else 1.0

def majority(values):
    """Value with >= 2 votes, else None (needs discussion)."""
    counts = pd.Series(values).value_counts()
    return counts.index[0] if counts.iloc[0] >= 2 else None

def consolidate_labels(annotations: dict) -> tuple:
    """(cases, status, agreement): majority vote, single annotator, or the draft."""
    raters = {}
    for n, v in annotations.items():
        if not all(x is not None for x in v):
            continue
        bad = [(g, t) for g, t in v if g not in GATE_CHOICES or t not in TIER_CHOICES]
        if bad:
            print(f"WARNING: {n} has invalid labels (e.g. {bad[0]}) — annotator ignored.")
            continue
        raters[n] = v
    if not raters:
        print("WARNING: no complete annotation set — using the DRAFT labels (status 'draft').")
        return list(HOLDOUT_CASES_DRAFT), "draft", None
    if len(raters) == 1:
        name, labels = next(iter(raters.items()))
        draft = [(l, t) for _, _, _, l, t in HOLDOUT_CASES_DRAFT]
        tier_rank = {t: i for i, t in enumerate(TIER_CHOICES)}
        agreement = {"annotators": 1,
                     "note": "single human annotator; kappa below is human vs AI draft, not inter-rater",
                     "Cohen kappa gate (human vs AI draft)": cohen_kappa_score([x[0] for x in labels], [x[0] for x in draft]),
                     "Weighted kappa tier (human vs AI draft)": cohen_kappa_score(
                         [tier_rank[x[1]] for x in labels], [tier_rank[x[1]] for x in draft], weights="quadratic")}
        print(f"Single annotator ({name}): these labels are used as final.")
        cases = [(t, g, d, gate, tier) for (t, g, d), (gate, tier) in zip(HOLDOUT_ITEMS, labels)]
        return cases, "single_annotator", agreement
    names = list(raters)
    gates = [[raters[n][i][0] for n in names] for i in range(len(HOLDOUT_ITEMS))]
    tiers = [[raters[n][i][1] for n in names] for i in range(len(HOLDOUT_ITEMS))]
    agreement = {"annotators": len(names),
                 "Fleiss kappa (gate)": fleiss_kappa(gates, GATE_CHOICES),
                 "Fleiss kappa (tier)": fleiss_kappa(tiers, TIER_CHOICES)}
    tier_rank = {t: i for i, t in enumerate(TIER_CHOICES)}
    for a, b in itertools.combinations(names, 2):
        agreement[f"Cohen kappa gate {a}–{b}"] = cohen_kappa_score([x[0] for x in raters[a]], [x[0] for x in raters[b]])
        agreement[f"Weighted kappa tier {a}–{b}"] = cohen_kappa_score(
            [tier_rank[x[1]] for x in raters[a]], [tier_rank[x[1]] for x in raters[b]], weights="quadratic")
    cases, disputed = [], []
    for (t, g, d), gate_votes, tier_votes in zip(HOLDOUT_ITEMS, gates, tiers):
        gate, tier = majority(gate_votes), majority(tier_votes)
        if gate is None or tier is None:
            disputed.append({"target": t, "guess": g, "gate votes": gate_votes, "tier votes": tier_votes})
            continue
        cases.append((t, g, d, gate, tier))
    if disputed:
        print("No majority (discuss, then add to HOLDOUT_ANNOTATIONS):")
        display(pd.DataFrame(disputed))
    return cases, "majority_vote", agreement

def evaluate_holdout(engine: LexicalCombatEngine, cases: list, status: str) -> tuple:
    """Run holdout cases through the engine; returns (result table, summary dict)."""
    holdout_df = pd.DataFrame(cases, columns=[
        "target", "guess", "required_domain", "human_label", "human_expected_tier"])
    seen = set(zip(damage_cases_df["target"].map(clean_text), damage_cases_df["guess"].map(clean_text)))
    overlap = [p for p in zip(holdout_df["target"].map(clean_text), holdout_df["guess"].map(clean_text)) if p in seen]
    if overlap:
        print("WARNING: holdout pairs also used for calibration:", overlap)

    result = evaluate_test_set(engine, holdout_df)
    result["Expected tier"] = holdout_df["human_expected_tier"].values
    hits = result["Hit %"].fillna(0) if "Hit %" in result else pd.Series(0.0, index=result.index)
    result["In band"] = [band_distance(p, TIER_BANDS_V2[t]) == 0 for p, t in zip(hits, result["Expected tier"])]
    kinds = result["Domain Match"].fillna("out_of_domain").to_numpy()
    metrics = classification_metrics(result["Human Label"], result["Predicted Label"])
    gate_feel = pass_block_metrics(result["Human Label"].to_numpy(), kinds)
    leaks = leak_breakdown(result["Human Label"], kinds, hits)
    summary = {"cases": len(result), "label status": status,
               "gate accuracy": metrics["accuracy"], "gate macro F1": metrics["macro_f1"],
               "in-domain -> primary": gate_feel["in_domain_primary"],
               "ambiguous pass": gate_feel["ambiguous_pass"],
               "gate leak (0 dmg)": leaks["gate leak (0 dmg)"], "damaging leak": leaks["damaging leak"],
               "damage in band": float(result["In band"].mean())}
    print("Holdout summary:")
    display(pd.Series(summary).to_frame("value"))
    display(pd.crosstab(result["Human Label"], result["Predicted Label"]))
    columns = ["Target", "Guess", "Human Label", "Domain Match", "Predicted Label", "Cosine Similarity",
               "Relevance r", "Hit %", "Predicted Tier", "Expected tier", "In band"]
    display(result[[c for c in columns if c in result]])
    print("Ceiling terms per holdout target:")
    for target, domain in sorted(set(zip(holdout_df["target"], holdout_df["required_domain"]))):
        cal = engine.target_calibration(target, domain)
        noise, ceiling = engine.bounds(cal)
        top = sorted(zip(cal["pool_sims"], cal["pool_terms"]), reverse=True)[:engine.config.ceiling_topk]
        print(f"  {target:<11} noise {noise:.3f} | ceiling {ceiling:.3f} | span {ceiling - noise:.3f} | "
              f"{', '.join(t for _, t in top)}")
    return result, summary

def success_criteria(summary: dict, pacing_df: pd.DataFrame, damage_cal: dict, params_on_edge: dict) -> pd.DataFrame:
    """Project success criteria (holdout + calibration/CV)."""
    pacing_ok = bool((pacing_df.loc[pacing_df["Goal met"] != "—", "Goal met"] == True).all())
    rows = [
        ("Holdout gate macro F1 >= 0.75", summary["gate macro F1"], summary["gate macro F1"] >= 0.75),
        ("Holdout damaging leak = 0", summary["damaging leak"], summary["damaging leak"] == 0),
        ("Holdout in-domain -> primary >= 80%", summary["in-domain -> primary"], summary["in-domain -> primary"] >= 0.80),
        ("Holdout damage in-band >= 70%", summary["damage in band"], summary["damage in band"] >= 0.70),
        ("Pacing goals met (Section 6 battles)", pacing_ok, pacing_ok),
        ("Pacing goals met (LOTO-CV)", damage_cal["cv_all_pacing_met"], damage_cal["cv_all_pacing_met"]),
        ("No tuned parameter on a grid edge", [k for k, v in params_on_edge.items() if v] or "none",
         not any(params_on_edge.values())),
    ]
    return pd.DataFrame(rows, columns=["criterion", "value", "pass"])

def run_fingerprint(engine: LexicalCombatEngine, cases: list) -> str:
    """Identifies one (model, parameters, labels) holdout run."""
    payload = json.dumps({"model": MODEL_SOURCE, "params": {k: str(v) for k, v in snapshot_params(engine).items()},
                          "cases": cases}, sort_keys=True)
    return hashlib.sha1(payload.encode()).hexdigest()[:12]

with section("7. Holdout") as _s:
    if not RUN_HOLDOUT:
        _s.skip("RUN_HOLDOUT = False")
        HOLDOUT_LABEL_STATUS, holdout_result, holdout_summary = "not run", None, None
    else:
        HOLDOUT_CASES, HOLDOUT_LABEL_STATUS, HOLDOUT_AGREEMENT = consolidate_labels(collect_annotations())
        print(f"Holdout labels: {HOLDOUT_LABEL_STATUS} | {len(HOLDOUT_CASES)} cases")
        if HOLDOUT_AGREEMENT:
            display(pd.Series(HOLDOUT_AGREEMENT).to_frame("value").round(3))
            _draft = {(t, g): (l, tr) for t, g, _, l, tr in HOLDOUT_CASES_DRAFT}
            _changed = [(t, g, _draft[(t, g)], (l, tr)) for t, g, _, l, tr in HOLDOUT_CASES if _draft[(t, g)] != (l, tr)]
            print("Draft -> final label changes:", _changed or "none")
        IS_FINAL = PERSISTENT and HOLDOUT_LABEL_STATUS == "majority_vote"
        if not IS_FINAL:
            why = [] if PERSISTENT else ["PERSISTENT = False"]
            if HOLDOUT_LABEL_STATUS != "majority_vote":
                why.append(f"label status '{HOLDOUT_LABEL_STATUS}' (needs >= 2 independent annotators)")
            print("=" * 72 + f"\nNOT FINAL — {'; '.join(why)}. No run log is written.\n" + "=" * 72)
        holdout_result, holdout_summary = evaluate_holdout(engine, HOLDOUT_CASES, HOLDOUT_LABEL_STATUS)
        criteria = success_criteria(holdout_summary, pacing_df, DAMAGE_CAL, PARAMS_ON_EDGE)
        print("Success criteria:")
        display(criteria)
        if IS_FINAL:
            fingerprint = run_fingerprint(engine, HOLDOUT_CASES)
            previous = json.loads(HOLDOUT_RUN_LOG.read_text()) if HOLDOUT_RUN_LOG.is_file() else None
            if previous is None:
                HOLDOUT_RUN_LOG.write_text(json.dumps({"fingerprint": fingerprint, "summary":
                                                       {k: str(v) for k, v in holdout_summary.items()}}, indent=2))
                print(f"Final run recorded: {fingerprint}")
            elif previous.get("fingerprint") != fingerprint:
                print(f"WARNING: the holdout was already evaluated with other settings ({previous.get('fingerprint')}); "
                      "these numbers are NOT a clean final result.")
            else:
                print(f"Same settings as the recorded final run ({fingerprint}).")

Single annotator (annotator_1): these labels are used as final.
Holdout labels: single_annotator | 28 cases


,value
annotators,1
note,"single human annotator; kappa below is human vs AI draft, not inter-rater"
Cohen kappa gate (human vs AI draft),0.93578
Weighted kappa tier (human vs AI draft),0.855535


Draft -> final label changes: [('brand', 'logo', ('in_domain', 'medium'), ('in_domain', 'strong')), ('brand', 'packaging', ('in_domain', 'weak'), ('in_domain', 'medium')), ('email', 'spam filter', ('in_domain', 'medium'), ('in_domain', 'strong')), ('email', 'mail server', ('in_domain', 'medium'), ('in_domain', 'strong')), ('email', 'chat app', ('in_domain', 'weak'), ('in_domain', 'medium')), ('email', 'newsletter', ('ambiguous', 'weak'), ('in_domain', 'medium')), ('comet', 'orbit', ('in_domain', 'medium'), ('in_domain', 'strong')), ('comet', 'light year', ('in_domain', 'weak'), ('in_domain', 'medium')), ('cartoon', 'caricature', ('in_domain', 'medium'), ('in_domain', 'strong')), ('cartoon', 'animated film', ('in_domain', 'medium'), ('in_domain', 'strong')), ('cartoon', 'puppet show', ('in_domain', 'weak'), ('in_domain', 'medium'))]
NOT FINAL — PERSISTENT = False; label status 'single_annotator' (needs >= 2 independent annotators). No run log is written.
Holdout summary:


,value
cases,28
label status,single_annotator
gate accuracy,0.678571
gate macro F1,0.470186
in-domain -> primary,0.705882
ambiguous pass,0.333333
gate leak (0 dmg),0
damaging leak,1
damage in band,0.607143


Predicted Label,in_domain,out_of_domain
Human Label,,
ambiguous,1,2
in_domain,12,5
out_of_domain,1,7


,Target,Guess,Human Label,Domain Match,Predicted Label,Cosine Similarity,Relevance r,Hit %,Predicted Tier,Expected tier,In band
0,brand,trademark,in_domain,primary,in_domain,0.636,1.00,45.0,CRITICAL STRIKE,strong,True
1,brand,logo,in_domain,primary,in_domain,0.548,1.00,45.0,CRITICAL STRIKE,strong,True
2,brand,customer loyalty,in_domain,primary,in_domain,0.428,0.58,25.7,SOLID HIT,medium,True
3,brand,packaging,in_domain,primary,in_domain,0.398,0.46,20.8,SOLID HIT,medium,True
4,brand,fashion label,ambiguous,out_of_domain,out_of_domain,0.590,1.00,0.0,BLOCKED — OUT OF DOMAIN,weak,False
5,brand,cowboy,out_of_domain,out_of_domain,out_of_domain,0.223,0.00,0.0,BLOCKED — OUT OF DOMAIN,blocked,True
6,brand,jellyfish,out_of_domain,out_of_domain,out_of_domain,0.313,0.14,0.0,BLOCKED — OUT OF DOMAIN,blocked,True
7,email,inbox,in_domain,out_of_domain,out_of_domain,0.667,1.00,0.0,BLOCKED — OUT OF DOMAIN,strong,False
8,email,spam filter,in_domain,out_of_domain,out_of_domain,0.332,0.79,0.0,BLOCKED — OUT OF DOMAIN,strong,False
9,email,mail server,in_domain,out_of_domain,out_of_domain,0.578,1.00,0.0,BLOCKED — OUT OF DOMAIN,strong,False


Ceiling terms per holdout target:
  brand       noise 0.276 | ceiling 0.540 | span 0.264 | corporation
  cartoon     noise 0.356 | ceiling 0.506 | span 0.150 | sketch
  comet       noise 0.271 | ceiling 0.421 | span 0.150 | hydrogen
  email       noise 0.214 | ceiling 0.364 | span 0.150 | program
Success criteria:


,criterion,value,pass
0,Holdout gate macro F1 >= 0.75,0.470186,False
1,Holdout damaging leak = 0,1,False
2,Holdout in-domain -> primary >= 80%,0.705882,False
3,Holdout damage in-band >= 70%,0.607143,False
4,Pacing goals met (Section 6 battles),True,True
5,Pacing goals met (LOTO-CV),True,True
6,No tuned parameter on a grid edge,"[damage.ceiling_topk, damage.relevance_min]",False


## 8. Play
Builds the screened target bank, then runs one simulated battle. Set `PLAY_INTERACTIVE = True` to play with `input()` (`hint`, `quit`). Random mode draws a topic and a screened target from the target bank.

In [18]:
# 8. Play: target bank, simulated battle (default) or interactive game
# What this cell does:
#   - Screens the target bank with the active model (drops weak targets).
#   - Simulated battle by default; interactive input() game when PLAY_INTERACTIVE = True.
RANDOM_ROUNDS = 4
RANDOM_BOSS_HP = 200
MIN_SPAN_RATIO = 0.6      # raw span must be >= this x span floor
GOOD_R = 0.6
MIN_GOOD_ANSWERS = 4
FALLBACK_TARGETS = 5      # used only if a topic has no eligible target

def extra_targets() -> dict:
    """Calibration and holdout targets per topic (playable after Section 7)."""
    out = {d: [] for d in DOMAINS}
    for t, d in CALIBRATION_TARGETS.items():
        out[d].append(t)
    for t, _, d, _, _ in HOLDOUT_CASES_DRAFT:
        if t not in out[d]:
            out[d].append(t)
    return out

def render_turn_result(engine: LexicalCombatEngine, result: dict, remaining_hp: int, required_domain: str) -> None:
    """Step-by-step feedback: gate, relevance, damage."""
    if not result["valid"]:
        print(f"  [!] {result['reason']}\n")
        return
    d = required_domain
    print(f"  Guess: '{result['word']}'")
    print(f"  Step 1 · Domain gate ({d}): raw {result['domain_raw_scores'][d]:.3f} − background "
          f"{result['domain_background'][d]:.3f} = {result['domain_scores'][d]:+.3f} | "
          f"lead vs best other {result['domain_lead']:+.3f} → {result['domain_match_type'].upper()}"
          + (f" ×{result['multiplier']:.2f}" if result["domain_allowed"] else ""))
    print("          Domain share (display): " + " | ".join(
        f"{x}: {result['domain_probs'][x]:.0%}" for x in engine.data.domains))
    print(f"  Step 2 · Target relevance: cos {result['similarity']:.3f} | noise floor "
          f"{result['noise_floor']:.3f} | ceiling {result['ceiling']:.3f} → r = {result['relevance']:.2f}")
    if result["domain_allowed"]:
        parts = [f"base {result['base_percent']:.1f}%", f"domain ×{result['multiplier']:.2f}"]
        if result["is_generic"]:
            parts.append(f"generic ×{result['generic_multiplier']:.2f}")
        if result.get("graze_floor") and result["base_percent"] <= 0:
            parts.append(f"in-domain graze floor ({engine.config.chip_percent}%)")
        if result["is_echo"]:
            parts.append(f"echo ×{result['echo_multiplier']:.2f} (cos {result['echo_similarity']:.2f} >= "
                         f"{engine.echo_threshold():.2f} to an earlier guess)")
        actual = result.get("actual_damage", result["damage"])
        overkill = f" (overkill {result['overkill']})" if result.get("overkill") else ""
        print(f"  Step 3 · Damage: {' · '.join(parts)} → {result['hit_percent']:.1f}% "
              f"= -{actual} HP{overkill} | {result['tier']}")
    else:
        print(f"  Step 3 · Damage: blocked by the domain gate → 0 HP | {result['tier']}")
    print(f"  Latency: {result['latency_ms']:.1f} ms | Enemy HP left: {max(remaining_hp, 0)}\n")

def play_stage(engine: LexicalCombatEngine, stage: dict):
    """One interactive stage (uses input()). Returns a record, or None on 'quit'."""
    hp, used_guesses, turns = stage["hp"], set(), 0
    print("=" * 72)
    print(f"STAGE {stage['stage']}: {stage['enemy']} | HP: {hp}")
    print(f"Target: '{stage['target'].upper()}' | Domain: {stage['domain'].upper()}")
    print("Enter a word/phrase (1–3 words), 'hint', or 'quit'.")
    print("=" * 72)
    while hp > 0:
        guess = input(f"[Stage {stage['stage']} | HP {hp}] Your guess: ").strip()
        if clean_text(guess) == "quit":
            return None
        if clean_text(guess) == "hint":
            print(f"  [Hint] Try something like: {engine.hint(stage['target'], stage['domain'], used_guesses)}\n")
            continue
        result = engine.evaluate_attack(guess, stage["target"], stage["domain"], used_guesses, stage["hp"])
        if result["valid"]:
            result = engine.resolve_damage(result, stage["hp"], hp)
            turns += 1
            used_guesses.add(result["word"])
            hp -= result["actual_damage"]
        render_turn_result(engine, result, hp, stage["domain"])
    print(f">>> {stage['enemy']} defeated in {turns} valid attack turns!\n")
    return {"Stage": stage["stage"], "Enemy": stage["enemy"], "Domain": stage["domain"],
            "Turns": turns, "Remaining HP": max(hp, 0)}

def pool_gate_kinds(engine: LexicalCombatEngine, domain: str) -> dict:
    """Gate kind of every pool term when played in its own domain."""
    pool, vectors = engine.pools[domain]
    adjusted, _ = engine.domain_scores(vectors, engine.config.anchor_topk)
    col = DOMAINS.index(domain)
    required = adjusted[:, col]
    lead = required - np.delete(adjusted, col, axis=1).max(axis=1)
    return dict(zip(pool, engine.gate_kinds(required, lead)))

def build_target_bank(engine: LexicalCombatEngine) -> pd.DataFrame:
    """Screen candidate targets per topic, with the reason for each decision."""
    rows, extras = [], extra_targets()
    for domain in DOMAINS:
        kinds = pool_gate_kinds(engine, domain)
        candidates = list(dict.fromkeys(extras[domain] + engine.pools[domain][0]))
        vectors = engine.encode(candidates)
        for target, vector in zip(candidates, vectors):
            adjusted, _, _, _ = engine.domain_scores_from_vector(vector)
            _, _, self_kind = engine.gate(adjusted, domain)
            cal = engine.target_calibration(target, domain)
            noise, ceiling_raw = engine.raw_bounds(cal)
            noise, ceiling = engine.bounds(cal)
            r = normalized_relevance(cal["pool_sims"], noise, ceiling)
            good = sum(1 for term, rel in zip(cal["pool_terms"], r)
                       if rel >= GOOD_R and kinds.get(term, "out_of_domain") != "out_of_domain")
            span = ceiling_raw - noise
            if len(target.split()) > 2 or engine.is_generic(target):
                reason = "generic / too long"
            elif self_kind != "primary":
                reason = f"target itself is {self_kind}"
            elif span < MIN_SPAN_RATIO * engine.span_floor:
                reason = "span too narrow"
            elif good < MIN_GOOD_ANSWERS:
                reason = f"only {good} good answers"
            else:
                reason = "ok"
            rows.append({"domain": domain, "target": target, "self gate": self_kind,
                         "span": round(span, 3), "good answers": good, "eligible": reason == "ok",
                         "reason": reason})
    return pd.DataFrame(rows)

def random_stages(bank: pd.DataFrame, rounds: int, seed: int) -> list:
    """Random topic order (all 4 before repeating), random eligible target per topic."""
    rng = random.Random(seed)
    eligible = {}
    for d in DOMAINS:
        rows = bank[bank["domain"] == d]
        ok = rows.loc[rows["eligible"], "target"].tolist()
        if not ok:
            ranked = rows.assign(primary=rows["self gate"] == "primary").sort_values(
                ["primary", "good answers", "span"], ascending=False)
            ok = ranked["target"].head(FALLBACK_TARGETS).tolist()
            print(f"WARNING: no {d} target passed every check; using the top {len(ok)} ranked candidates.")
        eligible[d] = ok
    topics = [d for d in DOMAINS if eligible[d]]
    stages, order, used = [], [], set()
    for number in range(1, rounds + 1):
        if not order:
            order = rng.sample(topics, len(topics))
        domain = order.pop()
        choices = [t for t in eligible[domain] if t not in used] or eligible[domain]
        target = rng.choice(choices)
        used.add(target)
        stages.append({"stage": number, "enemy": rng.choice(BOSS_NAMES[domain]), "target": target,
                       "domain": domain, "hp": RANDOM_BOSS_HP})
    return stages

def run_game(engine: LexicalCombatEngine, stages: list) -> list:
    """Interactive loop over stages; 'quit' ends the game."""
    records = []
    for stage in stages:
        print(f"\nTopic: {stage['domain'].upper()} | Boss: {stage['enemy']}")
        record = play_stage(engine, stage)
        if record is None:
            print("Game ended.")
            break
        records.append(record)
    if records:
        print("Summary:")
        display(pd.DataFrame(records))
    return records

def simulated_battle(engine: LexicalCombatEngine, bank: pd.DataFrame, seed: int) -> pd.DataFrame:
    """One random stage played by an auto-player (no input())."""
    stage = random_stages(bank, 1, seed)[0]
    cal = engine.target_calibration(stage["target"], stage["domain"])
    noise, ceiling = engine.bounds(cal)
    r = normalized_relevance(cal["pool_sims"], noise, ceiling)
    mid = [t for _, t in sorted(zip(np.abs(r - 0.6), cal["pool_terms"]))[:6]]
    guesses = ["banana", "spreadsheet"] + mid
    print(f"Simulated: {stage['enemy']} | target '{stage['target']}' ({stage['domain']}) | guesses {guesses}")
    battle = engine.simulate_battle(stage["target"], stage["domain"], stage["hp"], guesses,
                                    on_turn=lambda res, hp, dom: render_turn_result(engine, res, hp, dom))
    display(battle)
    return battle

with section("8. Play"):
    TARGET_BANK = build_target_bank(engine)
    print("Target bank (eligible / screened) per topic:")
    display(TARGET_BANK.groupby("domain")["eligible"].agg(["sum", "size"])
            .rename(columns={"sum": "eligible", "size": "screened"}))
    print("Rejected examples:")
    display(TARGET_BANK[~TARGET_BANK["eligible"]].groupby("reason").head(2)[["domain", "target", "reason"]])
    if not PLAY_INTERACTIVE:
        print("PLAY_INTERACTIVE = False — one simulated battle (set it to True to play).")
        simulated_battle(engine, TARGET_BANK, SEED)
    elif PLAY_MODE == "classic":
        run_game(engine, GAME_STAGES)
    else:
        print("\nRandom-topic game: 1–3 English words per guess | 'hint' | 'quit'")
        run_game(engine, random_stages(TARGET_BANK, RANDOM_ROUNDS, SEED))

Target bank (eligible / screened) per topic:


,eligible,screened
domain,,
Arts,33,53
Business,43,52
Science,36,50
Technology,29,51


Rejected examples:


,domain,target,reason
4,Business,manager,only 3 good answers
6,Business,profit,only 3 good answers
13,Business,chief executive officer,generic / too long
18,Business,board of directors,generic / too long
27,Business,budget,only 1 good answers
43,Business,customer,only 2 good answers
44,Business,client,target itself is out_of_domain
50,Business,performance review,target itself is out_of_domain
62,Technology,data structure,only 1 good answers
69,Technology,searching,only 2 good answers


PLAY_INTERACTIVE = False — one simulated battle (set it to True to play).
Simulated: Alchemist Golem | target 'hydrogen' (Science) | guesses ['banana', 'spreadsheet', 'element', 'oxidation', 'photosynthesis', 'organic chemistry', 'chemical compound', 'electron']
  Guess: 'banana'
  Step 1 · Domain gate (Science): raw 0.044 − background 0.070 = -0.026 | lead vs best other -0.043 → OUT_OF_DOMAIN
          Domain share (display): Business: 40% | Technology: 25% | Science: 17% | Arts: 17%
  Step 2 · Target relevance: cos 0.253 | noise floor 0.237 | ceiling 0.553 → r = 0.05
  Step 3 · Damage: blocked by the domain gate → 0 HP | BLOCKED — OUT OF DOMAIN
  Latency: 19.6 ms | Enemy HP left: 200

  Guess: 'spreadsheet'
  Step 1 · Domain gate (Science): raw 0.175 − background 0.070 = +0.105 | lead vs best other -0.040 → OUT_OF_DOMAIN
          Domain share (display): Business: 42% | Technology: 38% | Science: 19% | Arts: 2%
  Step 2 · Target relevance: cos 0.091 | noise floor 0.237 | ceiling 0.55

,Guess,Cosine,Relevance r,Gate,Generic,Echo,Hit %,Tier,Damage,HP before,HP left
0,banana,0.253,0.05,out_of_domain,False,False,0.0,BLOCKED — OUT OF DOMAIN,0,200,200
1,spreadsheet,0.091,0.00,out_of_domain,False,False,0.0,BLOCKED — OUT OF DOMAIN,0,200,200
2,element,0.423,0.59,primary,False,False,26.2,STRONG HIT,52,200,148
3,oxidation,0.415,0.56,primary,False,False,25.1,SOLID HIT,50,148,98
4,photosynthesis,0.413,0.56,primary,False,False,24.8,SOLID HIT,50,98,48
5,organic chemistry,0.409,0.55,primary,False,False,24.4,SOLID HIT,48,48,0


## Appendix. Fine-tuning (optional)
Runs only with `RUN_FINE_TUNING = True`. Fine-tunes the Sentence Transformer on WordNet synonym/definition pairs + curated pairs with MultipleNegativesRankingLoss (MNRL) and mined hard negatives. The checkpoint goes to the models folder; Section 4 picks it up on the next Run all with `MODEL_CHOICE = "fine_tuned"`.

In [19]:
# Appendix. Fine-tuning (guarded by RUN_FINE_TUNING; the new checkpoint is used on the next Run all
# What this cell does:
#   - Builds training pairs from WordNet + curated pairs (calibration / holdout excluded).
#   - Mines hard negatives with the base model.
#   - Fine-tunes with MNRL and saves the checkpoint (only when RUN_FINE_TUNING = True).
# with MODEL_CHOICE = "fine_tuned"). WordNet synonym/definition pairs + curated Science/Technology pairs,
# MNRL with mined hard negatives. Same data rules and hyper-parameters as round 3.
FT_SEEDS = {
    "Business": ["business", "company", "commerce", "finance", "market", "revenue",
                 "management", "executive", "director", "manager", "accounting",
                 "financial reporting", "quarterly report", "operations", "strategy"],
    "Technology": ["computer", "software", "algorithm", "database", "network",
                   "programming", "hardware", "motherboard", "cybersecurity",
                   "encryption", "firewall", "data structure", "machine learning",
                   "neural network", "source code", "logic", "compiler", "processor",
                   "server", "internet", "program", "circuit", "memory", "code"],
    "Science": ["science", "physics", "chemistry", "biology", "molecule", "catalyst",
                "experiment", "genetics", "atom", "ecosystem", "energy", "force", "gravity",
                "cell", "gene", "element", "compound", "planet", "laboratory", "hypothesis"],
    "Arts": ["art", "painting", "music", "literature", "sculpture", "theater",
             "dance", "film", "poetry", "design"],
}
FT_CURATED_PAIRS = {
    "Science": [
        ("atom", "basic unit of matter"), ("dna", "genetic material"), ("gene", "unit of heredity"),
        ("photosynthesis", "plants converting sunlight"), ("evolution", "natural selection"),
        ("velocity", "speed and direction"), ("acceleration", "change in velocity"),
        ("hypothesis", "testable prediction"), ("microscope", "magnifying instrument"),
        ("oxidation", "loss of electrons"), ("reduction reaction", "gain of electrons"),
        ("acid", "low ph solution"), ("chemical bond", "attraction between atoms"),
        ("periodic table", "table of elements"), ("ecology", "study of ecosystems"),
        ("geology", "study of rocks"), ("astronomy", "study of stars"),
        ("planet", "orbiting celestial body"), ("cell membrane", "cell boundary"),
        ("organism", "living thing"), ("kinetic energy", "energy of motion"),
        ("potential energy", "stored energy"), ("density", "mass per volume"),
        ("experiment", "scientific test"), ("laboratory", "science lab"),
        ("specimen", "lab sample"), ("ion", "charged particle"), ("compound", "combined elements"),
        ("climate", "long term weather"), ("electricity", "flow of charge"),
    ],
    "Technology": [
        ("software", "computer program"), ("programming", "writing code"),
        ("compiler", "code translator"), ("cpu", "central processing unit"),
        ("ram", "random access memory"), ("operating system", "system software"),
        ("router", "network device"), ("firewall", "network security barrier"),
        ("debugging", "fixing code errors"), ("recursion", "function calling itself"),
        ("array", "ordered list of elements"), ("hash table", "key value store"),
        ("cloud computing", "remote servers"), ("api", "application programming interface"),
        ("server", "host computer"), ("binary", "base two numbers"),
        ("transistor", "electronic switch"), ("microcontroller", "small embedded computer"),
        ("deep learning", "layered neural networks"), ("computer vision", "image recognition"),
        ("natural language processing", "text understanding"), ("bug", "software defect"),
        ("variable", "named storage location"), ("loop", "repeated instructions"),
        ("ip address", "network address"), ("malware", "malicious software"),
        ("backend", "server side code"), ("frontend", "user interface code"),
        ("gpu", "graphics processor"), ("source file", "code file"),
    ],
}

@dataclass(frozen=True)
class FineTuneConfig:
    """Fine-tuning data rules and hyper-parameters."""
    min_zipf: float = 2.5                 # every word must be reasonably common English
    strict_domain_consistency: bool = True  # both sides must map to the pair's domain (base model)
    min_pairs_after_filter: int = 120     # below this, fall back to the one-side rule
    max_pairs_per_synset: int = 2
    max_pairs_per_phrase: int = 3
    max_pairs_per_domain: int = 60
    bad_pairs: tuple = (("medicine", "music"),)   # wrong WordNet sense ("face the music")
    hard_negative_max_cos: float = 0.35
    max_negative_zipf: float = 5.0
    generic_negatives: tuple = ("design", "business", "process", "system", "method", "work", "activity",
                                "art", "science", "technology", "music", "company", "industry", "product",
                                "service", "practice", "study", "field", "theory", "model", "structure",
                                "group", "act", "thing", "matter", "form")
    batch_size: int = 32
    epochs: int = 1
    learning_rate: float = 1e-5

FT_CONFIG = FineTuneConfig()

def ft_excluded_phrases() -> set:
    """Every test, calibration and holdout phrase is kept out of training."""
    phrases = {clean_text(p) for case in TEST_CASES for p in case[:2]}
    return phrases | EVAL_PHRASES

def is_short_phrase(text: str, min_words: int = 1, max_words: int = 5) -> bool:
    """1–5 word phrase."""
    return min_words <= len(text.split()) <= max_words if text else False

def wordnet_pairs(excluded: set) -> list:
    """Synonym and short-definition pairs for each domain's seed nouns (empty if WordNet is missing)."""
    import nltk
    try:
        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)
        from nltk.corpus import wordnet as wn
        wn.synsets("test")
    except LookupError as error:
        reason = " ".join(str(error).split())[:100]
        print(f"WARNING: WordNet unavailable ({reason}); using curated pairs only.")
        return []
    seeds = {d: [s for s in ss if clean_text(s) not in HOLDOUT_PHRASES] for d, ss in FT_SEEDS.items()}
    pairs = []
    for domain, seed_list in seeds.items():
        synsets = sorted({s for seed in seed_list for s in wn.synsets(seed, pos=wn.NOUN)}, key=lambda s: s.name())
        for synset in synsets:
            lemmas = sorted({clean_text(l.name().replace("_", " ")) for l in synset.lemmas()})
            lemmas = [p for p in lemmas if is_short_phrase(p)]
            for left, right in itertools.combinations(lemmas, 2):
                if left != right and left not in excluded and right not in excluded:
                    pairs.append({"anchor": left, "positive": right, "domain": domain,
                                  "source": "wordnet_synonyms", "synset": synset.name()})
            definition = clean_text(synset.definition())
            if is_short_phrase(definition) and definition not in excluded:
                for lemma in lemmas:
                    if lemma != definition and lemma not in excluded:
                        pairs.append({"anchor": lemma, "positive": definition, "domain": domain,
                                      "source": "wordnet_definition", "synset": synset.name()})
    return pairs

def curated_pairs(excluded: set) -> list:
    """Hand-written Science / Technology pairs that pass the same filters."""
    out = []
    for domain, pairs in FT_CURATED_PAIRS.items():
        for left, right in pairs:
            left, right = clean_text(left), clean_text(right)
            if (left != right and is_short_phrase(left) and is_short_phrase(right)
                    and left not in excluded and right not in excluded):
                out.append({"anchor": left, "positive": right, "domain": domain,
                            "source": "curated", "synset": f"curated:{left}"})
    return out

def base_top_domains(base: LexicalCombatEngine, texts: list) -> np.ndarray:
    """Top domain per text under the base model (mean of the 2 best anchors, raw cosine)."""
    _, raw = base.domain_scores(base.encode(texts), 2)
    return np.array(DOMAINS)[raw.argmax(axis=1)]

def build_training_pairs(base: LexicalCombatEngine, cfg: FineTuneConfig, seed: int) -> tuple:
    """Filtered, de-duplicated, balanced pairs split into train / validation."""
    from wordfreq import zipf_frequency
    excluded = ft_excluded_phrases()
    pairs = wordnet_pairs(excluded) + curated_pairs(excluded)
    funnel = {"raw pairs": len(pairs)}
    pairs = [p for p in pairs if all(zipf_frequency(w, "en") >= cfg.min_zipf
                                     for w in (p["anchor"] + " " + p["positive"]).split())]
    funnel[f"both sides zipf >= {cfg.min_zipf}"] = len(pairs)
    if pairs:
        top_a = base_top_domains(base, [p["anchor"] for p in pairs])
        top_p = base_top_domains(base, [p["positive"] for p in pairs])
        strict = [p for p, a, b in zip(pairs, top_a, top_p) if p["domain"] == a == b]
        lenient = [p for p, a, b in zip(pairs, top_a, top_p) if p["domain"] in (a, b)]
        if cfg.strict_domain_consistency and len(strict) >= cfg.min_pairs_after_filter:
            pairs = strict
        elif len(lenient) >= cfg.min_pairs_after_filter:
            print(f"WARNING: strict domain filter left {len(strict)} pairs; using the one-side rule.")
            pairs = lenient
        else:
            print(f"WARNING: domain-consistency filter skipped: only {len(lenient)} pairs would remain.")
    funnel["base model agrees on domain"] = len(pairs)
    dedup = {(p["domain"], *sorted([p["anchor"], p["positive"]])): p for p in pairs}
    frame = pd.DataFrame(dedup.values())
    if frame.empty:
        return None, None, funnel
    bad = {frozenset(b) for b in cfg.bad_pairs}
    frame = frame[[frozenset((a, p)) not in bad for a, p in zip(frame["anchor"], frame["positive"])]]
    frame = frame.sample(frac=1, random_state=seed)
    frame = frame.groupby("synset", group_keys=False).head(cfg.max_pairs_per_synset)
    frame = frame.groupby("anchor", group_keys=False).head(cfg.max_pairs_per_phrase)
    frame = frame.assign(_curated=(frame["source"] == "curated")).sort_values("_curated", ascending=False)
    frame = frame.groupby("domain", group_keys=False).head(cfg.max_pairs_per_domain).drop(columns="_curated")
    funnel["after caps / blocklist"] = len(frame)
    train_parts, val_parts = [], []
    for _, group in frame.groupby("domain"):
        group = group.sample(frac=1, random_state=seed).reset_index(drop=True)
        n_val = min(max(1, int(round(len(group) * 0.15))), len(group) - 1) if len(group) >= 2 else 0
        val_parts.append(group.iloc[:n_val])
        train_parts.append(group.iloc[n_val:])
    train = pd.concat(train_parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    val = pd.concat(val_parts, ignore_index=True)
    texts = set(train["anchor"]) | set(train["positive"]) | set(val["anchor"]) | set(val["positive"])
    assert texts.isdisjoint(excluded), "evaluation phrases leaked into the training data"
    return train, val, funnel

def mine_hard_negatives(base: LexicalCombatEngine, train: pd.DataFrame, cfg: FineTuneConfig) -> pd.DataFrame:
    """Per pair: the most similar specific phrase of another domain with base cosine below the cap."""
    from wordfreq import zipf_frequency
    excluded = ft_excluded_phrases()
    pools = {d: set() for d in DOMAINS}
    for row in train.itertuples(index=False):
        pools[row.domain].update([row.anchor, row.positive])
    for domain, phrases in DOMAIN_EXAMPLES.items():
        pools[domain].update(clean_text(p) for p in phrases)
    counts = {}
    for phrases in pools.values():
        for p in phrases:
            counts[p] = counts.get(p, 0) + 1
    def usable(p):
        return (p and p not in excluded and counts[p] == 1 and p not in cfg.generic_negatives
                and all(zipf_frequency(w, "en") <= cfg.max_negative_zipf for w in p.split()))
    pools = {d: sorted(p for p in ps if usable(p)) for d, ps in pools.items()}
    phrases = sorted({p for ps in pools.values() for p in ps} | set(train["anchor"]))
    vectors = dict(zip(phrases, base.encode(phrases)))
    rows = []
    for row in train.itertuples(index=False):
        candidates = [p for d, ps in pools.items() if d != row.domain for p in ps]
        sims = np.array([vectors[c] @ vectors[row.anchor] for c in candidates])
        allowed = np.where(sims < cfg.hard_negative_max_cos)[0]
        pick = int(allowed[np.argmax(sims[allowed])]) if len(allowed) else int(np.argmin(sims))
        rows.append({"anchor": row.anchor, "positive": row.positive, "negative": candidates[pick],
                     "base_cos": round(float(sims[pick]), 3)})
    return pd.DataFrame(rows)

def run_fine_tuning(cfg: FineTuneConfig = FT_CONFIG, seed: int = SEED) -> Path:
    """Build data, train with MNRL + hard negatives, save to PATHS.models. Returns the checkpoint path."""
    import torch
    from torch.utils.data import DataLoader
    from sentence_transformers import InputExample, losses
    from sentence_transformers.evaluation import InformationRetrievalEvaluator
    torch.manual_seed(seed)
    base_model_obj, _, _ = load_model("base")
    base = LexicalCombatEngine(base_model_obj, ENGINE_DATA, EngineConfig())
    train, val, funnel = build_training_pairs(base, cfg, seed)
    display(pd.Series(funnel, name="pairs").to_frame())
    if train is None or val is None or val.empty:
        print("WARNING: not enough training pairs; fine-tuning skipped.")
        return None
    display(train.groupby(["domain", "source"]).size().unstack(fill_value=0))
    negatives = mine_hard_negatives(base, train, cfg)
    print(f"Hard negatives: max base_cos {negatives['base_cos'].max():.3f} | mean {negatives['base_cos'].mean():.3f}")
    examples = [InputExample(texts=[r.anchor, r.positive, r.negative]) for r in negatives.itertuples(index=False)]
    loader = DataLoader(examples, shuffle=True, batch_size=cfg.batch_size,
                        drop_last=len(examples) >= cfg.batch_size)
    student = SentenceTransformer(BASE_MODEL_NAME)
    evaluator = InformationRetrievalEvaluator(
        queries={f"q{i}": a for i, a in enumerate(val["anchor"])},
        corpus={f"d{i}": p for i, p in enumerate(val["positive"])},
        relevant_docs={f"q{i}": {f"d{i}"} for i in range(len(val))},
        name="lexical-domain-validation")
    output = PATHS.models / FINE_TUNED_DIRNAME
    if not PERSISTENT:
        print(f"WARNING: PERSISTENT = False — the checkpoint at {output} will not survive this session.")
    steps = len(loader) * cfg.epochs
    student.fit(train_objectives=[(loader, losses.MultipleNegativesRankingLoss(model=student))],
                evaluator=evaluator, epochs=cfg.epochs, warmup_steps=max(1, int(steps * 0.10)),
                output_path=str(output), evaluation_steps=0, save_best_model=True,
                show_progress_bar=True, optimizer_params={"lr": cfg.learning_rate})
    print(f"Checkpoint saved: {output}. Set MODEL_CHOICE = 'fine_tuned' and Run all to use it.")
    return output

with section("Appendix. Fine-tuning") as _s:
    if not RUN_FINE_TUNING:
        _s.skip("RUN_FINE_TUNING = False")
    else:
        FINE_TUNED_CHECKPOINT = run_fine_tuning()
        if FINE_TUNED_CHECKPOINT is None:
            _s["reason"] = "no usable training data"

Skipped: RUN_FINE_TUNING = False


## History (removed cells)

- **Cells 3–6 (original prototype):** first engine — raw-cosine thresholds, max-over-anchors domain score with 8/10/4/3 anchors per domain, fixed damage table, interactive loop. Replaced by the noise-normalised engine (Section 3); the 3 stages live on in `GAME_STAGES`.
- **Cell C / D (fine-tune before/after, plug-in demo):** compared Base vs fine-tuned on the 19 original tests with the old pipeline. Replaced by the Base vs Fine-tuned table in Section 6, which recalibrates each model.
- **Cells E, F (threshold grid + balance check):** tuned raw-cosine gate/damage thresholds on 19 cases (every threshold tied) and added an overkill cap. Superseded by the non-circular gate set, plateau selection and CV in Section 5.
- **Cells R1–R5 (rebalance round 2):** stricter convex curve and a generic-term list that wrongly penalised good answers (*director*, *molecule*); made Stages 2–3 unwinnable. Fixed in Section 3 (`GENERIC_TERMS_V2`) and Section 5 (pacing in the loss).
- **Cell G (contrastive negatives):** optional ContrastiveLoss run against score inflation; superseded by MNRL with mined hard negatives (Appendix). STS-B loading was dropped too: it produced 0 usable short pairs and was already disabled.

## 9. Regression check and run summary
Compares this run with the embedded results of the pre-refactor notebook (only when the active model matches that run), then prints the time and status of every section.

In [20]:
# 9. Regression check against the pre-refactor run + run summary (last cell)
# What this cell does:
#   - Compares this run with the embedded pre-refactor results (same model only).
#   - Prints the run summary: status and time of every section.
# Baseline: printed tables of the pre-refactor notebook on Colab (base model). It applies only when
# the active encoder matches that run (fingerprint below). LEXICAL_BASELINE_JSON (env var) can point
# to a full-precision baseline file instead; this is how the offline test harness checks to 1e-6.
EMBEDDED_BASELINE = json.loads(r'''{"source":"Colab run of the pre-refactor notebook (base model, 2026-11-17); values at display precision","encoder_fingerprint":{"anisotropy":0.291,"background_k2":{"Business":0.061,"Technology":0.086,"Science":0.07,"Arts":0.083}},"params":{"GATE_PRIMARY_MIN":0.04,"GATE_SECONDARY_MIN":0.14,"GATE_SECONDARY_MAX_GAP":0.08,"GATE_WEAK_MAX_GAP":0.2,"ANCHOR_TOPK":2,"NOISE_PERCENTILE":65,"CEILING_TOPK":1,"RELEVANCE_MIN":0.0,"DAMAGE_GAMMA":1.2,"MIN_SPAN_MODE":"fixed","SPAN_FLOOR":0.15},"banded":{"Target":["executive","executive","executive","executive","executive","executive","algorithm","algorithm","algorithm","algorithm","algorithm","algorithm","catalyst","catalyst","catalyst","catalyst","painting","painting","painting","executive","executive","executive","executive","executive","executive","executive","executive","algorithm","algorithm","algorithm","algorithm","algorithm","algorithm","algorithm","catalyst","catalyst","catalyst","catalyst","catalyst","catalyst","catalyst","catalyst","painting","painting","painting","painting","painting","painting","painting","painting"],"Guess":["ceo","director","business logic","logic","logic gate","banana","data structure","source code","logic gate","business logic","logic","abstract painting","chemical reaction","molecule","market strategy","quantum mechanics","artwork","sculpture","database","senior management","company boss","board member","senior official","office worker","sales team","sunflower","symphony","computer instructions","problem solving","graph theory","calculator","computer screen","dinosaur","ballet","reaction speed","chemical agent","platinum","chemistry lab","thermodynamics","electron","teacup","stock price","brush stroke","picture","art gallery","photograph","poem","dance","tractor","algebra"],"Domain Match":["primary","primary","primary","out_of_domain","out_of_domain","out_of_domain","primary","primary","primary","weak_secondary","primary","out_of_domain","primary","primary","out_of_domain","primary","primary","primary","out_of_domain","primary","primary","primary","primary","primary","primary","out_of_domain","out_of_domain","primary","out_of_domain","primary","primary","primary","out_of_domain","out_of_domain","primary","primary","out_of_domain","primary","primary","primary","out_of_domain","out_of_domain","primary","primary","primary","primary","primary","primary","out_of_domain","primary"],"Relevance r":[1.0,0.53,0.11,0.07,0.01,0.0,0.42,0.33,0.35,0.04,0.26,0.17,0.93,0.43,0.16,0.0,0.78,0.45,0.0,0.88,0.7,0.3,0.73,0.47,0.3,0.0,0.0,0.17,0.5,0.56,0.35,0.0,0.02,0.06,0.47,0.71,0.54,0.81,0.24,0.35,0.0,0.0,0.28,0.23,0.56,0.43,0.18,0.17,0.0,0.07],"Hit %":[45.0,23.8,8.0,0.0,0.0,0.0,19.0,15.5,16.2,4.0,12.9,0.0,41.7,19.5,0.0,0.0,34.7,20.2,0.0,39.4,31.1,14.3,32.2,21.2,14.6,0.0,0.0,9.8,0.0,25.0,16.2,0.0,0.0,0.0,21.1,31.4,0.0,36.3,12.3,16.4,0.0,0.0,13.7,11.9,24.8,19.6,10.1,9.7,0.0,6.7],"Cosine Similarity":[0.726,0.492,0.281,0.26,0.231,0.218,0.374,0.334,0.343,0.204,0.303,0.262,0.457,0.322,0.249,0.153,0.72,0.529,0.145,0.667,0.576,0.374,0.589,0.46,0.377,0.175,0.203,0.264,0.412,0.439,0.343,0.117,0.194,0.214,0.333,0.397,0.351,0.426,0.272,0.301,0.12,0.096,0.433,0.405,0.591,0.52,0.374,0.367,0.201,0.313]},"pacing":{"Stage":[1,1,1,2,2,2,3,3,3],"Play":["expert","typical","playlog","expert","typical","playlog","expert","typical","playlog"],"Valid turns":[4,8,6,4,8,8,3,8,5],"Turns to defeat":[4,8,6,4,8,8,3,8,5],"HP left":[0,0,0,0,0,0,0,0,0]},"holdout":{"Target":["brand","brand","brand","brand","brand","brand","brand","email","email","email","email","email","email","email","comet","comet","comet","comet","comet","comet","comet","cartoon","cartoon","cartoon","cartoon","cartoon","cartoon","cartoon"],"Guess":["trademark","logo","customer loyalty","packaging","fashion label","cowboy","jellyfish","inbox","spam filter","mail server","chat app","newsletter","postcard","seashell","asteroid","orbit","solar system","light year","space shuttle","horoscope","doorknob","comic strip","caricature","animated film","puppet show","television channel","stock chart","toothpaste"],"Domain Match":["primary","primary","primary","primary","out_of_domain","out_of_domain","out_of_domain","out_of_domain","out_of_domain","out_of_domain","primary","out_of_domain","out_of_domain","out_of_domain","primary","primary","primary","out_of_domain","primary","primary","out_of_domain","primary","primary","primary","primary","out_of_domain","out_of_domain","out_of_domain"],"Relevance r":[1.0,1.0,0.58,0.46,1.0,0.0,0.14,1.0,0.79,1.0,0.62,1.0,1.0,0.0,1.0,0.96,0.48,0.0,0.53,0.0,0.0,1.0,0.6,1.0,0.85,0.4,0.0,0.0],"Hit %":[45.0,45.0,25.7,20.8,0.0,0.0,0.0,0.0,0.0,0.0,27.5,0.0,0.0,0.0,45.0,43.0,21.7,0.0,23.6,4.0,0.0,45.0,26.8,45.0,37.8,0.0,0.0,0.0],"Cosine Similarity":[0.636,0.548,0.428,0.398,0.59,0.223,0.313,0.667,0.332,0.578,0.306,0.52,0.416,0.208,0.618,0.415,0.343,0.264,0.35,0.257,0.162,0.548,0.446,0.65,0.483,0.416,0.114,0.234]},"holdout_summary":{"cases":28.0,"label status":"single_annotator","gate accuracy":0.678571,"gate macro F1":0.470186,"in-domain -> primary":0.705882,"ambiguous pass":0.333333,"gate leak (0 dmg)":0.0,"damaging leak":1.0,"damage in band":0.607143}}''')

def regression_objects(engine: LexicalCombatEngine) -> dict:
    """The objects the baseline records, recomputed from this run."""
    def table(frame, columns):
        return {c: [None if (isinstance(v, float) and v != v) else v for v in frame[c].tolist()] for c in columns}
    cols = ["Target", "Guess", "Domain Match", "Relevance r", "Hit %", "Cosine Similarity"]
    out = {"params": {**snapshot_params(engine), "SPAN_FLOOR": engine.span_floor},
           "banded": table(v2_eval, cols),
           "pacing": table(pacing_df, ["Stage", "Play", "Valid turns", "Turns to defeat", "HP left"])}
    if holdout_result is not None:
        out["holdout"] = table(holdout_result, cols)
        out["holdout_summary"] = dict(holdout_summary)
    return out

def _same(a, b, tol: float) -> bool:
    if a is None or b is None:
        return a is None and b is None
    if isinstance(a, str) or isinstance(b, str):
        return str(a) == str(b)
    return abs(float(a) - float(b)) <= tol

def compare_to_baseline(current: dict, baseline: dict, tol: float, summary_tol: float) -> list:
    """List of mismatches (empty = identical within tolerance)."""
    problems = []
    for key in ["params", "banded", "pacing", "holdout", "holdout_summary"]:
        if key not in baseline:
            continue
        if key not in current:
            problems.append(f"{key}: not computed in this run")
            continue
        base, cur = baseline[key], current[key]
        for name, expected in base.items():
            got = cur.get(name)
            if isinstance(expected, list):
                if got is None or len(got) != len(expected):
                    problems.append(f"{key}.{name}: length {None if got is None else len(got)} vs {len(expected)}")
                    continue
                bad = [i for i, (x, y) in enumerate(zip(got, expected)) if not _same(x, y, tol)]
                if bad:
                    problems.append(f"{key}.{name}: {len(bad)} rows differ (first row {bad[0]}: {got[bad[0]]} vs {expected[bad[0]]})")
            elif not _same(got, expected, summary_tol if key == "holdout_summary" else tol):
                problems.append(f"{key}.{name}: {got} vs {expected}")
    return problems

def run_regression_check(engine: LexicalCombatEngine) -> str:
    """'PASS', 'FAIL (...)' or 'skipped (reason)'."""
    override = os.environ.get("LEXICAL_BASELINE_JSON")
    if override:
        baseline, tol, summary_tol, label = json.loads(Path(override).read_text()), 1e-6, 1e-6, f"override {override}"
    else:
        if ACTIVE_MODEL != "base":
            return "skipped (baseline is for the base model)"
        fp = EMBEDDED_BASELINE["encoder_fingerprint"]
        same_encoder = (round(engine.noise_pair_mean, 3) == fp["anisotropy"] and
                        all(round(engine.background_by_k[2][d], 3) == v for d, v in fp["background_k2"].items()))
        if not same_encoder:
            return "skipped (encoder differs from the embedded baseline's model)"
        baseline, tol, summary_tol, label = EMBEDDED_BASELINE, 1e-9, 1e-6, "embedded Colab baseline"
    if holdout_result is None:
        baseline = {k: v for k, v in baseline.items() if not k.startswith("holdout")}
    problems = compare_to_baseline(regression_objects(engine), baseline, tol, summary_tol)
    if problems:
        for p in problems:
            print("  mismatch:", p)
        return f"FAIL ({len(problems)} mismatches vs {label})"
    return f"PASS (identical to {label})"

with section("9. Regression check"):
    REGRESSION_RESULT = run_regression_check(engine)
    print("Regression check:", REGRESSION_RESULT)

print("\nRun summary")
display(pd.DataFrame(RUN_LOG)[["section", "status", "reason", "seconds"]])
print(f"Total section time: {sum(e['seconds'] for e in RUN_LOG):.1f} s | model: {ACTIVE_MODEL} ({MODEL_SOURCE}) | "
      f"PERSISTENT: {PERSISTENT} | calibration: {CALIBRATION_SOURCE} | holdout labels: {HOLDOUT_LABEL_STATUS} | "
      f"regression: {REGRESSION_RESULT}")

Regression check: PASS (identical to embedded Colab baseline)

Run summary


,section,status,reason,seconds
0,0. Setup,ran,,121.29
1,1. Data registry,ran,,0.07
2,2. Text utilities and checks,ran,,0.26
3,3. Engine,ran,,0.01
4,4. Model selection,ran,,7.78
5,5.2 Gate diagnosis (old holdout),ran,,0.85
6,5.5 Calibration,ran,,4.90
7,5.6 Gate review,ran,,0.23
8,6. Report,ran,,11.58
9,7. Holdout,ran,,0.61


Total section time: 151.9 s | model: base (sentence-transformers/all-MiniLM-L6-v2) | PERSISTENT: False | calibration: grid search | holdout labels: single_annotator | regression: PASS (identical to embedded Colab baseline)


## Play Game
Run this cell (or Run all) and play in the box that appears: each game draws random topics and a random target per topic from the screened target bank. Type 1–3 English words and press **Enter** or **Attack**; **Hint** suggests a word, **Skip boss** moves on, **Quit** ends the game, **New game** draws new stages. Each turn is shown as a card with one detail per line; when a stage ends you get its summary table (every guess, damage, boss HP left), and when the game ends a game summary. Run all does not wait here, because the game is a widget, not `input()`.

In [21]:
# Play Game — random topic + random target; type your own guesses in the box below.
# What this cell does:
#   - GameSession: game state (stages, HP, guesses, hints, skips).
#   - HTML cards for each turn, stage summaries and the game summary.
#   - ipywidgets UI (Attack / Hint / Skip / Quit / New game) with a text-mode fallback.
# Uses widgets, so Run all never waits for input. Without ipywidgets: call play_in_text_mode(GAME).

TOPIC_COLORS = {"Business": "#3b82f6", "Technology": "#10b981", "Science": "#a855f7", "Arts": "#f43f5e"}
TIER_COLORS = {"CRITICAL STRIKE": "#ef4444", "STRONG HIT": "#f97316", "SOLID HIT": "#eab308",
               "WEAK GRAZE": "#22c55e", "NO EFFECT": "#71717a", "BLOCKED — OUT OF DOMAIN": "#52525b"}
STATUS_COLORS = {"Defeated": "#22c55e", "Skipped": "#eab308", "Quit": "#f43f5e", "In progress": "#3b82f6"}
GATE_SHORT = {"primary": "primary", "secondary": "secondary", "weak_secondary": "weak secondary",
              "out_of_domain": "blocked"}
GATE_TEXT = {"primary": "PRIMARY — in the topic", "secondary": "SECONDARY — related topic",
             "weak_secondary": "WEAK SECONDARY — loosely related", "out_of_domain": "BLOCKED — wrong topic"}
CARD = "background:#1e1e24;border:1px solid #3f3f46;border-radius:10px;color:#f4f4f5;font-family:sans-serif;"

def _badge(text: str, color: str) -> str:
    """Small coloured label."""
    return (f"<span style='background:{color};color:#fff;padding:2px 8px;border-radius:999px;"
            f"font-size:11px;font-weight:bold;white-space:nowrap'>{_html.escape(text)}</span>")

def _line(label: str, value: str, note: str = "") -> str:
    """One labelled detail line of a turn card."""
    note_html = f"<span style='color:#a1a1aa;font-size:12px'> {note}</span>" if note else ""
    return (f"<div style='display:flex;gap:10px;padding:3px 0;font-size:13px'>"
            f"<div style='width:150px;color:#a1a1aa;flex-shrink:0'>{label}</div>"
            f"<div style='color:#f4f4f5'>{value}{note_html}</div></div>")

def turn_card_html(engine: LexicalCombatEngine, result: dict, hp_before: int, hp_after: int,
                   domain: str, turn: int) -> str:
    """One guess, one detail per line."""
    if not result["valid"]:
        return (f"<div style='{CARD}padding:8px 14px;margin:6px 0;border-color:#a16207'>"
                f"⚠️ <b>Not counted:</b> {_html.escape(result['reason'])}</div>")
    tier = result["tier"]
    color = TIER_COLORS.get(tier, "#71717a")
    damage = result.get("actual_damage", result["damage"])
    share = " · ".join(f"{d} {result['domain_probs'][d]:.0%}" for d in engine.data.domains)
    rival = max((d for d in engine.data.domains if d != domain), key=lambda d: result["domain_scores"][d])
    multipliers = [f"base {result['base_percent']:.1f}%", f"topic ×{result['multiplier']:.2f}"]
    if result["is_generic"]:
        multipliers.append(f"too generic ×{result['generic_multiplier']:.2f}")
    if result["is_echo"]:
        multipliers.append(f"repeat idea ×{result['echo_multiplier']:.2f}")
    if result.get("graze_floor") and result["base_percent"] <= 0:
        multipliers.append(f"in-topic graze floor {engine.config.chip_percent}%")
    damage_line = (" · ".join(multipliers) + f" → <b>{result['hit_percent']:.1f}%</b> of boss HP"
                   if result["domain_allowed"] else "blocked by the topic gate")
    lines = [
        _line("1 · Topic gate", f"<b>{GATE_TEXT[result['domain_match_type']]}</b>",
              f"({domain} score {result['domain_scores'][domain]:+.3f}, lead {result['domain_lead']:+.3f} vs {rival})"),
        _line("&nbsp;&nbsp;&nbsp;Topic share", share),
        _line("2 · Relevance", f"<b>r = {result['relevance']:.2f}</b>",
              f"(cos {result['similarity']:.3f} · noise floor {result['noise_floor']:.3f} · ceiling {result['ceiling']:.3f})"),
        _line("3 · Damage", damage_line),
        _line("&nbsp;&nbsp;&nbsp;Boss HP", f"{hp_before} → <b>{max(hp_after, 0)}</b>",
              f"({result['latency_ms']:.0f} ms)"),
    ]
    head = (f"<div style='display:flex;justify-content:space-between;align-items:center;margin-bottom:6px'>"
            f"<div style='font-size:15px'><span style='color:#a1a1aa'>Turn {turn}</span> · "
            f"<b>'{_html.escape(result['word'])}'</b></div>"
            f"<div>{_badge(tier, color)} <b style='color:{color};font-size:16px;margin-left:6px'>"
            f"{'−' + str(damage) if damage else '0'} HP</b></div></div>")
    return f"<div style='{CARD}padding:10px 14px;margin:8px 0;border-left:5px solid {color}'>{head}{''.join(lines)}</div>"

def _table_html(frame: pd.DataFrame, colors: dict = None) -> str:
    """Dark HTML table; colors maps column -> {value: colour} for badge cells."""
    colors = colors or {}
    head = "".join(f"<th style='text-align:left;padding:6px 10px;color:#a1a1aa;font-size:12px;"
                   f"border-bottom:1px solid #3f3f46'>{_html.escape(str(c))}</th>" for c in frame.columns)
    rows = []
    for _, row in frame.iterrows():
        cells = []
        for c in frame.columns:
            value = row[c]
            text = _badge(str(value), colors[c].get(value, "#52525b")) if c in colors else _html.escape(str(value))
            cells.append(f"<td style='padding:5px 10px;font-size:13px;color:#e4e4e7;border-bottom:1px solid #27272a'>{text}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")
    return f"<table style='border-collapse:collapse;width:100%'><tr>{head}</tr>{''.join(rows)}</table>"

def stage_summary_html(stage_log: dict) -> str:
    """Per-stage result: outcome, HP left and every counted guess."""
    color = STATUS_COLORS[stage_log["status"]]
    guesses = pd.DataFrame(stage_log["guesses"]) if stage_log["guesses"] else pd.DataFrame(
        columns=["Turn", "Guess", "Topic gate", "Relevance r", "Damage %", "Damage HP", "Tier", "Boss HP after"])
    total = int(guesses["Damage HP"].sum()) if len(guesses) else 0
    best = guesses.loc[guesses["Damage HP"].idxmax(), "Guess"] if len(guesses) and total > 0 else "—"
    hp_pct = stage_log["hp_left"] / stage_log["hp_max"]
    facts = (f"<div style='display:flex;gap:18px;flex-wrap:wrap;font-size:13px;color:#d4d4d8;margin:8px 0'>"
             f"<div>Outcome {_badge(stage_log['status'], color)}</div>"
             f"<div>Boss HP left <b>{stage_log['hp_left']} / {stage_log['hp_max']}</b> ({hp_pct:.0%})</div>"
             f"<div>Turns <b>{stage_log['turns']}</b></div><div>Total damage <b>{total} HP</b></div>"
             f"<div>Best guess <b>{_html.escape(str(best))}</b></div></div>")
    title = (f"<div style='font-size:16px;font-weight:bold'>📋 Stage {stage_log['stage']} summary — "
             f"{_html.escape(stage_log['enemy'])} · {stage_log['domain']} · target "
             f"<span style='color:#fde047'>'{_html.escape(stage_log['target'])}'</span></div>")
    body = _table_html(guesses, {"Tier": TIER_COLORS}) if len(guesses) else \
        "<div style='color:#a1a1aa;font-size:13px'>No counted guesses.</div>"
    return f"<div style='{CARD}padding:12px 16px;margin:10px 0;border:2px solid {color}'>{title}{facts}{body}</div>"

def game_summary_html(stage_logs: list) -> str:
    """Whole-game table: one row per stage played."""
    rows = []
    for s in stage_logs:
        damage = sum(g["Damage HP"] for g in s["guesses"])
        rows.append({"Stage": s["stage"], "Topic": s["domain"], "Target": s["target"], "Boss": s["enemy"],
                     "Outcome": s["status"], "Boss HP left": f"{s['hp_left']} / {s['hp_max']}",
                     "Turns": s["turns"], "Damage dealt": damage})
    frame = pd.DataFrame(rows)
    defeated = sum(s["status"] == "Defeated" for s in stage_logs)
    turns = sum(s["turns"] for s in stage_logs)
    title = (f"<div style='font-size:18px;font-weight:bold;margin-bottom:8px'>🏁 Game summary — "
             f"{defeated}/{len(stage_logs)} bosses defeated · {turns} turns</div>")
    return f"<div style='{CARD}padding:14px 16px;margin:10px 0;border:2px solid #a855f7'>{title}" \
           f"{_table_html(frame, {'Outcome': STATUS_COLORS})}</div>"

class GameSession:
    """One game: random stages played one guess at a time, with a per-stage log."""

    def __init__(self, engine: LexicalCombatEngine, bank: pd.DataFrame, rounds: int, seed: int):
        self.engine, self.bank, self.rounds = engine, bank, rounds
        self.seed = seed
        self.new_game(seed)

    def new_game(self, seed: int = None) -> None:
        """Draw new stages (topic order, target per topic) and reset."""
        self.seed = self.seed + 1 if seed is None else seed
        self.stages = random_stages(self.bank, self.rounds, self.seed)
        self.index, self.stage_logs, self.quit = 0, [], False
        self._start_stage()

    def _start_stage(self) -> None:
        s = self.stage
        self.hp, self.used, self.turns = s["hp"], set(), 0
        self.stage_logs.append({"stage": s["stage"], "enemy": s["enemy"], "domain": s["domain"],
                                "target": s["target"], "hp_max": s["hp"], "hp_left": s["hp"],
                                "turns": 0, "status": "In progress", "guesses": []})

    @property
    def stage(self) -> dict:
        return self.stages[min(self.index, len(self.stages) - 1)]

    @property
    def current_log(self) -> dict:
        return self.stage_logs[-1]

    @property
    def finished(self) -> bool:
        return self.quit or self.index >= len(self.stages)

    @property
    def records(self) -> list:
        """Defeated stages (kept for compatibility)."""
        return [s for s in self.stage_logs if s["status"] == "Defeated"]

    def status(self) -> str:
        if self.finished:
            return f"Game over — {len(self.records)}/{len(self.stage_logs)} bosses defeated."
        s = self.stage
        return (f"Stage {s['stage']}/{len(self.stages)} · Topic {s['domain'].upper()} · Boss {s['enemy']} · "
                f"Target '{s['target'].upper()}' · HP {self.hp}/{s['hp']} · turns {self.turns}")

    def _end_stage(self, status: str) -> str:
        """Close the current stage; returns its summary HTML."""
        log = self.current_log
        log.update(status=status, hp_left=max(self.hp, 0), turns=self.turns)
        self.index += 1
        summary = stage_summary_html(log)
        if not self.finished:
            self._start_stage()
        return summary

    def attack(self, guess: str) -> list:
        """Score one guess. Returns HTML blocks to show (turn card, then summaries if the stage ended)."""
        if self.finished:
            return [f"<div style='{CARD}padding:8px 14px'>Game over. Press <b>New game</b> to play again.</div>"]
        s = self.stage
        hp_before = self.hp
        result = self.engine.evaluate_attack(guess, s["target"], s["domain"], self.used, s["hp"])
        if result["valid"]:
            result = self.engine.resolve_damage(result, s["hp"], self.hp)
            self.turns += 1
            self.used.add(result["word"])
            self.hp -= result["actual_damage"]
            self.current_log["guesses"].append({
                "Turn": self.turns, "Guess": result["word"], "Topic gate": GATE_SHORT[result["domain_match_type"]],
                "Relevance r": round(result["relevance"], 2), "Damage %": round(result["hit_percent"], 1),
                "Damage HP": result["actual_damage"], "Tier": result["tier"], "Boss HP after": max(self.hp, 0)})
            self.current_log.update(hp_left=max(self.hp, 0), turns=self.turns)
        blocks = [turn_card_html(self.engine, result, hp_before, self.hp, s["domain"], self.turns)]
        if self.hp <= 0:
            blocks.append(self._end_stage("Defeated"))
            if self.finished:
                blocks.append(game_summary_html(self.stage_logs))
        return blocks

    def hint(self) -> list:
        if self.finished:
            return []
        word = self.engine.hint(self.stage["target"], self.stage["domain"], self.used)
        return [f"<div style='{CARD}padding:8px 14px;margin:6px 0;border-color:#06b6d4'>💡 <b>Hint:</b> "
                f"try something like <b style='color:#67e8f9'>{_html.escape(word)}</b></div>"]

    def skip(self) -> list:
        """Give up the current boss and move on."""
        if self.finished:
            return []
        blocks = [self._end_stage("Skipped")]
        if self.finished:
            blocks.append(game_summary_html(self.stage_logs))
        return blocks

    def end_game(self) -> list:
        """Quit now: summary of the current stage (with the boss's HP left) and of the whole game."""
        if self.finished:
            return [game_summary_html(self.stage_logs)] if self.stage_logs else []
        log = self.current_log
        log.update(status="Quit", hp_left=max(self.hp, 0), turns=self.turns)
        self.quit = True
        return [stage_summary_html(log), game_summary_html(self.stage_logs)]

def play_in_text_mode(session: GameSession) -> None:
    """Fallback without widgets: input() loop ('hint', 'skip', 'quit'); prints the summary tables."""
    while not session.finished:
        guess = input(f"[{session.status()}] Your guess: ").strip()
        command = clean_text(guess)
        if command == "quit":
            session.end_game()
            break
        if command == "hint":
            print(f"  Hint: {session.engine.hint(session.stage['target'], session.stage['domain'], session.used)}")
        elif command == "skip":
            session.skip()
        else:
            stage_log = session.current_log
            count = len(stage_log["guesses"])
            session.attack(guess)
            if len(stage_log["guesses"]) > count:
                g = stage_log["guesses"][-1]
                print(f"  {g['Guess']}: {g['Tier']} · r {g['Relevance r']} · -{g['Damage HP']} HP · boss HP {g['Boss HP after']}")
            else:
                print("  Not counted (invalid guess).")
    for s in session.stage_logs:
        print(f"\nStage {s['stage']} {s['enemy']} ({s['domain']}, target '{s['target']}'): {s['status']}, "
              f"boss HP left {s['hp_left']}/{s['hp_max']}, {s['turns']} turns")
        if s["guesses"]:
            display(pd.DataFrame(s["guesses"]))

def build_game_widget(session: GameSession):
    """Topic / target cards, HP bar, input row with Attack / Hint / Skip / Quit / New game, styled log."""
    import ipywidgets as widgets
    from IPython.display import HTML

    header_card = widgets.HTML()
    hp_bar = widgets.IntProgress(min=0, layout=widgets.Layout(width="100%", height="16px", margin="6px 0 12px 0"))
    box = widgets.Text(placeholder="Type 1–3 English words, then Enter or Attack…",
                       layout=widgets.Layout(width="330px", height="36px"))
    btn = widgets.Layout(height="36px", width="auto", margin="0 0 0 6px")
    buttons = {
        "Attack": widgets.Button(description="⚔️ Attack", button_style="danger", layout=btn),
        "Hint": widgets.Button(description="💡 Hint", button_style="info", layout=btn),
        "Skip": widgets.Button(description="⏭️ Skip boss", layout=btn),
        "Quit": widgets.Button(description="🏳️ Quit", button_style="warning", layout=btn),
        "New": widgets.Button(description="🔄 New game", button_style="success", layout=btn),
    }
    log = widgets.Output(layout=widgets.Layout(border="1px solid #3f3f46", padding="6px 10px", margin="12px 0 0 0",
                                               max_height="460px", overflow="auto"))

    def show(blocks: list) -> None:
        with log:
            for block in blocks:
                display(HTML(block))

    def refresh() -> None:
        for name in ("Attack", "Hint", "Skip", "Quit"):
            buttons[name].disabled = session.finished
        box.disabled = session.finished
        if session.finished:
            defeated = len(session.records)
            played = len(session.stage_logs)
            title = "🏆 VICTORY — every boss defeated!" if defeated == len(session.stages) else "🏁 GAME OVER"
            color = "#22c55e" if defeated == len(session.stages) else "#a855f7"
            header_card.value = (f"<div style='{CARD}padding:16px;border:2px solid {color}'>"
                                 f"<div style='font-size:20px;font-weight:bold;color:{color}'>{title}</div>"
                                 f"<div style='margin-top:6px;font-size:14px;color:#d4d4d8'>Defeated <b>{defeated}/{played}</b> "
                                 f"bosses played. The summary tables are below. Press <b>New game</b> to play again.</div></div>")
            hp_bar.max, hp_bar.value = 100, 0
            return
        s = session.stage
        color = TOPIC_COLORS.get(s["domain"], "#eab308")
        hp_left, hp_max = max(session.hp, 0), s["hp"]
        hp_pct = hp_left / hp_max if hp_max else 0
        hp_bar.max, hp_bar.value = hp_max, hp_left
        hp_bar.bar_style = "success" if hp_pct > 0.5 else "warning" if hp_pct > 0.25 else "danger"
        header_card.value = f"""
        <div style="{CARD}padding:14px 16px">
          <div style="display:flex;justify-content:space-between;align-items:center;border-bottom:1px solid #27272a;padding-bottom:8px;margin-bottom:12px">
            <div><span style="background:#27272a;color:#a1a1aa;padding:3px 8px;border-radius:4px;font-size:12px;font-weight:bold">
              STAGE {s['stage']} / {len(session.stages)}</span>
              <span style="margin-left:10px;font-size:16px;font-weight:bold;color:#e4e4e7">👾 Boss: {_html.escape(s['enemy'])}</span></div>
            <div style="font-size:13px;color:#a1a1aa">Turns: <b style="color:#fff;font-size:15px">{session.turns}</b></div>
          </div>
          <div style="display:flex;gap:14px;flex-wrap:wrap;margin-bottom:12px">
            <div style="flex:1;min-width:180px;background:#27272a;border:2px solid {color};border-radius:8px;padding:10px 14px">
              <div style="font-size:11px;letter-spacing:1px;color:{color};font-weight:bold">📌 REQUIRED TOPIC (DOMAIN)</div>
              <div style="font-size:22px;font-weight:900;color:#fff;margin-top:2px">{s['domain'].upper()}</div></div>
            <div style="flex:1;min-width:180px;background:#27272a;border:2px solid #f59e0b;border-radius:8px;padding:10px 14px">
              <div style="font-size:11px;letter-spacing:1px;color:#fbbf24;font-weight:bold">🎯 TARGET WORD</div>
              <div style="font-size:22px;font-weight:900;color:#fde047;margin-top:2px">'{_html.escape(s['target'].upper())}'</div></div>
          </div>
          <div style="display:flex;justify-content:space-between;font-size:13px;font-weight:bold;color:#d4d4d8">
            <span>❤️ BOSS HP</span><span>{hp_left} / {hp_max} HP ({hp_pct:.0%})</span></div>
        </div>"""

    def on_attack(_=None) -> None:
        guess = box.value.strip()
        if not guess or session.finished:
            return
        box.value = ""
        show(session.attack(guess))
        refresh()

    def on_hint(_=None) -> None:
        show(session.hint())

    def on_skip(_=None) -> None:
        show(session.skip())
        refresh()

    def on_quit(_=None) -> None:
        show(session.end_game())
        refresh()

    def on_new(_=None) -> None:
        log.clear_output()
        session.new_game()
        s = session.stage
        show([f"<div style='{CARD}padding:8px 14px'>🎮 New game · {len(session.stages)} stages · first topic "
              f"<b>{s['domain'].upper()}</b>, target <b style='color:#fde047'>'{_html.escape(s['target'].upper())}'</b></div>"])
        refresh()

    buttons["Attack"].on_click(on_attack)
    buttons["Hint"].on_click(on_hint)
    buttons["Skip"].on_click(on_skip)
    buttons["Quit"].on_click(on_quit)
    buttons["New"].on_click(on_new)
    with warnings.catch_warnings():            # Text.on_submit is deprecated but still the simplest Enter hook
        warnings.simplefilter("ignore")
        if hasattr(box, "on_submit"):
            box.on_submit(on_attack)
    refresh()
    controls = widgets.HBox([box, *buttons.values()], layout=widgets.Layout(align_items="center", flex_flow="row wrap"))
    return widgets.VBox([header_card, hp_bar, controls, log], layout=widgets.Layout(max_width="820px", padding="4px"))

with section("Play Game"):
    GAME = GameSession(engine, TARGET_BANK, RANDOM_ROUNDS, SEED)
    print("Type a guess and press Enter or Attack. Hint suggests a word, Skip boss moves on, "
          "Quit ends the game and shows the summary tables, New game draws new topics and targets.")
    try:
        display(build_game_widget(GAME))
    except ImportError:
        print("WARNING: ipywidgets is not available — run play_in_text_mode(GAME) in a new cell to play.")


Type a guess and press Enter or Attack. Hint suggests a word, Skip boss moves on, Quit ends the game and shows the summary tables, New game draws new topics and targets.
